<a href="https://colab.research.google.com/github/Juna-Saputra/Pengolahan-Citra-Pola/blob/main/UAS_Pola_Citra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import cv2
# import gradio as gr
# import numpy as np
# import matplotlib.pyplot as plt
# import imageio

# from skimage.morphology import skeletonize

In [ ]:
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7a76cc4d32e62ddf3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
"""
===============================================================================
 APLIKASI FORENSIK DAN REKONSTRUKSI CITRA BURAM
 Mata Kuliah  : Pola dan Pengolahan Citra
 Materi       : Pertemuan 1 - 12 (Operasi Dasar s/d Skeletonization)
 Teknologi    : Python, OpenCV, NumPy, Matplotlib, Gradio
===============================================================================

ARSITEKTUR SISTEM (ringkas, lihat juga README yang dihasilkan terpisah):

    [Upload Gambar] --> [State Manager] --> [Engine Pengolahan Citra]
                                                |
                    -----------------------------------------------------
                    |             |             |             |        |
               Operasi Dasar  Aritmatika   Geometri/Scaling  Filter   Morfologi
                    |             |             |             |        |
                    -----------------------------------------------------
                                                |
                                  [Modul Visualisasi: Histogram,
                                   Matriks Piksel, Metadata, Log]
                                                |
                                  [Mode Pipeline]  atau  [Mode Single Filter]
                                                |
                                  [Forensic Reconstruction Lab]
                                                |
                                  [Export Gambar / Export PDF Laporan]

Semua fungsi pengolahan citra murni (pure function): menerima numpy array RGB
uint8, mengembalikan numpy array RGB/Gray uint8. Tidak ada state tersembunyi
sehingga mudah diuji & dipresentasikan.
===============================================================================
"""

import os
import io
import datetime
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
# from fpdf import FPDF
from PIL import Image

# =============================================================================
# BAGIAN 0. UTILITAS UMUM
# =============================================================================

OUTPUT_DIR = "exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def to_rgb3(img):
    """Pastikan gambar selalu dalam bentuk 3 channel RGB uint8 untuk ditampilkan."""
    if img is None:
        return None
    img = np.asarray(img)
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    if img.ndim == 3 and img.shape[2] == 1:
        return cv2.cvtColor(img[:, :, 0], cv2.COLOR_GRAY2RGB)
    if img.ndim == 3 and img.shape[2] == 4:
        return cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    return img


def to_gray(img):
    """Konversi citra RGB/Gray ke grayscale (selalu mengembalikan 2D array)."""
    img = np.asarray(img)
    if img.ndim == 2:
        return img.astype(np.uint8)
    return cv2.cvtColor(to_rgb3(img), cv2.COLOR_RGB2GRAY)


def get_metadata(img):
    """Mengembalikan string metadata citra: ukuran, channel, statistik piksel."""
    if img is None:
        return "Belum ada gambar yang diunggah."
    arr = np.asarray(img)
    h, w = arr.shape[0], arr.shape[1]
    c = 1 if arr.ndim == 2 else arr.shape[2]
    flat = arr.astype(np.float32)
    text = (
        f"METADATA CITRA\n"
        f"--------------------------------\n"
        f"Width            : {w} px\n"
        f"Height           : {h} px\n"
        f"Channels         : {c}\n"
        f"Min Pixel        : {flat.min():.2f}\n"
        f"Max Pixel        : {flat.max():.2f}\n"
        f"Mean Pixel       : {flat.mean():.2f}\n"
        f"Std Pixel        : {flat.std():.2f}\n"
    )
    return text


def pixel_matrix_10x10(img):
    """
    Mengambil sub-area 10x10 piksel dari bagian tengah citra (grayscale) dan
    menampilkannya sebagai matriks teks, karena menampilkan seluruh citra
    sebagai angka akan terlalu besar dan tidak terbaca.
    """
    if img is None:
        return "Tidak ada data piksel."
    gray = to_gray(img)
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    half = 5
    y0, y1 = max(0, cy - half), max(0, cy - half) + 10
    x0, x1 = max(0, cx - half), max(0, cx - half) + 10
    y1, x1 = min(h, y1), min(w, x1)
    patch = gray[y0:y1, x0:x1]
    rows = []
    for row in patch:
        rows.append("[" + " ".join(f"{v:3d}" for v in row) + "]")
    header = f"Matriks Piksel 10x10 (area tengah citra, baris {y0}-{y1}, kolom {x0}-{x1})\n"
    return header + "\n".join(rows)


def plot_histogram(img, title="Histogram"):
    """Membuat histogram intensitas (Matplotlib) dan mengembalikannya sebagai array gambar."""
    fig, ax = plt.subplots(figsize=(4, 2.6), dpi=110)
    if img is None:
        ax.text(0.5, 0.5, "Tidak ada gambar", ha="center", va="center")
        ax.axis("off")
    else:
        arr = np.asarray(img)
        if arr.ndim == 2:
            ax.hist(arr.ravel(), bins=256, range=(0, 255), color="gray")
        else:
            colors = ("r", "g", "b")
            for i, col in enumerate(colors):
                ax.hist(arr[:, :, i].ravel(), bins=256, range=(0, 255),
                         color=col, alpha=0.5, label=col.upper())
            ax.legend(fontsize=7)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Intensitas", fontsize=8)
        ax.set_ylabel("Frekuensi", fontsize=8)
        ax.tick_params(labelsize=7)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)
    buf.seek(0)
    out = np.array(Image.open(buf).convert("RGB"))
    return out


def log_line(step, title, **kwargs):
    """Membuat satu blok teks Processing Log yang konsisten formatnya."""
    text = f"STEP {step}:\n{title}\n"
    for k, v in kwargs.items():
        text += f"{k}:\n{v}\n"
    text += "\n"
    return text


def stat_block(img):
    """Statistik singkat sebuah citra: resolusi, rentang piksel, mean intensitas."""
    arr = np.asarray(img).astype(np.float32)
    h, w = arr.shape[0], arr.shape[1]
    return {
        "Resolution": f"{w} x {h}",
        "Pixel Range": f"{arr.min():.0f} - {arr.max():.0f}",
        "Mean Intensity": f"{arr.mean():.2f}",
    }


def safe_input_check(img):
    """Validasi sederhana: pastikan gambar sudah diunggah sebelum diproses."""
    if img is None:
        raise gr.Error("Silakan unggah gambar terlebih dahulu.")
    return to_rgb3(img)


# =============================================================================
# BAGIAN 1. PERTEMUAN 1 - OPERASI DASAR CITRA
# =============================================================================

def op_grayscale(img):
    g = to_gray(img)
    return to_rgb3(g)


def op_rgb_channel(img, channel):
    rgb = to_rgb3(img)
    out = np.zeros_like(rgb)
    idx = {"Red": 0, "Green": 1, "Blue": 2}[channel]
    out[:, :, idx] = rgb[:, :, idx]
    return out


def op_negative(img):
    rgb = to_rgb3(img)
    return 255 - rgb


# =============================================================================
# BAGIAN 2. PERTEMUAN 2 - OPERASI ARITMATIKA
# =============================================================================

def op_arithmetic(img, operation, value):
    rgb = to_rgb3(img).astype(np.float32)
    value = float(value)
    if operation == "Penjumlahan":
        out = rgb + value
    elif operation == "Pengurangan":
        out = rgb - value
    elif operation == "Perkalian":
        out = rgb * value
    elif operation == "Pembagian":
        out = rgb / (value if value != 0 else 1)
    else:
        out = rgb
    return np.clip(out, 0, 255).astype(np.uint8)


# =============================================================================
# BAGIAN 3. PERTEMUAN 3 - TRANSFORMASI GEOMETRI
# =============================================================================

def op_geometry(img, operation, custom_angle=0):
    rgb = to_rgb3(img)
    if operation == "Flip Horizontal":
        return cv2.flip(rgb, 1)
    if operation == "Flip Vertical":
        return cv2.flip(rgb, 0)
    if operation == "Rotate 90":
        return cv2.rotate(rgb, cv2.ROTATE_90_CLOCKWISE)
    if operation == "Rotate 180":
        return cv2.rotate(rgb, cv2.ROTATE_180)
    if operation == "Rotate 270":
        return cv2.rotate(rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if operation == "Custom Rotation":
        h, w = rgb.shape[:2]
        m = cv2.getRotationMatrix2D((w / 2, h / 2), float(custom_angle), 1.0)
        return cv2.warpAffine(rgb, m, (w, h))
    return rgb


# =============================================================================
# BAGIAN 4. PERTEMUAN 4 - SCALING
# =============================================================================

def op_scaling(img, method, factor):
    rgb = to_rgb3(img)
    factor = max(0.1, float(factor))
    h, w = rgb.shape[:2]
    new_w, new_h = max(1, int(w * factor)), max(1, int(h * factor))
    interp = cv2.INTER_NEAREST if method == "Nearest Neighbor" else cv2.INTER_LINEAR
    return cv2.resize(rgb, (new_w, new_h), interpolation=interp)


# =============================================================================
# BAGIAN 5. PERTEMUAN 5 - BRIGHTNESS & CONTRAST
# =============================================================================

def op_brightness_contrast(img, brightness, contrast):
    rgb = to_rgb3(img).astype(np.float32)
    # contrast: faktor pengali di sekitar titik tengah 128, brightness: offset aditif
    out = (rgb - 128) * float(contrast) + 128 + float(brightness)
    return np.clip(out, 0, 255).astype(np.uint8)


# =============================================================================
# BAGIAN 6. PERTEMUAN 6 - SMOOTHING FILTER
# =============================================================================

def op_smoothing(img, method, ksize):
    rgb = to_rgb3(img)
    k = int(ksize)
    if k % 2 == 0:
        k += 1
    if method == "Mean Filter":
        return cv2.blur(rgb, (k, k))
    if method == "Gaussian Filter":
        return cv2.GaussianBlur(rgb, (k, k), 0)
    if method == "Median Filter":
        return cv2.medianBlur(rgb, k)
    return rgb


# =============================================================================
# BAGIAN 7. PERTEMUAN 7 - SHARPENING
# =============================================================================

SHARPEN_KERNELS = {
    "Sharpen Kernel 1": np.array([[0, -1, 0],
                                   [-1, 5, -1],
                                   [0, -1, 0]]),
    "Sharpen Kernel 2": np.array([[-1, -1, -1],
                                   [-1, 9, -1],
                                   [-1, -1, -1]]),
    "Sharpen Kernel 3": np.array([[1, -2, 1],
                                   [-2, 5, -2],
                                   [1, -2, 1]]),
}


def op_sharpen(img, kernel_name):
    rgb = to_rgb3(img)
    kernel = SHARPEN_KERNELS[kernel_name]
    return cv2.filter2D(rgb, -1, kernel)


# =============================================================================
# BAGIAN 8. PERTEMUAN 8 - EDGE DETECTION
# =============================================================================

def op_edge(img, method):
    gray = to_gray(img)
    if method == "Sobel":
        sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        out = cv2.magnitude(sx, sy)
    elif method == "Prewitt":
        kx = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=np.float32)
        ky = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)
        sx = cv2.filter2D(gray.astype(np.float32), -1, kx)
        sy = cv2.filter2D(gray.astype(np.float32), -1, ky)
        out = cv2.magnitude(sx, sy)
    elif method == "Roberts":
        kx = np.array([[1, 0], [0, -1]], dtype=np.float32)
        ky = np.array([[0, 1], [-1, 0]], dtype=np.float32)
        sx = cv2.filter2D(gray.astype(np.float32), -1, kx)
        sy = cv2.filter2D(gray.astype(np.float32), -1, ky)
        out = cv2.magnitude(sx, sy)
    elif method == "Canny":
        out = cv2.Canny(gray, 100, 200).astype(np.float32)
    else:
        out = gray.astype(np.float32)
    out = np.clip(out, 0, 255).astype(np.uint8)
    return to_rgb3(out)


# =============================================================================
# BAGIAN 9. PERTEMUAN 9 - THRESHOLDING
# =============================================================================

def op_threshold(img, method, thresh_val=127):
    gray = to_gray(img)
    t = int(thresh_val)
    if method == "Binary":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
    elif method == "Binary Inverse":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY_INV)
    elif method == "Trunc":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_TRUNC)
    elif method == "To Zero":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_TOZERO)
    elif method == "Adaptive Mean":
        out = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                     cv2.THRESH_BINARY, 11, 2)
    elif method == "Adaptive Gaussian":
        out = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, 2)
    elif method == "Otsu":
        _, out = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        out = gray
    return to_rgb3(out)


# =============================================================================
# BAGIAN 10. PERTEMUAN 10 - MORFOLOGI CITRA
# =============================================================================

def op_morphology(img, operation, ksize):
    gray = to_gray(img)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = int(ksize)
    kernel = np.ones((k, k), np.uint8)
    if operation == "Erosion":
        out = cv2.erode(binary, kernel, iterations=1)
    elif operation == "Dilation":
        out = cv2.dilate(binary, kernel, iterations=1)
    elif operation == "Opening":
        out = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    elif operation == "Closing":
        out = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    else:
        out = binary
    return to_rgb3(out)


# =============================================================================
# BAGIAN 11. PERTEMUAN 11 - SKELETONIZATION (ZHANG-SUEN THINNING)
# =============================================================================

def zhang_suen_thinning(binary_img):
    """
    Implementasi murni Zhang-Suen thinning algorithm.
    Input  : binary image (0/255) numpy array 2D
    Output : binary image hasil skeleton (0/255)

    NB: Untuk performa, citra besar di-resize sementara ke maksimal 300px
    pada sisi terpanjang sebelum thinning, lalu dikembalikan ke ukuran asli.
    """
    img = (binary_img > 0).astype(np.uint8)
    h, w = img.shape
    scale = 1.0
    max_side = max(h, w)
    if max_side > 300:
        scale = 300.0 / max_side
        img = cv2.resize(img, (int(w * scale), int(h * scale)),
                          interpolation=cv2.INTER_NEAREST)

    def neighbours(x, y, image):
        return [image[x - 1, y], image[x - 1, y + 1], image[x, y + 1], image[x + 1, y + 1],
                image[x + 1, y], image[x + 1, y - 1], image[x, y - 1], image[x - 1, y - 1]]

    def transitions(nb):
        n = nb + nb[0:1]
        return sum((n[i] == 0 and n[i + 1] == 1) for i in range(8))

    changing1 = changing2 = [(-1, -1)]
    while changing1 or changing2:
        rows, cols = img.shape
        changing1 = []
        for x in range(1, rows - 1):
            for y in range(1, cols - 1):
                p2, p3, p4, p5, p6, p7, p8, p9 = neighbours(x, y, img)
                if (img[x, y] == 1 and 2 <= sum([p2, p3, p4, p5, p6, p7, p8, p9]) <= 6
                        and transitions([p2, p3, p4, p5, p6, p7, p8, p9]) == 1
                        and p2 * p4 * p6 == 0 and p4 * p6 * p8 == 0):
                    changing1.append((x, y))
        for x, y in changing1:
            img[x, y] = 0

        changing2 = []
        for x in range(1, rows - 1):
            for y in range(1, cols - 1):
                p2, p3, p4, p5, p6, p7, p8, p9 = neighbours(x, y, img)
                if (img[x, y] == 1 and 2 <= sum([p2, p3, p4, p5, p6, p7, p8, p9]) <= 6
                        and transitions([p2, p3, p4, p5, p6, p7, p8, p9]) == 1
                        and p2 * p4 * p8 == 0 and p2 * p6 * p8 == 0):
                    changing2.append((x, y))
        for x, y in changing2:
            img[x, y] = 0

    if scale != 1.0:
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_NEAREST)
    return (img * 255).astype(np.uint8)


def op_skeleton(img):
    gray = to_gray(img)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    skeleton = zhang_suen_thinning(binary)
    return to_rgb3(binary), to_rgb3(skeleton)


# =============================================================================
# BAGIAN 12. PIPELINE LENGKAP (MODE BERANTAI) UNTUK FORENSIC LAB
# =============================================================================

PIPELINE_STEPS = [
    "Grayscale Conversion",
    "Flipping / Rotation",
    "Scaling",
    "Contrast Enhancement",
    "Smoothing Filter",
    "Edge Detection",
    "Thresholding",
    "Morphological Operation",
    "Thinning Zhang-Suen",
]


def run_pipeline(img, flip_choice, scale_factor, contrast_val, smooth_method,
                  smooth_k, edge_method, thresh_method, morph_op, morph_k):
    """
    Mode Berantai (Pipeline): output tahap sebelumnya menjadi input tahap
    berikutnya. Mengembalikan list of (nama_tahap, gambar, log_text).
    """
    rgb = safe_input_check(img)
    results = []
    log_text = ""

    # STEP 1: Grayscale
    cur = op_grayscale(rgb)
    s = stat_block(cur)
    log_text += log_line(1, "Grayscale Applied", **s)
    results.append(("Grayscale Conversion", cur))

    # STEP 2: Flip/Rotation
    cur = op_geometry(cur, flip_choice)
    s = stat_block(cur)
    log_text += log_line(2, f"{flip_choice} Applied", **s)
    results.append(("Flipping / Rotation", cur))

    # STEP 3: Scaling
    cur = op_scaling(cur, "Bilinear", scale_factor)
    s = stat_block(cur)
    log_text += log_line(3, "Scaling Applied", **{**s, "Factor": f"{scale_factor}x"})
    results.append(("Scaling", cur))

    # STEP 4: Contrast Enhancement
    cur = op_brightness_contrast(cur, 0, contrast_val)
    s = stat_block(cur)
    log_text += log_line(4, "Contrast Enhancement Applied", **{**s, "Contrast Factor": contrast_val})
    results.append(("Contrast Enhancement", cur))

    # STEP 5: Smoothing
    cur = op_smoothing(cur, smooth_method, smooth_k)
    s = stat_block(cur)
    log_text += log_line(5, f"{smooth_method} Applied", **{**s, "Kernel": f"{smooth_k}x{smooth_k}"})
    results.append(("Smoothing Filter", cur))

    # STEP 6: Edge Detection
    cur = op_edge(cur, edge_method)
    s = stat_block(cur)
    log_text += log_line(6, f"{edge_method} Edge Detection Applied", **s)
    results.append(("Edge Detection", cur))

    # STEP 7: Thresholding
    cur = op_threshold(cur, thresh_method)
    s = stat_block(cur)
    log_text += log_line(7, f"{thresh_method} Thresholding Applied", **s)
    results.append(("Thresholding", cur))

    # STEP 8: Morphology
    cur = op_morphology(cur, morph_op, morph_k)
    s = stat_block(cur)
    log_text += log_line(8, f"{morph_op} Applied", **{**s, "Kernel": f"{morph_k}x{morph_k}"})
    results.append(("Morphological Operation", cur))

    # STEP 9: Thinning
    gray_cur = to_gray(cur)
    _, binary_cur = cv2.threshold(gray_cur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    skel = zhang_suen_thinning(binary_cur)
    cur = to_rgb3(skel)
    s = stat_block(cur)
    log_text += log_line(9, "Zhang-Suen Thinning Applied", **s)
    results.append(("Thinning Zhang-Suen", cur))

    return results, log_text


def forensic_analysis_text(rgb_original, final_image, pipeline_results):
    """
    Membuat narasi analisis forensik otomatis berdasarkan perbandingan
    statistik antar tahap (edge density, noise level, dsb).
    """
    orig_gray = to_gray(rgb_original).astype(np.float32)
    final_gray = to_gray(final_image).astype(np.float32)

    edge_before = cv2.Laplacian(orig_gray.astype(np.uint8), cv2.CV_64F).var()
    # cari hasil tahap smoothing & edge & threshold untuk narasi lebih kaya
    stage_dict = {name: img for name, img in pipeline_results}

    notes = []

    if "Smoothing Filter" in stage_dict:
        before_std = orig_gray.std()
        after_std = to_gray(stage_dict["Smoothing Filter"]).astype(np.float32).std()
        noise_reduction = max(0.0, (before_std - after_std) / (before_std + 1e-6) * 100)
        notes.append(f"Noise berkurang sekitar {noise_reduction:.1f}% setelah tahap Smoothing Filter.")

    if "Edge Detection" in stage_dict:
        edge_after = cv2.Laplacian(to_gray(stage_dict["Edge Detection"]), cv2.CV_64F).var()
        edge_change = ((edge_after - edge_before) / (edge_before + 1e-6)) * 100
        notes.append(f"Edge detail berubah sekitar {edge_change:.1f}% setelah proses Edge Detection "
                     f"dibandingkan citra asli.")

    if "Thresholding" in stage_dict:
        notes.append("Objek pada citra semakin teridentifikasi (kontur lebih tegas) "
                      "setelah tahap Thresholding diterapkan.")

    if "Morphological Operation" in stage_dict:
        notes.append("Struktur objek dirapikan dan noise kecil dihilangkan setelah tahap Operasi Morfologi.")

    if "Thinning Zhang-Suen" in stage_dict:
        notes.append("Kerangka (skeleton) objek berhasil diekstraksi pada tahap akhir Thinning Zhang-Suen, "
                      "sehingga pola/bentuk dasar objek dapat dikenali kembali.")

    final_mean_diff = final_gray.mean() - orig_gray.mean()
    notes.append(f"Rata-rata intensitas piksel berubah sebesar {final_mean_diff:.2f} "
                 f"dari citra asli ke hasil rekonstruksi akhir.")

    header = "ANALISIS FORENSIK OTOMATIS\n" + "-" * 32 + "\n"
    return header + "\n".join(f"- {n}" for n in notes)


# =============================================================================
# BAGIAN 13. EXPORT GAMBAR & EXPORT LAPORAN PDF
# =============================================================================

def sanitize_for_pdf(text):
    """
    fpdf2 memiliki bug pada word-wrap untuk karakter berulang tanpa spasi
    (misal '---------------'), yang memicu FPDFException pada multi_cell.
    Fungsi ini menyisipkan spasi tipis di antara karakter berulang seperti
    itu agar PDF tetap aman dirender tanpa mengubah tampilan secara berarti.
    """
    import re

    def fix_run(m):
        return " ".join(m.group(0))

    return re.sub(r"[-=_]{4,}", fix_run, text)


def export_image(img):
    if img is None:
        raise gr.Error("Tidak ada gambar untuk diekspor.")
    rgb = to_rgb3(img)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(OUTPUT_DIR, f"hasil_{timestamp}.png")
    Image.fromarray(rgb).save(path)
    return path


def export_pdf_report(original_img, final_img, log_text, analysis_text, metadata_text):
    """Menghasilkan laporan proses lengkap dalam format PDF (gambar awal, akhir, log, analisis)."""
    if original_img is None or final_img is None:
        raise gr.Error("Jalankan pipeline terlebih dahulu sebelum mengekspor laporan.")

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    orig_path = os.path.join(OUTPUT_DIR, f"_tmp_orig_{timestamp}.png")
    final_path = os.path.join(OUTPUT_DIR, f"_tmp_final_{timestamp}.png")
    Image.fromarray(to_rgb3(original_img)).save(orig_path)
    Image.fromarray(to_rgb3(final_img)).save(final_path)

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, "Laporan Forensik & Rekonstruksi Citra Buram", ln=True)
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 8, f"Dibuat pada: {datetime.datetime.now().strftime('%d-%m-%Y %H:%M:%S')}", ln=True)
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "1. Gambar Sebelum & Sesudah", ln=True)
    pdf.set_font("Helvetica", "", 9)
    pdf.cell(0, 6, "Gambar Asli (Sebelum):", ln=True)
    pdf.image(orig_path, w=80)
    pdf.ln(3)
    pdf.cell(0, 6, "Hasil Rekonstruksi Akhir (Sesudah):", ln=True)
    pdf.image(final_path, w=80)
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "2. Metadata Citra Akhir", ln=True)
    pdf.set_font("Courier", "", 9)
    for line in sanitize_for_pdf(metadata_text).split("\n"):
        pdf.cell(0, 5, line if line.strip() else " ", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "3. Processing Log", ln=True)
    pdf.set_font("Courier", "", 8)
    for line in sanitize_for_pdf(log_text).split("\n"):
        pdf.cell(0, 4.2, line if line.strip() else " ", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "4. Analisis Forensik", ln=True)
    pdf.set_font("Helvetica", "", 9)
    for line in sanitize_for_pdf(analysis_text).split("\n"):
        wrapped = [line[i:i+95] for i in range(0, len(line), 95)] or [" "]
        for w in wrapped:
            pdf.cell(0, 5, w, new_x="LMARGIN", new_y="NEXT")

    out_path = os.path.join(OUTPUT_DIR, f"laporan_{timestamp}.pdf")
    pdf.output(out_path)

    for p in (orig_path, final_path):
        try:
            os.remove(p)
        except OSError:
            pass
    return out_path


# =============================================================================
# BAGIAN 14. WRAPPER GENERIK UNTUK TAB "MODE MANDIRI" (SINGLE FILTER LAB)
# Setiap fungsi wrapper mengembalikan:
#   gambar_output, hist_before, hist_after, matrix_before, matrix_after,
#   metadata_text, log_text
# =============================================================================

def generic_wrap(img, processed, step_title, extra_stats=None):
    rgb = safe_input_check(img)
    hist_before = plot_histogram(rgb, "Histogram Sebelum")
    hist_after = plot_histogram(processed, "Histogram Sesudah")
    matrix_before = pixel_matrix_10x10(rgb)
    matrix_after = pixel_matrix_10x10(processed)
    metadata = get_metadata(processed)
    s = stat_block(processed)
    if extra_stats:
        s.update(extra_stats)
    log_text = log_line(1, step_title, **s)
    return processed, hist_before, hist_after, matrix_before, matrix_after, metadata, log_text


def ui_grayscale(img):
    rgb = safe_input_check(img)
    out = op_grayscale(rgb)
    return generic_wrap(img, out, "Grayscale Applied")


def ui_rgb_channel(img, channel):
    rgb = safe_input_check(img)
    out = op_rgb_channel(rgb, channel)
    return generic_wrap(img, out, f"RGB Channel ({channel}) Applied")


def ui_negative(img):
    rgb = safe_input_check(img)
    out = op_negative(rgb)
    return generic_wrap(img, out, "Negative Image Applied")


def ui_arithmetic(img, operation, value):
    rgb = safe_input_check(img)
    out = op_arithmetic(rgb, operation, value)
    return generic_wrap(img, out, f"Operasi Aritmatika: {operation}", {"Value": value})


def ui_geometry(img, operation, angle):
    rgb = safe_input_check(img)
    out = op_geometry(rgb, operation, angle)
    return generic_wrap(img, out, f"{operation} Applied")


def ui_scaling(img, method, factor):
    rgb = safe_input_check(img)
    out = op_scaling(rgb, method, factor)
    return generic_wrap(img, out, f"Scaling ({method}) Applied", {"Factor": f"{factor}x"})


def ui_brightness_contrast(img, brightness, contrast):
    rgb = safe_input_check(img)
    out = op_brightness_contrast(rgb, brightness, contrast)
    return generic_wrap(img, out, "Brightness & Contrast Applied",
                         {"Brightness": brightness, "Contrast": contrast})


def ui_smoothing(img, method, ksize):
    rgb = safe_input_check(img)
    out = op_smoothing(rgb, method, ksize)
    return generic_wrap(img, out, f"{method} Applied", {"Kernel": f"{ksize}x{ksize}"})


def ui_sharpen(img, kernel_name):
    rgb = safe_input_check(img)
    out = op_sharpen(rgb, kernel_name)
    return generic_wrap(img, out, f"{kernel_name} Applied")


def ui_edge(img, method):
    rgb = safe_input_check(img)
    out = op_edge(rgb, method)
    return generic_wrap(img, out, f"{method} Edge Detection Applied")


def ui_threshold(img, method, t):
    rgb = safe_input_check(img)
    out = op_threshold(rgb, method, t)
    return generic_wrap(img, out, f"{method} Thresholding Applied", {"Threshold": t})


def ui_morphology(img, operation, ksize):
    rgb = safe_input_check(img)
    out = op_morphology(rgb, operation, ksize)
    return generic_wrap(img, out, f"{operation} Applied", {"Kernel": f"{ksize}x{ksize}"})


def ui_skeleton(img):
    rgb = safe_input_check(img)
    binary, skeleton = op_skeleton(rgb)
    hist_before = plot_histogram(binary, "Histogram Sebelum Thinning")
    hist_after = plot_histogram(skeleton, "Histogram Sesudah Thinning")
    matrix_before = pixel_matrix_10x10(binary)
    matrix_after = pixel_matrix_10x10(skeleton)
    metadata = get_metadata(skeleton)
    s = stat_block(skeleton)
    log_text = log_line(1, "Zhang-Suen Thinning Applied", **s)
    return binary, skeleton, hist_before, hist_after, matrix_before, matrix_after, metadata, log_text


# =============================================================================
# BAGIAN 15. FORENSIC RECONSTRUCTION LAB (TAB UTAMA) + MODE BERANTAI/MANDIRI
# =============================================================================

SINGLE_FILTER_CHOICES = [
    "Grayscale", "Negative", "Flip Horizontal", "Flip Vertical", "Rotate 90",
    "Scaling 2x (Bilinear)", "Brightness +40", "Gaussian Smoothing 5x5",
    "Sharpen Kernel 2", "Sobel Edge", "Canny Edge", "Otsu Threshold",
    "Dilation", "Erosion", "Zhang-Suen Thinning",
]


def apply_single_filter(rgb, choice):
    """Mode Mandiri: setiap operasi diterapkan langsung ke gambar ASLI, tidak berantai."""
    if choice == "Grayscale":
        return op_grayscale(rgb)
    if choice == "Negative":
        return op_negative(rgb)
    if choice == "Flip Horizontal":
        return op_geometry(rgb, "Flip Horizontal")
    if choice == "Flip Vertical":
        return op_geometry(rgb, "Flip Vertical")
    if choice == "Rotate 90":
        return op_geometry(rgb, "Rotate 90")
    if choice == "Scaling 2x (Bilinear)":
        return op_scaling(rgb, "Bilinear", 2.0)
    if choice == "Brightness +40":
        return op_brightness_contrast(rgb, 40, 1.0)
    if choice == "Gaussian Smoothing 5x5":
        return op_smoothing(rgb, "Gaussian Filter", 5)
    if choice == "Sharpen Kernel 2":
        return op_sharpen(rgb, "Sharpen Kernel 2")
    if choice == "Sobel Edge":
        return op_edge(rgb, "Sobel")
    if choice == "Canny Edge":
        return op_edge(rgb, "Canny")
    if choice == "Otsu Threshold":
        return op_threshold(rgb, "Otsu")
    if choice == "Dilation":
        return op_morphology(rgb, "Dilation", 3)
    if choice == "Erosion":
        return op_morphology(rgb, "Erosion", 3)
    if choice == "Zhang-Suen Thinning":
        _, skel = op_skeleton(rgb)
        return skel
    return rgb


def run_forensic_lab(img, mode, single_choice, flip_choice, scale_factor, contrast_val,
                      smooth_method, smooth_k, edge_method, thresh_method, morph_op, morph_k):
    rgb = safe_input_check(img)

    if mode == "Mode Mandiri (Single Filter)":
        final = apply_single_filter(rgb, single_choice)
        s = stat_block(final)
        log_text = log_line(1, f"Single Filter: {single_choice}", **s)
        gallery = [(rgb, "Original"), (final, single_choice)]
        analysis = forensic_analysis_text(rgb, final, [(single_choice, final)])
    else:
        results, log_text = run_pipeline(
            rgb, flip_choice, scale_factor, contrast_val, smooth_method,
            smooth_k, edge_method, thresh_method, morph_op, morph_k
        )
        final = results[-1][1]
        gallery = [(rgb, "Original")] + [(im, name) for name, im in results]
        analysis = forensic_analysis_text(rgb, final, results)

    hist_before = plot_histogram(rgb, "Histogram Original")
    hist_after = plot_histogram(final, "Histogram Hasil Akhir")
    matrix_before = pixel_matrix_10x10(rgb)
    matrix_after = pixel_matrix_10x10(final)
    metadata = get_metadata(final)

    return (gallery, final, hist_before, hist_after, matrix_before, matrix_after,
            metadata, log_text, analysis)


# =============================================================================
# BAGIAN 16. ANTARMUKA GRADIO (UI)
# =============================================================================

CUSTOM_CSS = """
#log_box textarea {font-family: monospace; font-size: 12px;}
#matrix_box textarea {font-family: monospace; font-size: 11px;}
"""

with gr.Blocks(title="Forensik & Rekonstruksi Citra Buram", css=CUSTOM_CSS) as demo:
    gr.Markdown(
        """
        # 🔍 Aplikasi Forensik dan Rekonstruksi Citra Buram
        **Mata Kuliah Pola dan Pengolahan Citra** — Laboratorium interaktif untuk seluruh
        materi Pertemuan 1–12, sekaligus sistem restorasi citra blur secara bertahap.
        """
    )

    # ---------- SIDEBAR GLOBAL (Upload, Mode, Forensic Lab parameter) ----------
    with gr.Accordion("📁 Sidebar Global - Upload & Mode (untuk Tab 'Forensic Reconstruction Lab')", open=True):
        with gr.Row():
            with gr.Column(scale=1):
                global_image = gr.Image(label="Upload Image (Blur/Buram)", type="numpy")
                mode_select = gr.Radio(
                    ["Mode Berantai (Pipeline)", "Mode Mandiri (Single Filter)"],
                    value="Mode Berantai (Pipeline)", label="Mode Pemrosesan"
                )
                single_choice = gr.Dropdown(SINGLE_FILTER_CHOICES, value="Grayscale",
                                             label="Pilihan Filter (khusus Mode Mandiri)")
            with gr.Column(scale=1):
                gr.Markdown("**Parameter Pipeline (khusus Mode Berantai)**")
                p_flip = gr.Dropdown(["Flip Horizontal", "Flip Vertical", "Rotate 90",
                                      "Rotate 180", "Rotate 270"], value="Flip Horizontal",
                                     label="Tahap 2: Flip/Rotasi")
                p_scale = gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="Tahap 3: Scaling Factor")
                p_contrast = gr.Slider(0.5, 3.0, value=1.3, step=0.1, label="Tahap 4: Contrast Factor")
                p_smooth_method = gr.Dropdown(["Mean Filter", "Gaussian Filter", "Median Filter"],
                                              value="Gaussian Filter", label="Tahap 5: Smoothing")
                p_smooth_k = gr.Slider(3, 7, value=5, step=2, label="Tahap 5: Kernel Smoothing")
                p_edge_method = gr.Dropdown(["Sobel", "Prewitt", "Roberts", "Canny"],
                                            value="Canny", label="Tahap 6: Edge Detection")
                p_thresh_method = gr.Dropdown(["Binary", "Binary Inverse", "Trunc", "To Zero",
                                               "Adaptive Mean", "Adaptive Gaussian", "Otsu"],
                                              value="Otsu", label="Tahap 7: Thresholding")
                p_morph_op = gr.Dropdown(["Erosion", "Dilation", "Opening", "Closing"],
                                         value="Closing", label="Tahap 8: Morfologi")
                p_morph_k = gr.Slider(3, 9, value=3, step=2, label="Tahap 8: Kernel Morfologi")

    with gr.Tabs():
        # ===================== TAB 1: OPERASI DASAR =====================
        with gr.Tab("1. Operasi Dasar Citra"):
            with gr.Row():
                t1_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t1_op = gr.Radio(["Grayscale", "RGB Channel", "Negative Image"],
                                     value="Grayscale", label="Pilih Operasi")
                    t1_channel = gr.Dropdown(["Red", "Green", "Blue"], value="Red",
                                             label="Channel (khusus RGB Channel)")
                    t1_btn = gr.Button("Proses", variant="primary")
            with gr.Row():
                t1_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t1_hist_b = gr.Image(label="Histogram Sebelum")
                t1_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t1_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t1_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t1_meta = gr.Textbox(label="Metadata", lines=7)
            t1_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            def t1_run(img, op, ch):
                if op == "Grayscale":
                    return ui_grayscale(img)
                if op == "RGB Channel":
                    return ui_rgb_channel(img, ch)
                return ui_negative(img)

            t1_btn.click(t1_run, [t1_img, t1_op, t1_channel],
                         [t1_out, t1_hist_b, t1_hist_a, t1_mat_b, t1_mat_a, t1_meta, t1_log])

        # ===================== TAB 2: OPERASI ARITMATIKA =====================
        with gr.Tab("2. Operasi Aritmatika"):
            with gr.Row():
                t2_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t2_op = gr.Radio(["Penjumlahan", "Pengurangan", "Perkalian", "Pembagian"],
                                     value="Penjumlahan", label="Operasi")
                    t2_val = gr.Slider(1, 100, value=30, step=1, label="Nilai (scalar)")
                    t2_btn = gr.Button("Proses", variant="primary")
            with gr.Row():
                t2_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t2_hist_b = gr.Image(label="Histogram Sebelum")
                t2_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel (sebelum/sesudah) & Metadata", open=True):
                with gr.Row():
                    t2_mat_b = gr.Textbox(label="Matriks Sebelum (10x10)", lines=11, elem_id="matrix_box")
                    t2_mat_a = gr.Textbox(label="Matriks Sesudah (10x10)", lines=11, elem_id="matrix_box")
                t2_meta = gr.Textbox(label="Metadata", lines=7)
            t2_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t2_btn.click(ui_arithmetic, [t2_img, t2_op, t2_val],
                         [t2_out, t2_hist_b, t2_hist_a, t2_mat_b, t2_mat_a, t2_meta, t2_log])

        # ===================== TAB 3: TRANSFORMASI GEOMETRI =====================
        with gr.Tab("3. Transformasi Geometri"):
            with gr.Row():
                t3_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t3_op = gr.Radio(["Flip Horizontal", "Flip Vertical", "Rotate 90",
                                      "Rotate 180", "Rotate 270", "Custom Rotation"],
                                     value="Flip Horizontal", label="Operasi")
                    t3_angle = gr.Slider(0, 360, value=45, step=1, label="Sudut (Custom Rotation)")
                    t3_btn = gr.Button("Proses", variant="primary")
            t3_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t3_hist_b = gr.Image(label="Histogram Sebelum")
                t3_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t3_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t3_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t3_meta = gr.Textbox(label="Metadata", lines=7)
            t3_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t3_btn.click(ui_geometry, [t3_img, t3_op, t3_angle],
                         [t3_out, t3_hist_b, t3_hist_a, t3_mat_b, t3_mat_a, t3_meta, t3_log])

        # ===================== TAB 4: SCALING =====================
        with gr.Tab("4. Scaling"):
            with gr.Row():
                t4_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t4_method = gr.Radio(["Nearest Neighbor", "Bilinear"], value="Bilinear",
                                         label="Metode Interpolasi")
                    t4_factor = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="Faktor Skala")
                    t4_btn = gr.Button("Proses", variant="primary")
            t4_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t4_hist_b = gr.Image(label="Histogram Sebelum")
                t4_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t4_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t4_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t4_meta = gr.Textbox(label="Metadata", lines=7)
            t4_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t4_btn.click(ui_scaling, [t4_img, t4_method, t4_factor],
                         [t4_out, t4_hist_b, t4_hist_a, t4_mat_b, t4_mat_a, t4_meta, t4_log])

        # ===================== TAB 5: BRIGHTNESS & CONTRAST =====================
        with gr.Tab("5. Brightness & Contrast"):
            with gr.Row():
                t5_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t5_brightness = gr.Slider(-100, 100, value=0, step=1, label="Brightness")
                    t5_contrast = gr.Slider(0.2, 3.0, value=1.0, step=0.1, label="Contrast")
                    t5_btn = gr.Button("Proses", variant="primary")
            t5_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t5_hist_b = gr.Image(label="Histogram Sebelum")
                t5_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t5_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t5_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t5_meta = gr.Textbox(label="Metadata", lines=7)
            t5_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t5_btn.click(ui_brightness_contrast, [t5_img, t5_brightness, t5_contrast],
                         [t5_out, t5_hist_b, t5_hist_a, t5_mat_b, t5_mat_a, t5_meta, t5_log])

        # ===================== TAB 6: SMOOTHING FILTER =====================
        with gr.Tab("6. Smoothing Filter"):
            with gr.Row():
                t6_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t6_method = gr.Radio(["Mean Filter", "Gaussian Filter", "Median Filter"],
                                         value="Gaussian Filter", label="Metode")
                    t6_k = gr.Radio([3, 5, 7], value=5, label="Ukuran Kernel")
                    t6_btn = gr.Button("Proses", variant="primary")
            t6_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t6_hist_b = gr.Image(label="Histogram Sebelum")
                t6_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t6_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t6_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t6_meta = gr.Textbox(label="Metadata", lines=7)
            t6_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t6_btn.click(ui_smoothing, [t6_img, t6_method, t6_k],
                         [t6_out, t6_hist_b, t6_hist_a, t6_mat_b, t6_mat_a, t6_meta, t6_log])

        # ===================== TAB 7: SHARPENING =====================
        with gr.Tab("7. Sharpening"):
            with gr.Row():
                t7_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t7_kernel = gr.Radio(list(SHARPEN_KERNELS.keys()),
                                         value="Sharpen Kernel 1", label="Pilih Kernel")
                    t7_btn = gr.Button("Proses", variant="primary")
            t7_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t7_hist_b = gr.Image(label="Histogram Sebelum")
                t7_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t7_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t7_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t7_meta = gr.Textbox(label="Metadata", lines=7)
            t7_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t7_btn.click(ui_sharpen, [t7_img, t7_kernel],
                         [t7_out, t7_hist_b, t7_hist_a, t7_mat_b, t7_mat_a, t7_meta, t7_log])

        # ===================== TAB 8: EDGE DETECTION =====================
        with gr.Tab("8. Edge Detection"):
            with gr.Row():
                t8_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t8_method = gr.Radio(["Sobel", "Prewitt", "Roberts", "Canny"],
                                         value="Sobel", label="Metode")
                    t8_btn = gr.Button("Proses", variant="primary")
            t8_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t8_hist_b = gr.Image(label="Histogram Sebelum")
                t8_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t8_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t8_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t8_meta = gr.Textbox(label="Metadata", lines=7)
            t8_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t8_btn.click(ui_edge, [t8_img, t8_method],
                         [t8_out, t8_hist_b, t8_hist_a, t8_mat_b, t8_mat_a, t8_meta, t8_log])

        # ===================== TAB 9: THRESHOLDING =====================
        with gr.Tab("9. Thresholding"):
            with gr.Row():
                t9_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t9_method = gr.Radio(["Binary", "Binary Inverse", "Trunc", "To Zero",
                                          "Adaptive Mean", "Adaptive Gaussian", "Otsu"],
                                         value="Binary", label="Metode")
                    t9_thresh = gr.Slider(0, 255, value=127, step=1,
                                          label="Threshold Value (non-adaptive/non-otsu)")
                    t9_btn = gr.Button("Proses", variant="primary")
            t9_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t9_hist_b = gr.Image(label="Histogram Sebelum")
                t9_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t9_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t9_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t9_meta = gr.Textbox(label="Metadata", lines=7)
            t9_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t9_btn.click(ui_threshold, [t9_img, t9_method, t9_thresh],
                         [t9_out, t9_hist_b, t9_hist_a, t9_mat_b, t9_mat_a, t9_meta, t9_log])

        # ===================== TAB 10: MORFOLOGI CITRA =====================
        with gr.Tab("10. Morfologi Citra"):
            with gr.Row():
                t10_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t10_op = gr.Radio(["Erosion", "Dilation", "Opening", "Closing"],
                                      value="Erosion", label="Operasi")
                    t10_k = gr.Slider(3, 9, value=3, step=2, label="Ukuran Kernel")
                    t10_btn = gr.Button("Proses", variant="primary")
            t10_out = gr.Image(label="Gambar Output (citra dibinerisasi Otsu lalu dimorfologi)")
            with gr.Row():
                t10_hist_b = gr.Image(label="Histogram Sebelum")
                t10_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t10_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t10_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t10_meta = gr.Textbox(label="Metadata", lines=7)
            t10_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t10_btn.click(ui_morphology, [t10_img, t10_op, t10_k],
                          [t10_out, t10_hist_b, t10_hist_a, t10_mat_b, t10_mat_a, t10_meta, t10_log])

        # ===================== TAB 11: SKELETONIZATION =====================
        with gr.Tab("11. Skeletonization (Zhang-Suen)"):
            gr.Markdown("⚠️ Proses thinning bersifat iteratif piksel-per-piksel, mohon tunggu beberapa saat.")
            with gr.Row():
                t11_img = gr.Image(label="Upload Image", type="numpy")
                t11_btn = gr.Button("Proses Thinning", variant="primary")
            with gr.Row():
                t11_before = gr.Image(label="Hasil Sebelum Thinning (Binary)")
                t11_after = gr.Image(label="Hasil Sesudah Thinning (Skeleton)")
            with gr.Row():
                t11_hist_b = gr.Image(label="Histogram Sebelum")
                t11_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t11_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t11_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t11_meta = gr.Textbox(label="Metadata", lines=7)
            t11_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t11_btn.click(ui_skeleton, [t11_img],
                          [t11_before, t11_after, t11_hist_b, t11_hist_a,
                           t11_mat_b, t11_mat_a, t11_meta, t11_log])

        # ===================== TAB 12: FORENSIC RECONSTRUCTION LAB =====================
        with gr.Tab("12. Forensic Reconstruction Lab"):
            gr.Markdown(
                """
                ### 🧪 Fitur Utama: Rekonstruksi Citra Buram
                Gunakan gambar & parameter pada **Sidebar Global** di atas, lalu pilih mode:
                - **Mode Berantai (Pipeline)**: Grayscale → Flip/Rotate → Scaling → Contrast →
                  Smoothing → Edge Detection → Thresholding → Morfologi → Thinning Zhang-Suen.
                - **Mode Mandiri (Single Filter)**: filter dipilih hanya diterapkan ke gambar asli.
                """
            )
            run_btn = gr.Button("🚀 Jalankan Rekonstruksi", variant="primary", size="lg")

            stage_gallery = gr.Gallery(label="Visualisasi Setiap Tahap (Original + Tiap Step)",
                                       columns=4, height=400)
            final_out = gr.Image(label="Final Reconstruction Result")

            with gr.Row():
                lab_hist_b = gr.Image(label="Histogram Original")
                lab_hist_a = gr.Image(label="Histogram Hasil Akhir")

            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    lab_mat_b = gr.Textbox(label="Matrix Sebelum (10x10)", lines=11, elem_id="matrix_box")
                    lab_mat_a = gr.Textbox(label="Matrix Sesudah (10x10)", lines=11, elem_id="matrix_box")
                lab_meta = gr.Textbox(label="Metadata Hasil Akhir", lines=7)

            lab_log = gr.Textbox(label="Processing Log (seluruh tahap)", lines=16, elem_id="log_box")
            lab_analysis = gr.Textbox(label="Analisis Forensik Otomatis", lines=8)

            with gr.Row():
                export_img_btn = gr.Button("💾 Export Hasil Gambar (PNG)")
                export_pdf_btn = gr.Button("📄 Export Laporan Proses (PDF)")
            with gr.Row():
                export_img_file = gr.File(label="File Gambar Hasil")
                export_pdf_file = gr.File(label="File Laporan PDF")

            run_btn.click(
                run_forensic_lab,
                [global_image, mode_select, single_choice, p_flip, p_scale, p_contrast,
                 p_smooth_method, p_smooth_k, p_edge_method, p_thresh_method, p_morph_op, p_morph_k],
                [stage_gallery, final_out, lab_hist_b, lab_hist_a, lab_mat_b, lab_mat_a,
                 lab_meta, lab_log, lab_analysis]
            )

            export_img_btn.click(export_image, [final_out], [export_img_file])
            export_pdf_btn.click(
                export_pdf_report,
                [global_image, final_out, lab_log, lab_analysis, lab_meta],
                [export_pdf_file]
            )

    gr.Markdown(
        """
        ---
        *Dibuat untuk keperluan akademik — Tugas Mata Kuliah Pola dan Pengolahan Citra.*
        """
    )


if __name__ == "__main__":
    # share=False secara default; ubah ke True jika ingin link publik sementara (mis. di Google Colab)
    demo.queue().launch(share=False)

/tmp/ipykernel_6494/3316848911.py:873: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Forensik & Rekonstruksi Citra Buram", css=CUSTOM_CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
pip install --upgrade pillow gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency reso

In [ ]:
"""
===============================================================================
 APLIKASI FORENSIK DAN REKONSTRUKSI CITRA BURAM
 Mata Kuliah  : Pola dan Pengolahan Citra
 Materi       : Pertemuan 1 - 12 (Operasi Dasar s/d Skeletonization)
 Teknologi    : Python, OpenCV, NumPy, Matplotlib, Gradio
===============================================================================

ARSITEKTUR SISTEM (ringkas, lihat juga README yang dihasilkan terpisah):

    [Upload Gambar] --> [State Manager] --> [Engine Pengolahan Citra]
                                        |
                    -----------------------------------------------------
                    |             |             |             |        |
              Operasi Dasar  Aritmatika   Geometri/Scaling  Filter   Morfologi
                    |             |             |             |        |
                    -----------------------------------------------------
                                                |
                 [Modul Visualisasi: Histogram, Matriks Piksel, Metadata, Log]
                                                |
                 [Mode Pipeline]  atau  [Mode Single Filter]
                                                |
                 [Forensic Reconstruction Lab]

Semua fungsi pengolahan citra murni (pure function): menerima numpy array RGB
uint8, mengembalikan numpy array RGB/Gray uint8.
Tidak ada state tersembunyi sehingga mudah diuji & dipresentasikan.
===============================================================================
"""

import os
import io
import datetime
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
from PIL import Image

# =============================================================================
# BAGIAN 0. UTILITAS UMUM
# =============================================================================

OUTPUT_DIR = "exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def to_rgb3(img):
    """Konversi ke RGB 3-channel uint8 yang valid untuk Gradio."""
    if img is None:
        return np.zeros((100, 100, 3), dtype=np.uint8)

    # Konversi ke numpy jika belum
    img = np.array(img)

    # Jika grayscale (2D)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    # Jika RGBA (4 channel)
    elif img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    # Jika grayscale 3D (misal: 100x100x1)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # Pastikan uint8 dan C-Contiguous
    return np.ascontiguousarray(img, dtype=np.uint8)


def prepare_for_gradio(img):
    """Helper untuk memastikan output benar-benar siap untuk komponen Gradio."""
    return to_rgb3(img)

def to_gray(img):
    """Konversi citra RGB/Gray ke grayscale (selalu mengembalikan 2D array)."""
    img = np.asarray(img)
    if img.ndim == 2:
        return img.astype(np.uint8)
    return cv2.cvtColor(to_rgb3(img), cv2.COLOR_RGB2GRAY)


def get_metadata(img):
    """Mengembalikan string metadata citra: ukuran, channel, statistik piksel."""
    if img is None:
        return "Belum ada gambar yang diunggah."
    arr = np.asarray(img)
    h, w = arr.shape[0], arr.shape[1]
    c = 1 if arr.ndim == 2 else arr.shape[2]
    flat = arr.astype(np.float32)
    text = (
        f"METADATA CITRA\n"
        f"--------------------------------\n"
        f"Width            : {w} px\n"
        f"Height           : {h} px\n"
        f"Channels         : {c}\n"
        f"Min Pixel        : {flat.min():.2f}\n"
        f"Max Pixel        : {flat.max():.2f}\n"
        f"Mean Pixel       : {flat.mean():.2f}\n"
        f"Std Pixel        : {flat.std():.2f}\n"
    )
    return text


def pixel_matrix_10x10(img):
    """
    Mengambil sub-area 10x10 piksel dari bagian tengah citra (grayscale) dan
    menampilkannya sebagai matriks teks, karena menampilkan seluruh citra
    sebagai angka akan terlalu besar dan tidak terbaca.
    """
    if img is None:
        return "Tidak ada data piksel."
    gray = to_gray(img)
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    half = 5
    y0, y1 = max(0, cy - half), max(0, cy - half) + 10
    x0, x1 = max(0, cx - half), max(0, cx - half) + 10
    y1, x1 = min(h, y1), min(w, x1)
    patch = gray[y0:y1, x0:x1]

    rows = []
    for row in patch:
        rows.append("[" + " ".join(f"{v:3d}" for v in row) + "]")
    header = f"Matriks Piksel 10x10 (area tengah citra, baris {y0}-{y1}, kolom {x0}-{x1})\n"
    return header + "\n".join(rows)


def plot_histogram(img, title="Histogram"):
    """
    Membuat histogram intensitas dengan pendekatan buffer yang lebih aman
    untuk integrasi Gradio.
    """
    # 1. Pastikan gambar dalam format yang valid
    img = to_rgb3(img)

    # 2. Setup Plot
    fig, ax = plt.subplots(figsize=(4, 2.6), dpi=110)

    if img is None or img.size == 0:
        ax.text(0.5, 0.5, "Tidak ada data", ha="center", va="center")
        ax.axis("off")
    else:
        # Menampilkan histogram berdasarkan channel
        if img.ndim == 3:
            colors = ("r", "g", "b")
            for i, col in enumerate(colors):
                ax.hist(img[:, :, i].ravel(), bins=256, range=(0, 255),
                        color=col, alpha=0.5, label=col.upper())
            ax.legend(fontsize=7)
        else:
            ax.hist(img.ravel(), bins=256, range=(0, 255), color="gray")

        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Intensitas", fontsize=8)
        ax.set_ylabel("Frekuensi", fontsize=8)
        ax.tick_params(labelsize=7)

    fig.tight_layout()

    # 3. PENTING: Gunakan BytesIO dengan cara yang sangat aman
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches='tight')
    plt.close(fig) # Tutup figure segera setelah disimpan

    # 4. Ambil data dari buffer dan konversi menjadi array
    buf.seek(0)
    img_pil = Image.open(buf)
    out = np.array(img_pil.convert("RGB"), dtype=np.uint8)
    buf.close() # Tutup buffer untuk membersihkan memori

    return np.ascontiguousarray(out)


def log_line(step, title, **kwargs):
    """Membuat satu blok teks Processing Log yang konsisten formatnya."""
    text = f"STEP {step}:\n{title}\n"
    for k, v in kwargs.items():
        text += f"{k}:\n{v}\n"
    text += "\n"
    return text


def stat_block(img):
    """Statistik singkat sebuah citra: resolusi, rentang piksel, mean intensitas."""
    arr = np.asarray(img).astype(np.float32)
    h, w = arr.shape[0], arr.shape[1]
    return {
        "Resolution": f"{w} x {h}",
        "Pixel Range": f"{arr.min():.0f} - {arr.max():.0f}",
        "Mean Intensity": f"{arr.mean():.2f}",
    }


def safe_input_check(img):
    """Validasi sederhana: pastikan gambar sudah diunggah sebelum diproses."""
    if img is None:
        raise gr.Error("Silakan unggah gambar terlebih dahulu.")
    return to_rgb3(img)


# =============================================================================
# BAGIAN 1. PERTEMUAN 1 - OPERASI DASAR CITRA
# =============================================================================

def op_grayscale(img):
    g = to_gray(img)
    return to_rgb3(g)


def op_rgb_channel(img, channel):
    rgb = to_rgb3(img)
    out = np.zeros_like(rgb)
    idx = {"Red": 0, "Green": 1, "Blue": 2}[channel]
    out[:, :, idx] = rgb[:, :, idx]
    return out


def op_negative(img):
    rgb = to_rgb3(img)
    return 255 - rgb


# =============================================================================
# BAGIAN 2. PERTEMUAN 2 - OPERASI ARITMATIKA
# =============================================================================

def op_arithmetic(img, operation, value):
    rgb = to_rgb3(img).astype(np.float32)
    value = float(value)
    if operation == "Penjumlahan":
        out = rgb + value
    elif operation == "Pengurangan":
        out = rgb - value
    elif operation == "Perkalian":
        out = rgb * value
    elif operation == "Pembagian":
        out = rgb / (value if value != 0 else 1)
    else:
        out = rgb
    return np.clip(out, 0, 255).astype(np.uint8)


# =============================================================================
# BAGIAN 3. PERTEMUAN 3 - TRANSFORMASI GEOMETRI
# =============================================================================

def op_geometry(img, operation, custom_angle=0):
    rgb = to_rgb3(img)
    if operation == "Flip Horizontal":
        return cv2.flip(rgb, 1)
    if operation == "Flip Vertical":
        return cv2.flip(rgb, 0)
    if operation == "Rotate 90":
        return cv2.rotate(rgb, cv2.ROTATE_90_CLOCKWISE)
    if operation == "Rotate 180":
        return cv2.rotate(rgb, cv2.ROTATE_180)
    if operation == "Rotate 270":
        return cv2.rotate(rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if operation == "Custom Rotation":
        h, w = rgb.shape[:2]
        m = cv2.getRotationMatrix2D((w / 2, h / 2), float(custom_angle), 1.0)
        return cv2.warpAffine(rgb, m, (w, h))
    return rgb


# =============================================================================
# BAGIAN 4. PERTEMUAN 4 - SCALING
# =============================================================================

def op_scaling(img, method, factor):
    rgb = to_rgb3(img)
    factor = max(0.1, float(factor))
    h, w = rgb.shape[:2]
    new_w, new_h = max(1, int(w * factor)), max(1, int(h * factor))
    interp = cv2.INTER_NEAREST if method == "Nearest Neighbor" else cv2.INTER_LINEAR
    return cv2.resize(rgb, (new_w, new_h), interpolation=interp)


# =============================================================================
# BAGIAN 5. PERTEMUAN 5 - BRIGHTNESS & CONTRAST
# =============================================================================

def op_brightness_contrast(img, brightness, contrast):
    rgb = to_rgb3(img).astype(np.float32)
    # contrast: faktor pengali di sekitar titik tengah 128, brightness: offset aditif
    out = (rgb - 128) * float(contrast) + 128 + float(brightness)
    return np.clip(out, 0, 255).astype(np.uint8)


# =============================================================================
# BAGIAN 6. PERTEMUAN 6 - SMOOTHING FILTER
# =============================================================================

def op_smoothing(img, method, ksize):
    rgb = to_rgb3(img)
    k = int(ksize)
    if k % 2 == 0:
        k += 1
    if method == "Mean Filter":
        return cv2.blur(rgb, (k, k))
    if method == "Gaussian Filter":
        return cv2.GaussianBlur(rgb, (k, k), 0)
    if method == "Median Filter":
        return cv2.medianBlur(rgb, k)
    return rgb


# =============================================================================
# BAGIAN 7. PERTEMUAN 7 - SHARPENING
# =============================================================================

SHARPEN_KERNELS = {
    "Sharpen Kernel 1": np.array([[0, -1, 0],
                                   [-1, 5, -1],
                                   [0, -1, 0]]),
    "Sharpen Kernel 2": np.array([[-1, -1, -1],
                                   [-1, 9, -1],
                                   [-1, -1, -1]]),
    "Sharpen Kernel 3": np.array([[1, -2, 1],
                                   [-2, 5, -2],
                                   [1, -2, 1]]),
}


def op_sharpen(img, kernel_name):
    rgb = to_rgb3(img)
    kernel = SHARPEN_KERNELS[kernel_name]
    return cv2.filter2D(rgb, -1, kernel)


# =============================================================================
# BAGIAN 8. PERTEMUAN 8 - EDGE DETECTION
# =============================================================================

def op_edge(img, method):
    gray = to_gray(img)
    if method == "Sobel":
        sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        out = cv2.magnitude(sx, sy)
    elif method == "Prewitt":
        kx = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=np.float32)
        ky = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)
        sx = cv2.filter2D(gray.astype(np.float32), -1, kx)
        sy = cv2.filter2D(gray.astype(np.float32), -1, ky)
        out = cv2.magnitude(sx, sy)
    elif method == "Roberts":
        kx = np.array([[1, 0], [0, -1]], dtype=np.float32)
        ky = np.array([[0, 1], [-1, 0]], dtype=np.float32)
        sx = cv2.filter2D(gray.astype(np.float32), -1, kx)
        sy = cv2.filter2D(gray.astype(np.float32), -1, ky)
        out = cv2.magnitude(sx, sy)
    elif method == "Canny":
        out = cv2.Canny(gray, 100, 200).astype(np.float32)
    else:
        out = gray.astype(np.float32)
    out = np.clip(out, 0, 255).astype(np.uint8)
    return to_rgb3(out)


# =============================================================================
# BAGIAN 9. PERTEMUAN 9 - THRESHOLDING
# =============================================================================

def op_threshold(img, method, thresh_val=127):
    gray = to_gray(img)
    t = int(thresh_val)
    if method == "Binary":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
    elif method == "Binary Inverse":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY_INV)
    elif method == "Trunc":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_TRUNC)
    elif method == "To Zero":
        _, out = cv2.threshold(gray, t, 255, cv2.THRESH_TOZERO)
    elif method == "Adaptive Mean":
        out = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                     cv2.THRESH_BINARY, 11, 2)
    elif method == "Adaptive Gaussian":
        out = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, 2)
    elif method == "Otsu":
        _, out = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        out = gray
    return to_rgb3(out)


# =============================================================================
# BAGIAN 10. PERTEMUAN 10 - MORFOLOGI CITRA
# =============================================================================

def op_morphology(img, operation, ksize):
    gray = to_gray(img)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = int(ksize)
    kernel = np.ones((k, k), np.uint8)
    if operation == "Erosion":
        out = cv2.erode(binary, kernel, iterations=1)
    elif operation == "Dilation":
        out = cv2.dilate(binary, kernel, iterations=1)
    elif operation == "Opening":
        out = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    elif operation == "Closing":
        out = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    else:
        out = binary
    return to_rgb3(out)


# =============================================================================
# BAGIAN 11. PERTEMUAN 11 - SKELETONIZATION (ZHANG-SUEN THINNING)
# =============================================================================

def zhang_suen_thinning(binary_img):
    """
    Implementasi murni Zhang-Suen thinning algorithm.
    Input  : binary image (0/255) numpy array 2D
    Output : binary image hasil skeleton (0/255)

    NB: Untuk performa, citra besar di-resize sementara ke maksimal 300px
    pada sisi terpanjang sebelum thinning, lalu dikembalikan ke ukuran asli.
    """
    img = (binary_img > 0).astype(np.uint8)
    h, w = img.shape
    scale = 1.0
    max_side = max(h, w)
    if max_side > 300:
        scale = 300.0 / max_side
        img = cv2.resize(img, (int(w * scale), int(h * scale)),
                          interpolation=cv2.INTER_NEAREST)

    def neighbours(x, y, image):
        return [image[x - 1, y], image[x - 1, y + 1], image[x, y + 1], image[x + 1, y + 1],
                image[x + 1, y], image[x + 1, y - 1], image[x, y - 1], image[x - 1, y - 1]]

    def transitions(nb):
        n = nb + nb[0:1]
        return sum((n[i] == 0 and n[i + 1] == 1) for i in range(8))

    changing1 = changing2 = [(-1, -1)]
    while changing1 or changing2:
        rows, cols = img.shape
        changing1 = []
        for x in range(1, rows - 1):
            for y in range(1, cols - 1):
                p2, p3, p4, p5, p6, p7, p8, p9 = neighbours(x, y, img)
                if (img[x, y] == 1 and 2 <= sum([p2, p3, p4, p5, p6, p7, p8, p9]) <= 6
                        and transitions([p2, p3, p4, p5, p6, p7, p8, p9]) == 1
                        and p2 * p4 * p6 == 0 and p4 * p6 * p8 == 0):
                    changing1.append((x, y))
        for x, y in changing1:
            img[x, y] = 0

        changing2 = []
        for x in range(1, rows - 1):
            for y in range(1, cols - 1):
                p2, p3, p4, p5, p6, p7, p8, p9 = neighbours(x, y, img)
                if (img[x, y] == 1 and 2 <= sum([p2, p3, p4, p5, p6, p7, p8, p9]) <= 6
                        and transitions([p2, p3, p4, p5, p6, p7, p8, p9]) == 1
                        and p2 * p4 * p8 == 0 and p2 * p6 * p8 == 0):
                    changing2.append((x, y))
        for x, y in changing2:
            img[x, y] = 0

    if scale != 1.0:
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_NEAREST)
    return (img * 255).astype(np.uint8)


def op_skeleton(img):
    gray = to_gray(img)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    skeleton = zhang_suen_thinning(binary)
    return to_rgb3(binary), to_rgb3(skeleton)


# =============================================================================
# BAGIAN 12. PIPELINE LENGKAP (MODE BERANTAI) UNTUK FORENSIC LAB
# =============================================================================

PIPELINE_STEPS = [
    "Grayscale Conversion",
    "Flipping / Rotation",
    "Scaling",
    "Contrast Enhancement",
    "Smoothing Filter",
    "Edge Detection",
    "Thresholding",
    "Morphological Operation",
    "Thinning Zhang-Suen",
]


def run_pipeline(img, flip_choice, scale_factor, contrast_val, smooth_method,
                 smooth_k, edge_method, thresh_method, morph_op, morph_k):
    """
    Mode Berantai (Pipeline): output tahap sebelumnya menjadi input tahap
    berikutnya. Mengembalikan list of (nama_tahap, gambar, log_text).
    """
    rgb = safe_input_check(img)
    results = []
    log_text = ""

    # STEP 1: Grayscale
    cur = op_grayscale(rgb)
    s = stat_block(cur)
    log_text += log_line(1, "Grayscale Applied", **s)
    results.append(("Grayscale Conversion", cur))

    # STEP 2: Flip/Rotation
    cur = op_geometry(cur, flip_choice)
    s = stat_block(cur)
    log_text += log_line(2, f"{flip_choice} Applied", **s)
    results.append(("Flipping / Rotation", cur))

    # STEP 3: Scaling
    cur = op_scaling(cur, "Bilinear", scale_factor)
    s = stat_block(cur)
    log_text += log_line(3, "Scaling Applied", **{**s, "Factor": f"{scale_factor}x"})
    results.append(("Scaling", cur))

    # STEP 4: Contrast Enhancement
    cur = op_brightness_contrast(cur, 0, contrast_val)
    s = stat_block(cur)
    log_text += log_line(4, "Contrast Enhancement Applied", **{**s, "Contrast Factor": contrast_val})
    results.append(("Contrast Enhancement", cur))

    # STEP 5: Smoothing
    cur = op_smoothing(cur, smooth_method, smooth_k)
    s = stat_block(cur)
    log_text += log_line(5, f"{smooth_method} Applied", **{**s, "Kernel": f"{smooth_k}x{smooth_k}"})
    results.append(("Smoothing Filter", cur))

    # STEP 6: Edge Detection
    cur = op_edge(cur, edge_method)
    s = stat_block(cur)
    log_text += log_line(6, f"{edge_method} Edge Detection Applied", **s)
    results.append(("Edge Detection", cur))

    # STEP 7: Thresholding
    cur = op_threshold(cur, thresh_method)
    s = stat_block(cur)
    log_text += log_line(7, f"{thresh_method} Thresholding Applied", **s)
    results.append(("Thresholding", cur))

    # STEP 8: Morphology
    cur = op_morphology(cur, morph_op, morph_k)
    s = stat_block(cur)
    log_text += log_line(8, f"{morph_op} Applied", **{**s, "Kernel": f"{morph_k}x{morph_k}"})
    results.append(("Morphological Operation", cur))

    # STEP 9: Thinning
    gray_cur = to_gray(cur)
    _, binary_cur = cv2.threshold(gray_cur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    skel = zhang_suen_thinning(binary_cur)
    cur = to_rgb3(skel)
    s = stat_block(cur)
    log_text += log_line(9, "Zhang-Suen Thinning Applied", **s)
    results.append(("Thinning Zhang-Suen", cur))

    return results, log_text


def forensic_analysis_text(rgb_original, final_image, pipeline_results):
    """
    Membuat narasi analisis forensik otomatis berdasarkan perbandingan
    statistik antar tahap (edge density, noise level, dsb).
    """
    orig_gray = to_gray(rgb_original).astype(np.float32)
    final_gray = to_gray(final_image).astype(np.float32)

    edge_before = cv2.Laplacian(orig_gray.astype(np.uint8), cv2.CV_64F).var()
    # cari hasil tahap smoothing & edge & threshold untuk narasi lebih kaya
    stage_dict = {name: img for name, img in pipeline_results}

    notes = []

    if "Smoothing Filter" in stage_dict:
        before_std = orig_gray.std()
        after_std = to_gray(stage_dict["Smoothing Filter"]).astype(np.float32).std()
        noise_reduction = max(0.0, (before_std - after_std) / (before_std + 1e-6) * 100)
        notes.append(f"Noise berkurang sekitar {noise_reduction:.1f}% setelah tahap Smoothing Filter.")

    if "Edge Detection" in stage_dict:
        edge_after = cv2.Laplacian(to_gray(stage_dict["Edge Detection"]), cv2.CV_64F).var()
        edge_change = ((edge_after - edge_before) / (edge_before + 1e-6)) * 100
        notes.append(f"Edge detail berubah sekitar {edge_change:.1f}% setelah proses Edge Detection "
                     f"dibandingkan citra asli.")

    if "Thresholding" in stage_dict:
        notes.append("Objek pada citra semakin teridentifikasi (kontur lebih tegas) "
                      "setelah tahap Thresholding diterapkan.")

    if "Morphological Operation" in stage_dict:
        notes.append("Struktur objek dirapikan dan noise kecil dihilangkan setelah tahap Operasi Morfologi.")

    if "Thinning Zhang-Suen" in stage_dict:
        notes.append("Kerangka (skeleton) objek berhasil diekstraksi pada tahap akhir Thinning Zhang-Suen, "
                     "sehingga pola/bentuk dasar objek dapat dikenali kembali.")

    final_mean_diff = final_gray.mean() - orig_gray.mean()
    notes.append(f"Rata-rata intensitas piksel berubah sebesar {final_mean_diff:.2f} "
                 f"dari citra asli ke hasil rekonstruksi akhir.")

    header = "ANALISIS FORENSIK OTOMATIS\n" + "-" * 32 + "\n"
    return header + "\n".join(f"- {n}" for n in notes)


# =============================================================================
# BAGIAN 13. EXPORT GAMBAR & EXPORT LAPORAN PDF (TELAH DIHAPUS)
# =============================================================================
# Fitur export telah dihapus dari sistem.

# =============================================================================
# BAGIAN 14. WRAPPER GENERIK UNTUK TAB "MODE MANDIRI" (SINGLE FILTER LAB)
# Setiap fungsi wrapper mengembalikan:
#   gambar_output, hist_before, hist_after, matrix_before, matrix_after,
#   metadata_text, log_text
# =============================================================================

def generic_wrap(img, processed, step_title, extra_stats=None):
    rgb = safe_input_check(img)
    hist_before = plot_histogram(rgb, "Histogram Sebelum")
    hist_after = plot_histogram(processed, "Histogram Sesudah")
    matrix_before = pixel_matrix_10x10(rgb)
    matrix_after = pixel_matrix_10x10(processed)
    metadata = get_metadata(processed)
    s = stat_block(processed)
    if extra_stats:
        s.update(extra_stats)
    log_text = log_line(1, step_title, **s)
    return processed, hist_before, hist_after, matrix_before, matrix_after, metadata, log_text


# Pastikan fungsi pengolahan Anda memiliki return yang konsisten
def ui_grayscale(img):
    rgb = safe_input_check(img)
    out = op_grayscale(rgb)
    # Harus mengembalikan 7 variabel sesuai dengan definisi output di bawah
    return generic_wrap(img, out, "Grayscale Applied")

# Pastikan klik handler memiliki jumlah output yang sama
t1_btn.click(
    ui_grayscale,
    inputs=[t1_img],
    outputs=[t1_out, t1_hist_b, t1_hist_a, t1_mat_b, t1_mat_a, t1_meta, t1_log]
)


def ui_rgb_channel(img, channel):
    rgb = safe_input_check(img)
    out = op_rgb_channel(rgb, channel)
    return generic_wrap(img, out, f"RGB Channel ({channel}) Applied")


def ui_negative(img):
    rgb = safe_input_check(img)
    out = op_negative(rgb)
    return generic_wrap(img, out, "Negative Image Applied")


def ui_arithmetic(img, operation, value):
    rgb = safe_input_check(img)
    out = op_arithmetic(rgb, operation, value)
    return generic_wrap(img, out, f"Operasi Aritmatika: {operation}", {"Value": value})


def ui_geometry(img, operation, angle):
    rgb = safe_input_check(img)
    out = op_geometry(rgb, operation, angle)
    return generic_wrap(img, out, f"{operation} Applied")


def ui_scaling(img, method, factor):
    rgb = safe_input_check(img)
    out = op_scaling(rgb, method, factor)
    return generic_wrap(img, out, f"Scaling ({method}) Applied", {"Factor": f"{factor}x"})


def ui_brightness_contrast(img, brightness, contrast):
    rgb = safe_input_check(img)
    out = op_brightness_contrast(rgb, brightness, contrast)
    return generic_wrap(img, out, "Brightness & Contrast Applied",
                         {"Brightness": brightness, "Contrast": contrast})


def ui_smoothing(img, method, ksize):
    rgb = safe_input_check(img)
    out = op_smoothing(rgb, method, ksize)
    return generic_wrap(img, out, f"{method} Applied", {"Kernel": f"{ksize}x{ksize}"})


def ui_sharpen(img, kernel_name):
    rgb = safe_input_check(img)
    out = op_sharpen(rgb, kernel_name)
    return generic_wrap(img, out, f"{kernel_name} Applied")


def ui_edge(img, method):
    rgb = safe_input_check(img)
    out = op_edge(rgb, method)
    return generic_wrap(img, out, f"{method} Edge Detection Applied")


def ui_threshold(img, method, t):
    rgb = safe_input_check(img)
    out = op_threshold(rgb, method, t)
    return generic_wrap(img, out, f"{method} Thresholding Applied", {"Threshold": t})


def ui_morphology(img, operation, ksize):
    rgb = safe_input_check(img)
    out = op_morphology(rgb, operation, ksize)
    return generic_wrap(img, out, f"{operation} Applied", {"Kernel": f"{ksize}x{ksize}"})


def ui_skeleton(img):
    rgb = safe_input_check(img)
    binary, skeleton = op_skeleton(rgb)
    hist_before = plot_histogram(binary, "Histogram Sebelum Thinning")
    hist_after = plot_histogram(skeleton, "Histogram Sesudah Thinning")
    matrix_before = pixel_matrix_10x10(binary)
    matrix_after = pixel_matrix_10x10(skeleton)
    metadata = get_metadata(skeleton)
    s = stat_block(skeleton)
    log_text = log_line(1, "Zhang-Suen Thinning Applied", **s)
    return binary, skeleton, hist_before, hist_after, matrix_before, matrix_after, metadata, log_text


# =============================================================================
# BAGIAN 15. FORENSIC RECONSTRUCTION LAB (TAB UTAMA) + MODE BERANTAI/MANDIRI
# =============================================================================

SINGLE_FILTER_CHOICES = [
    "Grayscale", "Negative", "Flip Horizontal", "Flip Vertical", "Rotate 90",
    "Scaling 2x (Bilinear)", "Brightness +40", "Gaussian Smoothing 5x5",
    "Sharpen Kernel 2", "Sobel Edge", "Canny Edge", "Otsu Threshold",
    "Dilation", "Erosion", "Zhang-Suen Thinning",
]


def apply_single_filter(rgb, choice):
    """Mode Mandiri: setiap operasi diterapkan langsung ke gambar ASLI, tidak berantai."""
    if choice == "Grayscale":
        return op_grayscale(rgb)
    if choice == "Negative":
        return op_negative(rgb)
    if choice == "Flip Horizontal":
        return op_geometry(rgb, "Flip Horizontal")
    if choice == "Flip Vertical":
        return op_geometry(rgb, "Flip Vertical")
    if choice == "Rotate 90":
        return op_geometry(rgb, "Rotate 90")
    if choice == "Scaling 2x (Bilinear)":
        return op_scaling(rgb, "Bilinear", 2.0)
    if choice == "Brightness +40":
        return op_brightness_contrast(rgb, 40, 1.0)
    if choice == "Gaussian Smoothing 5x5":
        return op_smoothing(rgb, "Gaussian Filter", 5)
    if choice == "Sharpen Kernel 2":
        return op_sharpen(rgb, "Sharpen Kernel 2")
    if choice == "Sobel Edge":
        return op_edge(rgb, "Sobel")
    if choice == "Canny Edge":
        return op_edge(rgb, "Canny")
    if choice == "Otsu Threshold":
        return op_threshold(rgb, "Otsu")
    if choice == "Dilation":
        return op_morphology(rgb, "Dilation", 3)
    if choice == "Erosion":
        return op_morphology(rgb, "Erosion", 3)
    if choice == "Zhang-Suen Thinning":
        _, skel = op_skeleton(rgb)
        return skel
    return rgb


def run_forensic_lab(img, mode, single_choice, flip_choice, scale_factor, contrast_val,
                     smooth_method, smooth_k, edge_method, thresh_method, morph_op, morph_k):

    # Pastikan input sudah di-prepare
    rgb = prepare_for_gradio(safe_input_check(img))

    if mode == "Mode Mandiri (Single Filter)":
        final = apply_single_filter(rgb, single_choice)
        final = prepare_for_gradio(final) # Konversi ke format aman
        s = stat_block(final)
        log_text = log_line(1, f"Single Filter: {single_choice}", **s)
        # Pastikan list tuple untuk Gallery berisi array yang sudah diprepare
        gallery = [(rgb, "Original"), (final, single_choice)]
        analysis = forensic_analysis_text(rgb, final, [(single_choice, final)])

    else:
        results, log_text = run_pipeline(
            rgb, flip_choice, scale_factor, contrast_val, smooth_method,
            smooth_k, edge_method, thresh_method, morph_op, morph_k
        )

        # Bersihkan hasil pipeline ke format Gradio
        processed_results = [(name, prepare_for_gradio(im)) for name, im in results]
        final = processed_results[-1][1]

        gallery = [(rgb, "Original")] + [(im, name) for name, im in processed_results]
        analysis = forensic_analysis_text(rgb, final, processed_results)

    # Histogram, Matriks, Metadata tetap sama (atau gunakan prepare_for_gradio jika perlu)
    hist_before = plot_histogram(rgb, "Histogram Original")
    hist_after = plot_histogram(final, "Histogram Hasil Akhir")
    matrix_before = pixel_matrix_10x10(rgb)
    matrix_after = pixel_matrix_10x10(final)
    metadata = get_metadata(final)

    return (gallery, final, hist_before, hist_after, matrix_before, matrix_after,
            metadata, log_text, analysis)


# =============================================================================
# BAGIAN 16. ANTARMUKA GRADIO (UI)
# =============================================================================

CUSTOM_CSS = """
#log_box textarea {font-family: monospace; font-size: 12px;}
#matrix_box textarea {font-family: monospace; font-size: 11px;}
"""

with gr.Blocks(title="Forensik & Rekonstruksi Citra Buram", css=CUSTOM_CSS) as demo:
    gr.Markdown(
        """
        # 🔍 Aplikasi Forensik dan Rekonstruksi Citra Buram
        **Mata Kuliah Pola dan Pengolahan Citra** — Laboratorium interaktif untuk seluruh
        materi Pertemuan 1–12, sekaligus sistem restorasi citra blur secara bertahap.
        """
    )

    # ---------- SIDEBAR GLOBAL (Upload, Mode, Forensic Lab parameter) ----------
    with gr.Accordion("📁 Sidebar Global - Upload & Mode (untuk Tab 'Forensic Reconstruction Lab')", open=True):
        with gr.Row():
            with gr.Column(scale=1):
                global_image = gr.Image(label="Upload Image (Blur/Buram)", type="numpy")
                mode_select = gr.Radio(
                    ["Mode Berantai (Pipeline)", "Mode Mandiri (Single Filter)"],
                    value="Mode Berantai (Pipeline)", label="Mode Pemrosesan"
                )
                single_choice = gr.Dropdown(SINGLE_FILTER_CHOICES, value="Grayscale",
                                            label="Pilihan Filter (khusus Mode Mandiri)")
            with gr.Column(scale=1):
                gr.Markdown("**Parameter Pipeline (khusus Mode Berantai)**")
                p_flip = gr.Dropdown(["Flip Horizontal", "Flip Vertical", "Rotate 90",
                                      "Rotate 180", "Rotate 270"], value="Flip Horizontal",
                                     label="Tahap 2: Flip/Rotasi")
                p_scale = gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="Tahap 3: Scaling Factor")
                p_contrast = gr.Slider(0.5, 3.0, value=1.3, step=0.1, label="Tahap 4: Contrast Factor")
                p_smooth_method = gr.Dropdown(["Mean Filter", "Gaussian Filter", "Median Filter"],
                                              value="Gaussian Filter", label="Tahap 5: Smoothing")
                p_smooth_k = gr.Slider(3, 7, value=5, step=2, label="Tahap 5: Kernel Smoothing")
                p_edge_method = gr.Dropdown(["Sobel", "Prewitt", "Roberts", "Canny"],
                                            value="Canny", label="Tahap 6: Edge Detection")
                p_thresh_method = gr.Dropdown(["Binary", "Binary Inverse", "Trunc", "To Zero",
                                               "Adaptive Mean", "Adaptive Gaussian", "Otsu"],
                                              value="Otsu", label="Tahap 7: Thresholding")
                p_morph_op = gr.Dropdown(["Erosion", "Dilation", "Opening", "Closing"],
                                         value="Closing", label="Tahap 8: Morfologi")
                p_morph_k = gr.Slider(3, 9, value=3, step=2, label="Tahap 8: Kernel Morfologi")

    with gr.Tabs():
        # ===================== TAB 1: OPERASI DASAR =====================
        with gr.Tab("1. Operasi Dasar Citra"):
            with gr.Row():
                t1_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t1_op = gr.Radio(["Grayscale", "RGB Channel", "Negative Image"],
                                     value="Grayscale", label="Pilih Operasi")
                    t1_channel = gr.Dropdown(["Red", "Green", "Blue"], value="Red",
                                             label="Channel (khusus RGB Channel)")
                    t1_btn = gr.Button("Proses", variant="primary")
            with gr.Row():
                t1_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t1_hist_b = gr.Image(label="Histogram Sebelum")
                t1_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t1_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t1_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t1_meta = gr.Textbox(label="Metadata", lines=7)
            t1_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            def t1_run(img, op, ch):
                if op == "Grayscale":
                    return ui_grayscale(img)
                if op == "RGB Channel":
                    return ui_rgb_channel(img, ch)
                return ui_negative(img)

            t1_btn.click(t1_run, [t1_img, t1_op, t1_channel],
                         [t1_out, t1_hist_b, t1_hist_a, t1_mat_b, t1_mat_a, t1_meta, t1_log])

        # ===================== TAB 2: OPERASI ARITMATIKA =====================
        with gr.Tab("2. Operasi Aritmatika"):
            with gr.Row():
                t2_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t2_op = gr.Radio(["Penjumlahan", "Pengurangan", "Perkalian", "Pembagian"],
                                     value="Penjumlahan", label="Operasi")
                    t2_val = gr.Slider(1, 100, value=30, step=1, label="Nilai (scalar)")
                    t2_btn = gr.Button("Proses", variant="primary")
            with gr.Row():
                t2_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t2_hist_b = gr.Image(label="Histogram Sebelum")
                t2_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel (sebelum/sesudah) & Metadata", open=True):
                with gr.Row():
                    t2_mat_b = gr.Textbox(label="Matriks Sebelum (10x10)", lines=11, elem_id="matrix_box")
                    t2_mat_a = gr.Textbox(label="Matriks Sesudah (10x10)", lines=11, elem_id="matrix_box")
                t2_meta = gr.Textbox(label="Metadata", lines=7)
            t2_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t2_btn.click(ui_arithmetic, [t2_img, t2_op, t2_val],
                         [t2_out, t2_hist_b, t2_hist_a, t2_mat_b, t2_mat_a, t2_meta, t2_log])

        # ===================== TAB 3: TRANSFORMASI GEOMETRI =====================
        with gr.Tab("3. Transformasi Geometri"):
            with gr.Row():
                t3_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t3_op = gr.Radio(["Flip Horizontal", "Flip Vertical", "Rotate 90",
                                      "Rotate 180", "Rotate 270", "Custom Rotation"],
                                     value="Flip Horizontal", label="Operasi")
                    t3_angle = gr.Slider(0, 360, value=45, step=1, label="Sudut (Custom Rotation)")
                    t3_btn = gr.Button("Proses", variant="primary")
            t3_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t3_hist_b = gr.Image(label="Histogram Sebelum")
                t3_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t3_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t3_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t3_meta = gr.Textbox(label="Metadata", lines=7)
            t3_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t3_btn.click(ui_geometry, [t3_img, t3_op, t3_angle],
                         [t3_out, t3_hist_b, t3_hist_a, t3_mat_b, t3_mat_a, t3_meta, t3_log])

        # ===================== TAB 4: SCALING =====================
        with gr.Tab("4. Scaling"):
            with gr.Row():
                t4_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t4_method = gr.Radio(["Nearest Neighbor", "Bilinear"], value="Bilinear",
                                         label="Metode Interpolasi")
                    t4_factor = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="Faktor Skala")
                    t4_btn = gr.Button("Proses", variant="primary")
            t4_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t4_hist_b = gr.Image(label="Histogram Sebelum")
                t4_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t4_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t4_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t4_meta = gr.Textbox(label="Metadata", lines=7)
            t4_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t4_btn.click(ui_scaling, [t4_img, t4_method, t4_factor],
                         [t4_out, t4_hist_b, t4_hist_a, t4_mat_b, t4_mat_a, t4_meta, t4_log])

        # ===================== TAB 5: BRIGHTNESS & CONTRAST =====================
        with gr.Tab("5. Brightness & Contrast"):
            with gr.Row():
                t5_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t5_brightness = gr.Slider(-100, 100, value=0, step=1, label="Brightness")
                    t5_contrast = gr.Slider(0.2, 3.0, value=1.0, step=0.1, label="Contrast")
                    t5_btn = gr.Button("Proses", variant="primary")
            t5_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t5_hist_b = gr.Image(label="Histogram Sebelum")
                t5_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t5_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t5_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t5_meta = gr.Textbox(label="Metadata", lines=7)
            t5_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t5_btn.click(ui_brightness_contrast, [t5_img, t5_brightness, t5_contrast],
                         [t5_out, t5_hist_b, t5_hist_a, t5_mat_b, t5_mat_a, t5_meta, t5_log])

        # ===================== TAB 6: SMOOTHING FILTER =====================
        with gr.Tab("6. Smoothing Filter"):
            with gr.Row():
                t6_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t6_method = gr.Radio(["Mean Filter", "Gaussian Filter", "Median Filter"],
                                         value="Gaussian Filter", label="Metode")
                    t6_k = gr.Radio([3, 5, 7], value=5, label="Ukuran Kernel")
                    t6_btn = gr.Button("Proses", variant="primary")
            t6_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t6_hist_b = gr.Image(label="Histogram Sebelum")
                t6_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t6_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t6_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t6_meta = gr.Textbox(label="Metadata", lines=7)
            t6_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t6_btn.click(ui_smoothing, [t6_img, t6_method, t6_k],
                         [t6_out, t6_hist_b, t6_hist_a, t6_mat_b, t6_mat_a, t6_meta, t6_log])

        # ===================== TAB 7: SHARPENING =====================
        with gr.Tab("7. Sharpening"):
            with gr.Row():
                t7_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t7_kernel = gr.Radio(list(SHARPEN_KERNELS.keys()),
                                         value="Sharpen Kernel 1", label="Pilih Kernel")
                    t7_btn = gr.Button("Proses", variant="primary")
            t7_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t7_hist_b = gr.Image(label="Histogram Sebelum")
                t7_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t7_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t7_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t7_meta = gr.Textbox(label="Metadata", lines=7)
            t7_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t7_btn.click(ui_sharpen, [t7_img, t7_kernel],
                         [t7_out, t7_hist_b, t7_hist_a, t7_mat_b, t7_mat_a, t7_meta, t7_log])

        # ===================== TAB 8: EDGE DETECTION =====================
        with gr.Tab("8. Edge Detection"):
            with gr.Row():
                t8_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t8_method = gr.Radio(["Sobel", "Prewitt", "Roberts", "Canny"],
                                         value="Sobel", label="Metode")
                    t8_btn = gr.Button("Proses", variant="primary")
            t8_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t8_hist_b = gr.Image(label="Histogram Sebelum")
                t8_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t8_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t8_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t8_meta = gr.Textbox(label="Metadata", lines=7)
            t8_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t8_btn.click(ui_edge, [t8_img, t8_method],
                         [t8_out, t8_hist_b, t8_hist_a, t8_mat_b, t8_mat_a, t8_meta, t8_log])

        # ===================== TAB 9: THRESHOLDING =====================
        with gr.Tab("9. Thresholding"):
            with gr.Row():
                t9_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t9_method = gr.Radio(["Binary", "Binary Inverse", "Trunc", "To Zero",
                                          "Adaptive Mean", "Adaptive Gaussian", "Otsu"],
                                         value="Binary", label="Metode")
                    t9_thresh = gr.Slider(0, 255, value=127, step=1,
                                          label="Threshold Value (non-adaptive/non-otsu)")
                    t9_btn = gr.Button("Proses", variant="primary")
            t9_out = gr.Image(label="Gambar Output")
            with gr.Row():
                t9_hist_b = gr.Image(label="Histogram Sebelum")
                t9_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t9_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t9_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t9_meta = gr.Textbox(label="Metadata", lines=7)
            t9_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t9_btn.click(ui_threshold, [t9_img, t9_method, t9_thresh],
                         [t9_out, t9_hist_b, t9_hist_a, t9_mat_b, t9_mat_a, t9_meta, t9_log])

        # ===================== TAB 10: MORFOLOGI CITRA =====================
        with gr.Tab("10. Morfologi Citra"):
            with gr.Row():
                t10_img = gr.Image(label="Upload Image", type="numpy")
                with gr.Column():
                    t10_op = gr.Radio(["Erosion", "Dilation", "Opening", "Closing"],
                                      value="Erosion", label="Operasi")
                    t10_k = gr.Slider(3, 9, value=3, step=2, label="Ukuran Kernel")
                    t10_btn = gr.Button("Proses", variant="primary")
            t10_out = gr.Image(label="Gambar Output (citra dibinerisasi Otsu lalu dimorfologi)")
            with gr.Row():
                t10_hist_b = gr.Image(label="Histogram Sebelum")
                t10_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t10_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t10_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t10_meta = gr.Textbox(label="Metadata", lines=7)
            t10_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t10_btn.click(ui_morphology, [t10_img, t10_op, t10_k],
                          [t10_out, t10_hist_b, t10_hist_a, t10_mat_b, t10_mat_a, t10_meta, t10_log])

        # ===================== TAB 11: SKELETONIZATION =====================
        with gr.Tab("11. Skeletonization (Zhang-Suen)"):
            gr.Markdown("⚠️ Proses thinning bersifat iteratif piksel-per-piksel, mohon tunggu beberapa saat.")
            with gr.Row():
                t11_img = gr.Image(label="Upload Image", type="numpy")
                t11_btn = gr.Button("Proses Thinning", variant="primary")
            with gr.Row():
                t11_before = gr.Image(label="Hasil Sebelum Thinning (Binary)")
                t11_after = gr.Image(label="Hasil Sesudah Thinning (Skeleton)")
            with gr.Row():
                t11_hist_b = gr.Image(label="Histogram Sebelum")
                t11_hist_a = gr.Image(label="Histogram Sesudah")
            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    t11_mat_b = gr.Textbox(label="Matrix Sebelum", lines=11, elem_id="matrix_box")
                    t11_mat_a = gr.Textbox(label="Matrix Sesudah", lines=11, elem_id="matrix_box")
                t11_meta = gr.Textbox(label="Metadata", lines=7)
            t11_log = gr.Textbox(label="Processing Log", lines=6, elem_id="log_box")

            t11_btn.click(ui_skeleton, [t11_img],
                          [t11_before, t11_after, t11_hist_b, t11_hist_a,
                           t11_mat_b, t11_mat_a, t11_meta, t11_log])

        # ===================== TAB 12: FORENSIC RECONSTRUCTION LAB =====================
        with gr.Tab("12. Forensic Reconstruction Lab"):
            gr.Markdown(
                """
                ### 🧪 Fitur Utama: Rekonstruksi Citra Buram
                Gunakan gambar & parameter pada **Sidebar Global** di atas, lalu pilih mode:
                - **Mode Berantai (Pipeline)**: Grayscale → Flip/Rotate → Scaling → Contrast →
                  Smoothing → Edge Detection → Thresholding → Morfologi → Thinning Zhang-Suen.
                - **Mode Mandiri (Single Filter)**: filter dipilih hanya diterapkan ke gambar asli.
                """
            )
            run_btn = gr.Button("🚀 Jalankan Rekonstruksi", variant="primary", size="lg")

            stage_gallery = gr.Gallery(label="Visualisasi Setiap Tahap (Original + Tiap Step)",
                                       columns=4, height=400)
            final_out = gr.Image(label="Final Reconstruction Result")

            with gr.Row():
                lab_hist_b = gr.Image(label="Histogram Original")
                lab_hist_a = gr.Image(label="Histogram Hasil Akhir")

            with gr.Accordion("Matriks Piksel & Metadata", open=False):
                with gr.Row():
                    lab_mat_b = gr.Textbox(label="Matrix Sebelum (10x10)", lines=11, elem_id="matrix_box")
                    lab_mat_a = gr.Textbox(label="Matrix Sesudah (10x10)", lines=11, elem_id="matrix_box")
                lab_meta = gr.Textbox(label="Metadata Hasil Akhir", lines=7)

            lab_log = gr.Textbox(label="Processing Log (seluruh tahap)", lines=16, elem_id="log_box")
            lab_analysis = gr.Textbox(label="Analisis Forensik Otomatis", lines=8)

            run_btn.click(
                run_forensic_lab,
                [global_image, mode_select, single_choice, p_flip, p_scale, p_contrast,
                 p_smooth_method, p_smooth_k, p_edge_method, p_thresh_method, p_morph_op, p_morph_k],
                [stage_gallery, final_out, lab_hist_b, lab_hist_a, lab_mat_b, lab_mat_a,
                 lab_meta, lab_log, lab_analysis]
            )

    gr.Markdown(
        """
        ---
        *Dibuat untuk keperluan akademik — Tugas Mata Kuliah Pola dan Pengolahan Citra.*
        """
    )


if __name__ == "__main__":
    # share=False secara default; ubah ke True jika ingin link publik sementara (mis. di Google Colab)
    demo.queue().launch(share=False)

AttributeError: Cannot call click outside of a gradio.Blocks context.

In [ ]:
# forensic_restoration_dashboard.py
# Dashboard Restorasi dan Analisis Forensik Citra Buram
# Mata Kuliah: Pola dan Pengolahan Citra Digital
# Teknologi: Python, OpenCV, NumPy, Matplotlib, PIL, Gradio, Pandas

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Wajib untuk lingkungan tanpa GUI seperti Colab
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math

# ============================================================
# SECTION 1: UTILITY & METRIC FUNCTIONS
# ============================================================

def pil_to_cv2(pil_img):
    """Konversi PIL Image ke NumPy array BGR (OpenCV)."""
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    """Konversi NumPy array BGR ke PIL Image."""
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def gray_to_pil(gray_img):
    """Konversi grayscale NumPy array ke PIL Image."""
    return Image.fromarray(gray_img)

def compute_mse(original, processed):
    """Hitung Mean Squared Error antara dua citra."""
    original = original.astype(np.float64)
    processed = processed.astype(np.float64)
    if original.shape != processed.shape:
        processed = cv2.resize(processed, (original.shape[1], original.shape[0]))
    return float(np.mean((original - processed) ** 2))

def compute_psnr(mse):
    """Hitung PSNR dari nilai MSE."""
    if mse == 0:
        return float('inf')
    return float(20 * math.log10(255.0 / math.sqrt(mse)))

def compute_sharpness(gray_img):
    """Hitung Sharpness menggunakan Variance of Laplacian."""
    return float(cv2.Laplacian(gray_img, cv2.CV_64F).var())

def compute_contrast(gray_img):
    """Hitung Contrast menggunakan standar deviasi intensitas piksel."""
    return float(gray_img.std())

def compute_brightness(gray_img):
    """Hitung Brightness menggunakan rata-rata intensitas piksel."""
    return float(gray_img.mean())

def get_metrics(original_gray, processed_gray, label):
    """Kembalikan dictionary metrik untuk satu tahap."""
    mse = compute_mse(original_gray, processed_gray)
    psnr = compute_psnr(mse)
    sharpness = compute_sharpness(processed_gray)
    contrast = compute_contrast(processed_gray)
    brightness = compute_brightness(processed_gray)
    return {
        "Tahap": label,
        "MSE": round(mse, 4),
        "PSNR (dB)": round(psnr, 4) if psnr != float('inf') else "∞",
        "Sharpness": round(sharpness, 4),
        "Contrast": round(contrast, 4),
        "Brightness": round(brightness, 4),
    }

# ============================================================
# SECTION 2: PIPELINE PROCESSING FUNCTIONS
# ============================================================

def step_grayscale(bgr_img):
    """Konversi ke Grayscale."""
    return cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)

def step_median_filter(gray_img, ksize=5):
    """Terapkan Median Filter untuk reduksi noise."""
    return cv2.medianBlur(gray_img, ksize)

def step_clahe(gray_img, clip_limit=2.0, tile_grid=(8, 8)):
    """Terapkan CLAHE (Contrast Limited Adaptive Histogram Equalization)."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(gray_img)

def step_sharpening(gray_img):
    """Terapkan Sharpening menggunakan kernel unsharp mask."""
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(gray_img, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def step_rotate(gray_img, angle_str):
    """Rotasi citra sesuai pilihan."""
    angle_map = {
        "Tidak Ada": None,
        "90°": cv2.ROTATE_90_CLOCKWISE,
        "180°": cv2.ROTATE_180,
        "270°": cv2.ROTATE_90_COUNTERCLOCKWISE,
    }
    code = angle_map.get(angle_str, None)
    if code is not None:
        return cv2.rotate(gray_img, code)
    return gray_img

def step_zoom(gray_img, zoom_factor=1.0):
    """Zoom citra menggunakan resize OpenCV."""
    if zoom_factor == 1.0:
        return gray_img
    h, w = gray_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(gray_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    zoomed = zoomed[:h, :w] if zoomed.shape[0] >= h and zoomed.shape[1] >= w else \
             cv2.resize(zoomed, (w, h), interpolation=cv2.INTER_LINEAR)
    return zoomed

def step_edge_detection(gray_img):
    """Deteksi tepi menggunakan Canny Edge Detector."""
    return cv2.Canny(gray_img, threshold1=50, threshold2=150)

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(gray_img, title):
    """Buat histogram grayscale menggunakan Matplotlib, kembalikan PIL Image."""
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='teal', alpha=0.85)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#1a535c')
    ax.set_xlabel("Intensitas Piksel", fontsize=9)
    ax.set_ylabel("Frekuensi", fontsize=9)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.4)
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(gray_img, size=50):
    """Ambil area 50x50 kiri atas dan kembalikan sebagai DataFrame Pandas."""
    h, w = gray_img.shape
    region = gray_img[:min(size, h), :min(size, w)]
    df = pd.DataFrame(region)
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_gray, final_gray):
    """Buat gambar perbandingan Sebelum vs Sesudah berdampingan."""
    h1, w1 = original_gray.shape
    h2, w2 = final_gray.shape
    max_h = max(h1, h2)
    pad1 = np.zeros((max_h, w1), dtype=np.uint8)
    pad1[:h1, :w1] = original_gray
    pad2 = np.zeros((max_h, w2), dtype=np.uint8)
    pad2[:h2, :w2] = final_gray
    divider = np.ones((max_h, 4), dtype=np.uint8) * 80
    combined = np.hstack([pad1, divider, pad2])

    combined_bgr = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR)
    cv2.putText(combined_bgr, "SEBELUM", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 180), 2)
    cv2.putText(combined_bgr, "SESUDAH", (w1 + 14, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 180), 2)
    return cv2_to_pil(combined_bgr)

# ============================================================
# SECTION 4: FORENSIC SUMMARY GENERATOR
# ============================================================

def generate_forensic_summary(metrics_list):
    """Buat ringkasan forensik otomatis berdasarkan hasil metrik."""
    if not metrics_list:
        return "Tidak ada data metrik untuk dianalisis."

    lines = ["📋 **LAPORAN FORENSIK OTOMATIS**\n"]
    lines.append("---")

    grayscale_m = next((m for m in metrics_list if m["Tahap"] == "Grayscale"), None)
    clahe_m     = next((m for m in metrics_list if m["Tahap"] == "CLAHE"), None)
    sharp_m     = next((m for m in metrics_list if m["Tahap"] == "Sharpening"), None)
    edge_m      = next((m for m in metrics_list if m["Tahap"] == "Edge Detection"), None)

    if grayscale_m and sharp_m:
        s_before = grayscale_m["Sharpness"]
        s_after  = sharp_m["Sharpness"]
        diff = s_after - s_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🔍 **Ketajaman (Sharpness):** Setelah proses CLAHE dan Sharpening, "
                     f"nilai ketajaman {direction} dari **{s_before:.2f}** menjadi **{s_after:.2f}** "
                     f"(selisih: {abs(diff):.2f}).")

    if grayscale_m and clahe_m:
        c_before = grayscale_m["Contrast"]
        c_after  = clahe_m["Contrast"]
        diff = c_after - c_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🎨 **Kontras (Contrast):** Setelah CLAHE, kontras citra {direction} "
                     f"dari **{c_before:.2f}** menjadi **{c_after:.2f}**, "
                     f"sehingga detail objek lebih {'mudah' if diff > 0 else 'sulit'} dikenali.")

    if sharp_m:
        psnr_val = sharp_m["PSNR (dB)"]
        mse_val  = sharp_m["MSE"]
        lines.append(f"📡 **Kualitas Rekonstruksi:** Nilai PSNR pada tahap Sharpening = **{psnr_val} dB** "
                     f"dengan MSE = **{mse_val}__. "
                     + ("Kualitas restorasi sangat baik (PSNR > 30 dB)." if isinstance(psnr_val, (int, float)) and psnr_val > 30
                        else "Kualitas cukup baik."))

    if edge_m:
        lines.append(f"🔎 **Deteksi Tepi:** Brightness rata-rata pada tahap Edge Detection = "
                     f"**{edge_m['Brightness']:.2f}**, menunjukkan "
                     + ("tepi yang cukup dominan." if edge_m['Brightness'] > 20
                        else "tepi halus — citra relatif seragam."))

    lines.append("\n✅ **Kesimpulan:** Pipeline restorasi berhasil dieksekusi seluruhnya. "
                 "Citra buram berhasil diperbaiki melalui tahapan Median Filter, CLAHE, "
                 "dan Sharpening. Analisis forensik menunjukkan peningkatan kualitas visual "
                 "yang signifikan dibandingkan citra asli.")
    return "\n\n".join(lines)

# ============================================================
# SECTION 5: MAIN PIPELINE FUNCTION
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Fungsi utama pipeline. Menerima PIL Image, mengembalikan semua hasil."""
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(240, 248, 248))
        empty_df = pd.DataFrame()
        empty_hist = plot_histogram(np.zeros((100, 100), dtype=np.uint8), "Tidak Ada Data")
        metrics_df = pd.DataFrame()
        forensic = "Silakan upload citra terlebih dahulu."
        comparison = empty
        return (
            empty, empty, empty, empty, empty, empty,
            empty_hist, empty_hist, empty_hist, empty_hist, empty_hist,
            empty_df, empty_df, empty_df, empty_df, empty_df,
            metrics_df, forensic, comparison
        )

    bgr = pil_to_cv2(input_image)
    gray = step_grayscale(bgr)
    original_gray = gray.copy()

    median = step_median_filter(gray)
    clahe_img = step_clahe(median)
    sharp = step_sharpening(clahe_img)
    rotated = step_rotate(sharp, rotation_choice)
    zoomed = step_zoom(rotated, float(zoom_factor))
    edge = step_edge_detection(zoomed)

    pil_gray   = gray_to_pil(gray)
    pil_median = gray_to_pil(median)
    pil_clahe  = gray_to_pil(clahe_img)
    pil_sharp  = gray_to_pil(sharp)
    pil_edge   = gray_to_pil(edge)

    hist_gray   = plot_histogram(gray,      "Histogram Grayscale")
    hist_median = plot_histogram(median,    "Histogram Median Filter")
    hist_clahe  = plot_histogram(clahe_img, "Histogram CLAHE")
    hist_sharp  = plot_histogram(sharp,     "Histogram Sharpening")
    hist_edge   = plot_histogram(edge,      "Histogram Edge Detection")

    mat_gray   = get_matrix_dataframe(gray)
    mat_median = get_matrix_dataframe(median)
    mat_clahe  = get_matrix_dataframe(clahe_img)
    mat_sharp  = get_matrix_dataframe(sharp)
    mat_edge   = get_matrix_dataframe(edge)

    metrics_list = [
        get_metrics(original_gray, gray,      "Grayscale"),
        get_metrics(original_gray, median,    "Median Filter"),
        get_metrics(original_gray, clahe_img, "CLAHE"),
        get_metrics(original_gray, sharp,     "Sharpening"),
        get_metrics(original_gray, edge,      "Edge Detection"),
    ]
    metrics_df = pd.DataFrame(metrics_list)
    forensic_text = generate_forensic_summary(metrics_list)
    comparison_img = make_comparison_image(original_gray, sharp)

    return (
        pil_gray, pil_median, pil_clahe, pil_sharp, pil_edge,
        hist_gray, hist_median, hist_clahe, hist_sharp, hist_edge,
        mat_gray, mat_median, mat_clahe, mat_sharp, mat_edge,
        metrics_df, forensic_text, comparison_img
    )

# ============================================================
# SECTION 6: CUSTOM CSS
# ============================================================

CUSTOM_CSS = """
body, .gradio-container {
    background: #f0f7f7 !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
}
.app-header {
    background: linear-gradient(135deg, #1a535c, #4ecdc4);
    color: white;
    padding: 24px 32px;
    border-radius: 16px;
    margin-bottom: 20px;
    box-shadow: 0 6px 20px rgba(26,83,92,0.3);
    text-align: center;
}
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; }
.app-header p { margin: 6px 0 0; font-size: 0.95rem; opacity: 0.9; }
.section-label {
    font-size: 1rem; font-weight: 700; color: #1a535c;
    margin-bottom: 8px; padding-bottom: 4px; border-bottom: 2px solid #4ecdc4;
    display: inline-block;
}
.run-btn {
    background: linear-gradient(90deg, #1a535c, #4ecdc4) !important;
    color: white !important; font-weight: 700 !important;
    border-radius: 10px !important; padding: 12px 24px !important;
    box-shadow: 0 4px 12px rgba(78,205,196,0.4) !important;
}
.forensic-box {
    background: #eafaf8; border-left: 5px solid #4ecdc4;
    border-radius: 10px; padding: 16px 20px; color: #1a535c;
}
img { border-radius: 10px !important; }
"""

# ============================================================
# SECTION 7: GRADIO UI BUILD & WIRING
# ============================================================

def build_ui():
    with gr.Blocks(css=CUSTOM_CSS, title="Dashboard Forensik Citra Buram") as demo:

        gr.HTML("""
        <div class="app-header">
            <h1>🔬 Dashboard Restorasi & Analisis Forensik Citra Buram</h1>
            <p>Mata Kuliah Pola dan Pengolahan Citra Digital &nbsp;|&nbsp; Pipeline: Grayscale → Median → CLAHE → Sharpening → Edge Detection</p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1, min_width=280):
                gr.HTML('<div class="section-label">📁 Upload Citra</div>')
                input_image = gr.Image(type="pil", label="Unggah Citra", height=220)

                gr.HTML('<div class="section-label" style="margin-top:12px">⚙️ Kontrol Pipeline</div>')
                rotation_choice = gr.Dropdown(
                    choices=["Tidak Ada", "90°", "180°", "270°"],
                    value="Tidak Ada",
                    label="🔄 Rotasi"
                )
                zoom_slider = gr.Slider(
                    minimum=1.0, maximum=4.0, step=0.5, value=1.0,
                    label="🔍 Zoom (1x–4x)"
                )
                run_btn = gr.Button("▶ Jalankan Pipeline", elem_classes="run-btn")

            with gr.Column(scale=2):
                gr.HTML('<div class="section-label">🖼️ Visual Comparison: Sebelum vs Sesudah</div>')
                comparison_output = gr.Image(
                    label="Sebelum (Grayscale) | Sesudah (Sharpening)",
                    height=260
                )

        gr.HTML("<hr style='border-color:#d9f0ee; margin:10px 0'>")

        with gr.Tabs():
            # ---- TAB 1: Input & Output ----
            with gr.TabItem("📥 Tab 1 · Input & Output"):
                gr.HTML('<div class="section-label">Citra Input & Output Akhir</div>')
                with gr.Row():
                    with gr.Column():
                        gr.HTML('<p style="color:#1a535c;font-weight:600">Input Asli</p>')
                        tab1_original = gr.Image(label="Citra Asli", height=300)
                    with gr.Column():
                        gr.HTML('<p style="color:#1a535c;font-weight:600">Output Sharpening</p>')
                        tab1_sharp_out = gr.Image(label="Output Sharpening", height=300)
                    with gr.Column():
                        gr.HTML('<p style="color:#1a535c;font-weight:600">Output Edge Detection</p>')
                        tab1_edge_out = gr.Image(label="Output Edge Detection", height=300)

            # ---- TAB 2: Pipeline Process ----
            with gr.TabItem("🔄 Tab 2 · Pipeline Process"):
                gr.HTML('<div class="section-label">Semua Tahap Hasil Pipeline</div>')
                with gr.Row():
                    t2_gray   = gr.Image(label="1. Grayscale",     height=200)
                    t2_median = gr.Image(label="2. Median Filter",  height=200)
                    t2_clahe  = gr.Image(label="3. CLAHE",          height=200)
                with gr.Row():
                    t2_sharp  = gr.Image(label="4. Sharpening",     height=200)
                    t2_edge   = gr.Image(label="5. Edge Detection",  height=200)

            # ---- TAB 3: Histogram Analysis ----
            with gr.TabItem("📊 Tab 3 · Histogram Analysis"):
                gr.HTML('<div class="section-label">Histogram Setiap Tahap Pipeline</div>')
                with gr.Row():
                    t3_hist_gray   = gr.Image(label="Histogram Grayscale",     height=220)
                    t3_hist_median = gr.Image(label="Histogram Median Filter",  height=220)
                    t3_hist_clahe  = gr.Image(label="Histogram CLAHE",          height=220)
                with gr.Row():
                    t3_hist_sharp  = gr.Image(label="Histogram Sharpening",    height=220)
                    t3_hist_edge   = gr.Image(label="Histogram Edge Detection", height=220)

            # ---- TAB 4: Matrix Analysis ----
            with gr.TabItem("🔢 Tab 4 · Matrix Analysis"):
                gr.HTML('<div class="section-label">Matriks Piksel 50×50 (Area Kiri Atas)</div>')
                with gr.Accordion("📐 Matriks Grayscale", open=False):
                    t4_mat_gray   = gr.Dataframe(label="Grayscale 50×50", wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Median Filter", open=False):
                    t4_mat_median = gr.Dataframe(label="Median 50×50", wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks CLAHE", open=False):
                    t4_mat_clahe  = gr.Dataframe(label="CLAHE 50×50", wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Sharpening", open=False):
                    t4_mat_sharp  = gr.Dataframe(label="Sharpening 50×50", wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Edge Detection", open=False):
                    t4_mat_edge   = gr.Dataframe(label="Edge 50×50", wrap=False, max_height=280)

            # ---- TAB 5: Metrics & Forensic Report ----
            with gr.TabItem("📈 Tab 5 · Metrics & Forensic Report"):
                gr.HTML('<div class="section-label">Tabel Metrik Lengkap</div>')
                t5_metrics_table = gr.Dataframe(
                    label="MSE | PSNR | Sharpness | Contrast | Brightness",
                    wrap=False, max_height=250
                )
                gr.HTML('<div class="section-label" style="margin-top:16px">📋 Laporan Forensik Otomatis</div>')
                t5_forensic = gr.Markdown(elem_classes="forensic-box")

        # ============================================================
        # DISUNTING: Penggabungan output agar komputasi hanya berjalan 1 KALI
        # ============================================================
        all_outputs = [
            t2_gray, t2_median, t2_clahe, t2_sharp, t2_edge,           # 1-5: gambar proses pipeline
            t3_hist_gray, t3_hist_median, t3_hist_clahe,                # 6-8: histogram awal
            t3_hist_sharp, t3_hist_edge,                                # 9-10: histogram akhir
            t4_mat_gray, t4_mat_median, t4_mat_clahe,                   # 11-13: dataframe matriks awal
            t4_mat_sharp, t4_mat_edge,                                  # 14-15: dataframe matriks akhir
            t5_metrics_table,                                           # 16: Tabel evaluasi
            t5_forensic,                                                # 17: Teks kesimpulan laporan
            comparison_output,                                          # 18: Gambar komparasi
            tab1_original, tab1_sharp_out, tab1_edge_out                 # 19-21: Target Tab 1
        ]

        def pipeline_wrapper(img, rot, zoom):
            results = run_pipeline(img, rot, zoom)
            (pg, pm, pc, ps, pe,
             hg, hm, hc, hs, he,
             mg, mm, mc, ms, me,
             metrics_df, forensic, comp) = results

            # Handle pengisian Tab 1 secara simultan
            t1_orig = img if img is not None else pg

            return (pg, pm, pc, ps, pe,
                    hg, hm, hc, hs, he,
                    mg, mm, mc, ms, me,
                    metrics_df, forensic, comp,
                    t1_orig, ps, pe)

        run_btn.click(
            fn=pipeline_wrapper,
            inputs=[input_image, rotation_choice, zoom_slider],
            outputs=all_outputs
        )

    return demo

# ============================================================
# RUN UNTUK GOOGLE COLAB
# ============================================================
if __name__ == "__main__":
    demo = build_ui()
    # inline=True merender UI langsung di dalam notebook cell.
    # share=True membuat link publik sementara (.gradio.live) untuk didemokan ke dosen/teman.
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_2712/1411914122.py:340: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Dashboard Forensik Citra Buram") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fbdf7f9d14a876922b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fbdf7f9d14a876922b.gradio.live


In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Wajib untuk lingkungan tanpa GUI seperti Colab
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math

# ============================================================
# SECTION 1: UTILITY & METRIC FUNCTIONS
# ============================================================

def pil_to_cv2(pil_img):
    """Konversi PIL Image ke NumPy array BGR (OpenCV)."""
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    """Konversi NumPy array BGR ke PIL Image Berwarna."""
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def gray_to_pil(gray_img):
    """Konversi grayscale NumPy array ke PIL Image."""
    return Image.fromarray(gray_img)

def compute_mse(original_gray, processed_gray):
    """Hitung Mean Squared Error antara dua citra grayscale."""
    original_gray = original_gray.astype(np.float64)
    processed_gray = processed_gray.astype(np.float64)
    if original_gray.shape != processed_gray.shape:
        processed_gray = cv2.resize(processed_gray, (original_gray.shape[1], original_gray.shape[0]))
    return float(np.mean((original_gray - processed_gray) ** 2))

def compute_psnr(mse):
    """Hitung PSNR dari nilai MSE."""
    if mse == 0:
        return float('inf')
    return float(20 * math.log10(255.0 / math.sqrt(mse)))

def compute_sharpness(gray_img):
    """Hitung Sharpness menggunakan Variance of Laplacian."""
    return float(cv2.Laplacian(gray_img, cv2.CV_64F).var())

def compute_contrast(gray_img):
    """Hitung Contrast menggunakan standar deviasi intensitas piksel."""
    return float(gray_img.std())

def compute_brightness(gray_img):
    """Hitung Brightness menggunakan rata-rata intensitas piksel."""
    return float(gray_img.mean())

def get_metrics(original_gray, processed_bgr_or_gray, label):
    """Konversi internal ke grayscale untuk menghitung metrik secara akurat."""
    if len(processed_bgr_or_gray.shape) == 3:
        target_gray = cv2.cvtColor(processed_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        target_gray = processed_bgr_or_gray

    mse = compute_mse(original_gray, target_gray)
    psnr = compute_psnr(mse)
    sharpness = compute_sharpness(target_gray)
    contrast = compute_contrast(target_gray)
    brightness = compute_brightness(target_gray)
    return {
        "Tahap": label,
        "MSE": round(mse, 4),
        "PSNR (dB)": round(psnr, 4) if psnr != float('inf') else "∞",
        "Sharpness": round(sharpness, 4),
        "Contrast": round(contrast, 4),
        "Brightness": round(brightness, 4),
    }

# ============================================================
# SECTION 2: RGB COLOR PIPELINE PROCESSING FUNCTIONS
# ============================================================

def step_median_filter_rgb(bgr_img, ksize=5):
    """Terapkan Median Filter langsung pada citra berwarna BGR."""
    return cv2.medianBlur(bgr_img, ksize)

def step_clahe_rgb(bgr_img, clip_limit=2.0, tile_grid=(8, 8)):
    """Terapkan CLAHE pada ruang warna LAB (Channel L) agar warna tetap natural."""
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cl = clahe.apply(l)
    enhanced_lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

def step_sharpening_rgb(bgr_img):
    """Terapkan kernel Unsharp Mask pada citra berwarna BGR."""
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(bgr_img, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def step_rotate_rgb(bgr_img, angle_str):
    """Rotasi citra berwarna."""
    angle_map = {
        "Tidak Ada": None,
        "90°": cv2.ROTATE_90_CLOCKWISE,
        "180°": cv2.ROTATE_180,
        "270°": cv2.ROTATE_90_COUNTERCLOCKWISE,
    }
    code = angle_map.get(angle_str, None)
    if code is not None:
        return cv2.rotate(bgr_img, code)
    return bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    """Zoom citra berwarna."""
    if zoom_factor == 1.0:
        return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    zoomed = zoomed[:h, :w] if zoomed.shape[0] >= h and zoomed.shape[1] >= w else \
             cv2.resize(zoomed, (w, h), interpolation=cv2.INTER_LINEAR)
    return zoomed

def step_edge_detection(bgr_img):
    """Cabang Analisis: Konversi ke Grayscale lalu jalankan Canny Edge Detector."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, threshold1=50, threshold2=150)

# ============================================================
# SECTION 3: VISUALIZATION HELPERS (DARK THEME ADJUSTED)
# ============================================================

def plot_histogram(bgr_or_gray_img, title):
    """Buat histogram grayscale/intensitas dengan tema gelap."""
    fig, ax = plt.subplots(figsize=(5, 3), facecolor='#111827')
    ax.set_facecolor('#111827')

    if len(bgr_or_gray_img.shape) == 3:
        gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY)
    else:
        gray_img = bgr_or_gray_img

    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#2dd4bf')
    ax.set_xlabel("Intensitas Piksel", fontsize=9, color='#9ca3af')
    ax.set_ylabel("Frekuensi", fontsize=9, color='#9ca3af')
    ax.tick_params(colors='#9ca3af', labelsize=8)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15, color='#9ca3af')
    fig.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_or_gray_img, size=50):
    """Ambil matriks kecerahan area 50x50 kiri atas (Grayscale branch)."""
    if len(bgr_or_gray_img.shape) == 3:
        gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY)
    else:
        gray_img = bgr_or_gray_img
    h, w = gray_img.shape
    region = gray_img[:min(size, h), :min(size, w)]
    df = pd.DataFrame(region)
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    """Buat gambar perbandingan warna Sebelum vs Sesudah berdampingan."""
    h1, w1, _ = original_bgr.shape
    h2, w2, _ = final_bgr.shape
    max_h = max(h1, h2)

    pad1 = np.zeros((max_h, w1, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2 = np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150 # Garis pembatas abu-abu
    combined = np.hstack([pad1, divider, pad2])

    cv2.putText(combined, "ASLI (RGB)", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "RESTORASI (RGB)", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

# ============================================================
# SECTION 4: FORENSIC SUMMARY GENERATOR
# ============================================================

def generate_forensic_summary(metrics_list):
    """Laporan otomatis berdasarkan komparasi metrik."""
    if not metrics_list:
        return "Tidak ada data metrik untuk dianalisis."

    lines = ["📋 **LAPORAN ANALISIS FORENSIK CITRA**\n"]
    lines.append("---")

    orig_m  = next((m for m in metrics_list if m["Tahap"] == "Input RGB"), None)
    clahe_m = next((m for m in metrics_list if m["Tahap"] == "CLAHE RGB"), None)
    sharp_m = next((m for m in metrics_list if m["Tahap"] == "Sharpening RGB"), None)

    if orig_m and sharp_m:
        s_before = orig_m["Sharpness"]
        s_after  = sharp_m["Sharpness"]
        diff = s_after - s_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🔍 **Metrik Ketajaman (Sharpness):** Melalui teknik Unsharp Mask RGB, tingkat ketajaman struktural {direction} signifikan dari **{s_before:.2f}** menjadi **{s_after:.2f}** (Selisih: {abs(diff):.2f}).")

    if orig_m and clahe_m:
        c_before = orig_m["Contrast"]
        c_after  = clahe_m["Contrast"]
        diff = c_after - c_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🎨 **Metrik Kontras (Luminance):** CLAHE pada ruang warna LAB berhasil mendistribusikan pencahayaan secara adaptif. Kontras berubah dari **{c_before:.2f}** menjadi **{c_after:.2f}** tanpa merusak saturasi warna asli.")

    if sharp_m:
        psnr_val = sharp_m["PSNR (dB)"]
        lines.append(f"📡 **Fidelitas Rekonstruksi (PSNR):** Nilai puncak rasio sinyal terhadap noise pada tahap restorasi warna adalah **{psnr_val} dB**, mengonfirmasi proses perbaikan citra berada pada rentang akurasi data forensik yang valid.")

    lines.append("\n✅ **Kesimpulan Forensik:** Pipeline restorasi warna (RGB) berhasil mereduksi blur dan mengoptimalkan visibilitas objek tanpa kehilangan data kromatik/warna asli barang bukti digital.")
    return "\n\n".join(lines)

# ============================================================
# SECTION 5: MAIN PIPELINE FUNCTION
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        empty_hist = plot_histogram(np.zeros((100, 100), dtype=np.uint8), "Tidak Ada Data")
        metrics_df = pd.DataFrame()
        forensic = "Silakan upload citra terlebih dahulu."
        return (
            empty, empty, empty, empty, empty, empty,
            empty_hist, empty_hist, empty_hist, empty_hist, empty_hist,
            empty_df, empty_df, empty_df, empty_df, empty_df,
            metrics_df, forensic, empty
        )

    # --- PIPELINE UTAMA (RGB BERWARNA) ---
    bgr_input = pil_to_cv2(input_image)
    original_gray = cv2.cvtColor(bgr_input, cv2.COLOR_BGR2GRAY)  # Baseline untuk metrik

    median_bgr = step_median_filter_rgb(bgr_input)
    clahe_bgr  = step_clahe_rgb(median_bgr)
    sharp_bgr  = step_sharpening_rgb(clahe_bgr)
    rotated_bgr = step_rotate_rgb(sharp_bgr, rotation_choice)
    zoomed_bgr  = step_zoom_rgb(rotated_bgr, float(zoom_factor))

    # --- CABANG ANALISIS FORENSIK (GRAYSCALE BRANCH) ---
    edge_gray   = step_edge_detection(zoomed_bgr)

    # Konversi hasil untuk UI
    pil_input  = input_image
    pil_median = cv2_to_pil(median_bgr)
    pil_clahe  = cv2_to_pil(clahe_bgr)
    pil_sharp  = cv2_to_pil(sharp_bgr)
    pil_edge   = gray_to_pil(edge_gray)

    # Pembuatan Histogram per tahap
    hist_input  = plot_histogram(bgr_input,  "Histogram Input RGB")
    hist_median = plot_histogram(median_bgr, "Histogram Median RGB")
    hist_clahe  = plot_histogram(clahe_bgr,  "Histogram CLAHE RGB")
    hist_sharp  = plot_histogram(sharp_bgr,  "Histogram Sharpening RGB")
    hist_edge   = plot_histogram(edge_gray,  "Histogram Edge Detection")

    # Pembuatan Matriks Piksel 50x50 (Grayscale Analysis)
    mat_input  = get_matrix_dataframe(bgr_input)
    mat_median = get_matrix_dataframe(median_bgr)
    mat_clahe  = get_matrix_dataframe(clahe_bgr)
    mat_sharp  = get_matrix_dataframe(sharp_bgr)
    mat_edge   = get_matrix_dataframe(edge_gray)

    # Perhitungan Metrik Evaluasi Komparatif
    metrics_list = [
        get_metrics(original_gray, bgr_input,  "Input RGB"),
        get_metrics(original_gray, median_bgr, "Median Filter RGB"),
        get_metrics(original_gray, clahe_bgr,  "CLAHE RGB"),
        get_metrics(original_gray, sharp_bgr,  "Sharpening RGB"),
        get_metrics(original_gray, edge_gray,   "Edge Detection"),
    ]
    metrics_df = pd.DataFrame(metrics_list)
    forensic_text = generate_forensic_summary(metrics_list)
    comparison_img = make_comparison_image(bgr_input, zoomed_bgr)

    return (
        pil_input, pil_median, pil_clahe, pil_sharp, pil_edge,
        hist_input, hist_median, hist_clahe, hist_sharp, hist_edge,
        mat_input, mat_median, mat_clahe, mat_sharp, mat_edge,
        metrics_df, forensic_text, comparison_img
    )

# ============================================================
# SECTION 6: PREMIUM DARK THEME CSS
# ============================================================

CUSTOM_CSS = """
body, .gradio-container {
    background-color: #0b0f19 !important;
    color: #e2e8f0 !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
}
.app-header {
    background: linear-gradient(135deg, #0f172a, #115e59);
    color: white;
    padding: 24px;
    border-radius: 14px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.6);
    text-align: center;
    border: 1px solid #14b8a6;
}
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.app-header p { margin: 6px 0 0; font-size: 0.95rem; color: #9ca3af; }
.section-label {
    font-size: 1rem; font-weight: 700; color: #2dd4bf;
    margin-bottom: 8px; padding-bottom: 4px; border-bottom: 2px solid #0f766e;
    display: inline-block;
}
.run-btn {
    background: linear-gradient(90deg, #0f766e, #14b8a6) !important;
    color: white !important; font-weight: 700 !important;
    border-radius: 10px !important; border: none !important;
    box-shadow: 0 4px 12px rgba(20, 184, 166, 0.3) !important;
}
.forensic-box {
    background: #111827; border-left: 5px solid #14b8a6;
    border-radius: 10px; padding: 16px 20px; color: #e2e8f0;
    border-top: 1px solid #1f2937; border-right: 1px solid #1f2937; border-bottom: 1px solid #1f2937;
}
img { border-radius: 10px !important; border: 1px solid #1f2937; }
"""

# ============================================================
# SECTION 7: GRADIO UI BUILD (COMPATIBLE WITH GRADIO 4+)
# ============================================================

def build_ui():
    # Menggunakan injeksi JS untuk memaksa dokumen HTML menggunakan class 'dark' bawaan Gradio
    force_dark_mode_js = "() => document.body.classList.add('dark')"

    with gr.Blocks(
        theme=gr.themes.Default(),
        css=CUSTOM_CSS,
        title="Dashboard Forensik RGB",
        js=force_dark_mode_js
    ) as demo:

        gr.HTML("""
        <div class="app-header">
            <h1>🔬 Dashboard Restorasi & Analisis Forensik Citra (Mode Warna RGB)</h1>
            <p>Sistem Komputasi Forensik Berwarna &nbsp;|&nbsp; Pipeline: Input RGB → Median RGB → CLAHE RGB (LAB) → Sharpening RGB</p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1, min_width=280):
                gr.HTML('<div class="section-label">📁 Upload Citra</div>')
                input_image = gr.Image(type="pil", label="Unggah Citra", height=220)

                gr.HTML('<div class="section-label" style="margin-top:12px">⚙️ Kontrol Geometri</div>')
                rotation_choice = gr.Dropdown(
                    choices=["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi Citra"
                )
                zoom_slider = gr.Slider(
                    minimum=1.0, maximum=4.0, step=0.5, value=1.0, label="🔍 Zoom Restorasi"
                )
                run_btn = gr.Button("▶ Jalankan Pipeline Forensik", elem_classes="run-btn")

            with gr.Column(scale=2):
                gr.HTML('<div class="section-label">🖼️ Komparasi Mutu: Citra Awal vs Hasil Restorasi Akhir</div>')
                comparison_output = gr.Image(label="Sisi Berdampingan (Side-by-Side)", height=260)

        gr.HTML("<hr style='border-color:#1f2937; margin:10px 0'>")

        with gr.Tabs():
            # ---- TAB 1: Input & Output ----
            with gr.TabItem("📥 Tab 1 · Hasil Akhir Berwarna"):
                gr.HTML('<div class="section-label">Hasil Inspeksi Akhir Proyeksi Berwarna</div>')
                with gr.Row():
                    with gr.Column():
                        gr.HTML('<p style="color:#2dd4bf;font-weight:600">Citra Input Asli</p>')
                        tab1_original = gr.Image(label="Citra Asli", height=300)
                    with gr.Column():
                        gr.HTML('<p style="color:#2dd4bf;font-weight:600">Output Restorasi (Berwarna)</p>')
                        tab1_sharp_out = gr.Image(label="Output Tajam Warna", height=300)
                    with gr.Column():
                        gr.HTML('<p style="color:#2dd4bf;font-weight:600">Output Edge (Grayscale Branch)</p>')
                        tab1_edge_out = gr.Image(label="Output Deteksi Tepi", height=300)

            # ---- TAB 2: Pipeline Process ----
            with gr.TabItem("🔄 Tab 2 · Tahapan Pipeline RGB"):
                gr.HTML('<div class="section-label">Sekuensial Proses Enchancement Berwarna</div>')
                with gr.Row():
                    t2_input  = gr.Image(label="1. Input Citra RGB", height=200)
                    t2_median = gr.Image(label="2. Median Filter RGB", height=200)
                    t2_clahe  = gr.Image(label="3. CLAHE (LAB Space)", height=200)
                with gr.Row():
                    t2_sharp  = gr.Image(label="4. Sharpening RGB", height=200)
                    t2_edge   = gr.Image(label="5. Edge Detection (Analyzed)", height=200)

            # ---- TAB 3: Histogram Analysis ----
            with gr.TabItem("📊 Tab 3 · Analisis Histogram"):
                gr.HTML('<div class="section-label">Karakteristik Distribusi Intensitas Citra</div>')
                with gr.Row():
                    t3_hist_gray   = gr.Image(label="Histogram Input", height=220)
                    t3_hist_median = gr.Image(label="Histogram Median", height=220)
                    t3_hist_clahe  = gr.Image(label="Histogram CLAHE", height=220)
                with gr.Row():
                    t3_hist_sharp  = gr.Image(label="Histogram Sharpening", height=220)
                    t3_hist_edge   = gr.Image(label="Histogram Edge", height=220)

            # ---- TAB 4: Matrix Analysis ----
            with gr.TabItem("🔢 Tab 4 · Matriks Kecerahan (50×50)"):
                gr.HTML('<div class="section-label">Ekstraksi Nilai Piksel Grayscale Kiri Atas</div>')
                with gr.Accordion("📐 Matriks Tahap Input", open=False):
                    t4_mat_gray   = gr.Dataframe(wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Tahap Median Filter", open=False):
                    t4_mat_median = gr.Dataframe(wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Tahap CLAHE (LAB)", open=False):
                    t4_mat_clahe  = gr.Dataframe(wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Tahap Sharpening", open=False):
                    t4_mat_sharp  = gr.Dataframe(wrap=False, max_height=280)
                with gr.Accordion("📐 Matriks Tahap Edge Detection", open=False):
                    t4_mat_edge   = gr.Dataframe(wrap=False, max_height=280)

            # ---- TAB 5: Metrics & Forensic Report ----
            with gr.TabItem("📈 Tab 5 · Laporan & Metrik Evaluasi"):
                gr.HTML('<div class="section-label">Kalkulasi Validasi Objektif</div>')
                t5_metrics_table = gr.Dataframe(wrap=False, max_height=250)
                gr.HTML('<div class="section-label" style="margin-top:16px">📋 Hasil Keputusan Forensik Otomatis</div>')
                t5_forensic = gr.Markdown(elem_classes="forensic-box")

        # Sinkronisasi Output Komputasi Tunggal (Efisien RAM)
        all_outputs = [
            t2_input, t2_median, t2_clahe, t2_sharp, t2_edge,
            t3_hist_gray, t3_hist_median, t3_hist_clahe, t3_hist_sharp, t3_hist_edge,
            t4_mat_gray, t4_mat_median, t4_mat_clahe, t4_mat_sharp, t4_mat_edge,
            t5_metrics_table, t5_forensic, comparison_output,
            tab1_original, tab1_sharp_out, tab1_edge_out
        ]

        def pipeline_wrapper(img, rot, zoom):
            results = run_pipeline(img, rot, zoom)
            (pi, pm, pc, ps, pe,
             hi, hm, hc, hs, he,
             mi, mm, mc, ms, me,
             metrics_df, forensic, comp) = results

            t1_orig = img if img is not None else pi
            return (pi, pm, pc, ps, pe,
                    hi, hm, hc, hs, he,
                    mi, mm, mc, ms, me,
                    metrics_df, forensic, comp,
                    t1_orig, ps, pe)

        run_btn.click(
            fn=pipeline_wrapper,
            inputs=[input_image, rotation_choice, zoom_slider],
            outputs=all_outputs
        )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_2712/2027622179.py:349: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2712/2027622179.py:349: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2712/2027622179.py:349: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c4d347c16341061478.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c4d347c16341061478.gradio.live


In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Fitur: Mode Berantai (Otomatis) & Mode Mandiri (Kustom)
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Wajib untuk lingkungan tanpa GUI seperti Colab
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math

# ============================================================
# SECTION 1: UTILITY & METRIC FUNCTIONS
# ============================================================

def pil_to_cv2(pil_img):
    """Konversi PIL Image ke NumPy array BGR (OpenCV)."""
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    """Konversi NumPy array BGR ke PIL Image Berwarna."""
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def gray_to_pil(gray_img):
    """Konversi grayscale NumPy array ke PIL Image."""
    return Image.fromarray(gray_img)

def compute_mse(original_gray, processed_gray):
    """Hitung Mean Squared Error antara dua citra grayscale."""
    original_gray = original_gray.astype(np.float64)
    processed_gray = processed_gray.astype(np.float64)
    if original_gray.shape != processed_gray.shape:
        processed_gray = cv2.resize(processed_gray, (original_gray.shape[1], original_gray.shape[0]))
    return float(np.mean((original_gray - processed_gray) ** 2))

def compute_psnr(mse):
    """Hitung PSNR dari nilai MSE."""
    if mse == 0:
        return float('inf')
    return float(20 * math.log10(255.0 / math.sqrt(mse)))

def compute_sharpness(gray_img):
    """Hitung Sharpness menggunakan Variance of Laplacian."""
    return float(cv2.Laplacian(gray_img, cv2.CV_64F).var())

def compute_contrast(gray_img):
    """Hitung Contrast menggunakan standar deviasi intensitas piksel."""
    return float(gray_img.std())

def compute_brightness(gray_img):
    """Hitung Brightness menggunakan rata-rata intensitas piksel."""
    return float(gray_img.mean())

def get_metrics(original_gray, processed_bgr_or_gray, label):
    """Konversi internal ke grayscale untuk menghitung metrik secara akurat."""
    if len(processed_bgr_or_gray.shape) == 3:
        target_gray = cv2.cvtColor(processed_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        target_gray = processed_bgr_or_gray

    mse = compute_mse(original_gray, target_gray)
    psnr = compute_psnr(mse)
    sharpness = compute_sharpness(target_gray)
    contrast = compute_contrast(target_gray)
    brightness = compute_brightness(target_gray)
    return {
        "Tahap": label,
        "MSE": round(mse, 4),
        "PSNR (dB)": round(psnr, 4) if psnr != float('inf') else "∞",
        "Sharpness": round(sharpness, 4),
        "Contrast": round(contrast, 4),
        "Brightness": round(brightness, 4),
    }

# ============================================================
# SECTION 2: RGB COLOR PIPELINE PROCESSING FUNCTIONS
# ============================================================

def step_median_filter_rgb(bgr_img, ksize=5):
    """Terapkan Median Filter langsung pada citra berwarna BGR."""
    return cv2.medianBlur(bgr_img, ksize)

def step_clahe_rgb(bgr_img, clip_limit=2.0, tile_grid=(8, 8)):
    """Terapkan CLAHE pada ruang warna LAB (Channel L) agar warna tetap natural."""
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cl = clahe.apply(l)
    enhanced_lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

def step_sharpening_rgb(bgr_img):
    """Terapkan kernel Unsharp Mask pada citra berwarna BGR."""
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(bgr_img, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def step_rotate_rgb(bgr_img, angle_str):
    """Rotasi citra berwarna."""
    angle_map = {
        "Tidak Ada": None,
        "90°": cv2.ROTATE_90_CLOCKWISE,
        "180°": cv2.ROTATE_180,
        "270°": cv2.ROTATE_90_COUNTERCLOCKWISE,
    }
    code = angle_map.get(angle_str, None)
    if code is not None:
        return cv2.rotate(bgr_img, code)
    return bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    """Zoom citra berwarna."""
    if zoom_factor == 1.0:
        return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    zoomed = zoomed[:h, :w] if zoomed.shape[0] >= h and zoomed.shape[1] >= w else \
             cv2.resize(zoomed, (w, h), interpolation=cv2.INTER_LINEAR)
    return zoomed

def step_edge_detection(bgr_img):
    """Cabang Analisis: Konversi ke Grayscale lalu jalankan Canny Edge Detector."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, threshold1=50, threshold2=150)

# ============================================================
# SECTION 3: VISUALIZATION HELPERS (DARK THEME ADJUSTED)
# ============================================================

def plot_histogram(bgr_or_gray_img, title):
    """Buat histogram grayscale/intensitas dengan tema gelap."""
    fig, ax = plt.subplots(figsize=(5, 3), facecolor='#111827')
    ax.set_facecolor('#111827')

    if len(bgr_or_gray_img.shape) == 3:
        gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY)
    else:
        gray_img = bgr_or_gray_img

    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#2dd4bf')
    ax.set_xlabel("Intensitas Piksel", fontsize=9, color='#9ca3af')
    ax.set_ylabel("Frekuensi", fontsize=9, color='#9ca3af')
    ax.tick_params(colors='#9ca3af', labelsize=8)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15, color='#9ca3af')
    fig.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_or_gray_img, size=50):
    """Ambil matriks kecerahan area 50x50 kiri atas (Grayscale branch)."""
    if len(bgr_or_gray_img.shape) == 3:
        gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY)
    else:
        gray_img = bgr_or_gray_img
    h, w = gray_img.shape
    region = gray_img[:min(size, h), :min(size, w)]
    df = pd.DataFrame(region)
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    """Buat gambar perbandingan warna Sebelum vs Sesudah berdampingan."""
    h1, w1, _ = original_bgr.shape
    h2, w2, _ = final_bgr.shape
    max_h = max(h1, h2)

    pad1 = np.zeros((max_h, w1, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2 = np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150 # Garis pembatas abu-abu
    combined = np.hstack([pad1, divider, pad2])

    cv2.putText(combined, "ASLI (RGB)", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "RESTORASI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

# ============================================================
# SECTION 4: FORENSIC SUMMARY GENERATOR
# ============================================================

def generate_forensic_summary(metrics_list):
    """Laporan otomatis berdasarkan komparasi metrik."""
    if not metrics_list:
        return "Tidak ada data metrik untuk dianalisis."

    lines = ["📋 **LAPORAN ANALISIS FORENSIK CITRA**\n"]
    lines.append("---")

    orig_m  = next((m for m in metrics_list if m["Tahap"] == "Input RGB"), None)
    clahe_m = next((m for m in metrics_list if m["Tahap"] == "CLAHE RGB"), None)
    sharp_m = next((m for m in metrics_list if m["Tahap"] == "Sharpening RGB"), None)

    if orig_m and sharp_m:
        s_before = orig_m["Sharpness"]
        s_after  = sharp_m["Sharpness"]
        diff = s_after - s_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🔍 **Metrik Ketajaman (Sharpness):** Melalui teknik Unsharp Mask RGB, tingkat ketajaman struktural {direction} signifikan dari **{s_before:.2f}** menjadi **{s_after:.2f}** (Selisih: {abs(diff):.2f}).")

    if orig_m and clahe_m:
        c_before = orig_m["Contrast"]
        c_after  = clahe_m["Contrast"]
        diff = c_after - c_before
        direction = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🎨 **Metrik Kontras (Luminance):** CLAHE pada ruang warna LAB berhasil mendistribusikan pencahayaan secara adaptif. Kontras berubah dari **{c_before:.2f}** menjadi **{c_after:.2f}** tanpa merusak saturasi warna asli.")

    if sharp_m:
        psnr_val = sharp_m["PSNR (dB)"]
        lines.append(f"📡 **Fidelitas Rekonstruksi (PSNR):** Nilai puncak rasio sinyal terhadap noise pada tahap restorasi warna adalah **{psnr_val} dB**, mengonfirmasi proses perbaikan citra berada pada rentang akurasi data forensik yang valid.")

    lines.append("\n✅ **Kesimpulan Forensik:** Pipeline restorasi warna (RGB) berhasil mereduksi blur dan mengoptimalkan visibilitas objek tanpa kehilangan data kromatik/warna asli barang bukti digital.")
    return "\n\n".join(lines)

# ============================================================
# SECTION 5: PIPELINE FUNCTIONS (BERANTAI & MANDIRI)
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Fungsi eksekusi untuk Mode Berantai (Original)."""
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        empty_hist = plot_histogram(np.zeros((100, 100), dtype=np.uint8), "Tidak Ada Data")
        return (
            empty, empty, empty, empty, empty, empty,
            empty_hist, empty_hist, empty_hist, empty_hist, empty_hist,
            empty_df, empty_df, empty_df, empty_df, empty_df,
            empty_df, "Silakan upload citra terlebih dahulu.", empty
        )

    bgr_input = pil_to_cv2(input_image)
    original_gray = cv2.cvtColor(bgr_input, cv2.COLOR_BGR2GRAY)

    median_bgr = step_median_filter_rgb(bgr_input)
    clahe_bgr  = step_clahe_rgb(median_bgr)
    sharp_bgr  = step_sharpening_rgb(clahe_bgr)
    rotated_bgr = step_rotate_rgb(sharp_bgr, rotation_choice)
    zoomed_bgr  = step_zoom_rgb(rotated_bgr, float(zoom_factor))
    edge_gray   = step_edge_detection(zoomed_bgr)

    pil_input  = input_image
    pil_median = cv2_to_pil(median_bgr)
    pil_clahe  = cv2_to_pil(clahe_bgr)
    pil_sharp  = cv2_to_pil(sharp_bgr)
    pil_edge   = gray_to_pil(edge_gray)

    hist_input  = plot_histogram(bgr_input,  "Histogram Input RGB")
    hist_median = plot_histogram(median_bgr, "Histogram Median RGB")
    hist_clahe  = plot_histogram(clahe_bgr,  "Histogram CLAHE RGB")
    hist_sharp  = plot_histogram(sharp_bgr,  "Histogram Sharpening RGB")
    hist_edge   = plot_histogram(edge_gray,  "Histogram Edge Detection")

    mat_input  = get_matrix_dataframe(bgr_input)
    mat_median = get_matrix_dataframe(median_bgr)
    mat_clahe  = get_matrix_dataframe(clahe_bgr)
    mat_sharp  = get_matrix_dataframe(sharp_bgr)
    mat_edge   = get_matrix_dataframe(edge_gray)

    metrics_list = [
        get_metrics(original_gray, bgr_input,  "Input RGB"),
        get_metrics(original_gray, median_bgr, "Median Filter RGB"),
        get_metrics(original_gray, clahe_bgr,  "CLAHE RGB"),
        get_metrics(original_gray, sharp_bgr,  "Sharpening RGB"),
        get_metrics(original_gray, edge_gray,   "Edge Detection"),
    ]
    metrics_df = pd.DataFrame(metrics_list)
    forensic_text = generate_forensic_summary(metrics_list)
    comparison_img = make_comparison_image(bgr_input, zoomed_bgr)

    return (
        pil_input, pil_median, pil_clahe, pil_sharp, pil_edge,
        hist_input, hist_median, hist_clahe, hist_sharp, hist_edge,
        mat_input, mat_median, mat_clahe, mat_sharp, mat_edge,
        metrics_df, forensic_text, comparison_img
    )

def run_mandiri_pipeline(input_image, selected_steps, rotation_choice, zoom_factor):
    """Fungsi eksekusi kustom untuk Mode Mandiri."""
    if input_image is None:
        empty_img = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_hist = plot_histogram(np.zeros((10, 10), dtype=np.uint8), "Tidak Ada Data")
        return empty_img, empty_img, empty_hist, pd.DataFrame(), pd.DataFrame()

    bgr_img = pil_to_cv2(input_image)
    original_gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    current_img = bgr_img.copy()

    # Eksekusi sesuai pilihan user (urutan tetap logis)
    if "Median Filter" in selected_steps:
        current_img = step_median_filter_rgb(current_img)
    if "CLAHE (LAB)" in selected_steps:
        current_img = step_clahe_rgb(current_img)
    if "Sharpening" in selected_steps:
        current_img = step_sharpening_rgb(current_img)

    # Rotasi & Zoom
    current_img = step_rotate_rgb(current_img, rotation_choice)
    current_img = step_zoom_rgb(current_img, float(zoom_factor))

    # Cek jika pengguna memilih Edge Detection di tahap akhir
    if "Edge Detection" in selected_steps:
        final_result = step_edge_detection(current_img)
        pil_out = gray_to_pil(final_result)
    else:
        final_result = current_img
        pil_out = cv2_to_pil(final_result)

    # Ekstraksi Visual & Data
    comparison_img = make_comparison_image(bgr_img, final_result if len(final_result.shape) == 3 else cv2.cvtColor(final_result, cv2.COLOR_GRAY2BGR))
    hist_out = plot_histogram(final_result, "Histogram Hasil Kustom")
    mat_out = get_matrix_dataframe(final_result)

    # Metrik
    metrics_list = [
        get_metrics(original_gray, bgr_img, "Citra Asli"),
        get_metrics(original_gray, final_result, "Hasil Mandiri")
    ]
    metrics_df = pd.DataFrame(metrics_list)

    return pil_out, comparison_img, hist_out, mat_out, metrics_df

# ============================================================
# SECTION 6: PREMIUM DARK THEME CSS
# ============================================================

CUSTOM_CSS = """
body, .gradio-container {
    background-color: #0b0f19 !important;
    color: #e2e8f0 !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
}
.app-header {
    background: linear-gradient(135deg, #0f172a, #115e59);
    color: white;
    padding: 24px;
    border-radius: 14px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.6);
    text-align: center;
    border: 1px solid #14b8a6;
}
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.app-header p { margin: 6px 0 0; font-size: 0.95rem; color: #9ca3af; }
.section-label {
    font-size: 1rem; font-weight: 700; color: #2dd4bf;
    margin-bottom: 8px; padding-bottom: 4px; border-bottom: 2px solid #0f766e;
    display: inline-block;
}
.run-btn {
    background: linear-gradient(90deg, #0f766e, #14b8a6) !important;
    color: white !important; font-weight: 700 !important;
    border-radius: 10px !important; border: none !important;
    box-shadow: 0 4px 12px rgba(20, 184, 166, 0.3) !important;
}
.forensic-box {
    background: #111827; border-left: 5px solid #14b8a6;
    border-radius: 10px; padding: 16px 20px; color: #e2e8f0;
    border-top: 1px solid #1f2937; border-right: 1px solid #1f2937; border-bottom: 1px solid #1f2937;
}
img { border-radius: 10px !important; border: 1px solid #1f2937; }
"""

# ============================================================
# SECTION 7: GRADIO UI BUILD
# ============================================================

def build_ui():
    force_dark_mode_js = "() => document.body.classList.add('dark')"

    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:

        gr.HTML("""
        <div class="app-header">
            <h1>🔬 Dashboard Restorasi & Analisis Forensik Citra (Mode Warna RGB)</h1>
            <p>Sistem Komputasi Forensik Multi-Mode &nbsp;|&nbsp; Dilengkapi Pemrosesan Otomatis & Analisis Kustom Mandiri</p>
        </div>
        """)

        with gr.Tabs():
            # ========================================================
            # TAB UTAMA 1: MODE BERANTAI (ORIGINAL)
            # ========================================================
            with gr.TabItem("⛓️ Mode Berantai (Pipeline Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1, min_width=280):
                        gr.HTML('<div class="section-label">📁 Upload Citra</div>')
                        input_image = gr.Image(type="pil", label="Unggah Citra", height=220)

                        gr.HTML('<div class="section-label" style="margin-top:12px">⚙️ Kontrol Geometri</div>')
                        rotation_choice = gr.Dropdown(
                            choices=["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi Citra"
                        )
                        zoom_slider = gr.Slider(
                            minimum=1.0, maximum=4.0, step=0.5, value=1.0, label="🔍 Zoom Restorasi"
                        )
                        run_btn = gr.Button("▶ Jalankan Pipeline Forensik", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        gr.HTML('<div class="section-label">🖼️ Komparasi Mutu: Citra Awal vs Hasil Restorasi Akhir</div>')
                        comparison_output = gr.Image(label="Sisi Berdampingan (Side-by-Side)", height=260)

                gr.HTML("<hr style='border-color:#1f2937; margin:10px 0'>")

                with gr.Tabs():
                    with gr.TabItem("📥 Tab 1 · Hasil Akhir Berwarna"):
                        gr.HTML('<div class="section-label">Hasil Inspeksi Akhir Proyeksi Berwarna</div>')
                        with gr.Row():
                            with gr.Column():
                                gr.HTML('<p style="color:#2dd4bf;font-weight:600">Citra Input Asli</p>')
                                tab1_original = gr.Image(label="Citra Asli", height=300)
                            with gr.Column():
                                gr.HTML('<p style="color:#2dd4bf;font-weight:600">Output Restorasi (Berwarna)</p>')
                                tab1_sharp_out = gr.Image(label="Output Tajam Warna", height=300)
                            with gr.Column():
                                gr.HTML('<p style="color:#2dd4bf;font-weight:600">Output Edge (Grayscale Branch)</p>')
                                tab1_edge_out = gr.Image(label="Output Deteksi Tepi", height=300)

                    with gr.TabItem("🔄 Tab 2 · Tahapan Pipeline"):
                        gr.HTML('<div class="section-label">Sekuensial Proses Enchancement Berwarna</div>')
                        with gr.Row():
                            t2_input  = gr.Image(label="1. Input Citra RGB", height=200)
                            t2_median = gr.Image(label="2. Median Filter RGB", height=200)
                            t2_clahe  = gr.Image(label="3. CLAHE (LAB Space)", height=200)
                        with gr.Row():
                            t2_sharp  = gr.Image(label="4. Sharpening RGB", height=200)
                            t2_edge   = gr.Image(label="5. Edge Detection (Analyzed)", height=200)

                    with gr.TabItem("📊 Tab 3 · Analisis Histogram"):
                        gr.HTML('<div class="section-label">Karakteristik Distribusi Intensitas Citra</div>')
                        with gr.Row():
                            t3_hist_gray   = gr.Image(label="Histogram Input", height=220)
                            t3_hist_median = gr.Image(label="Histogram Median", height=220)
                            t3_hist_clahe  = gr.Image(label="Histogram CLAHE", height=220)
                        with gr.Row():
                            t3_hist_sharp  = gr.Image(label="Histogram Sharpening", height=220)
                            t3_hist_edge   = gr.Image(label="Histogram Edge", height=220)

                    with gr.TabItem("🔢 Tab 4 · Matriks Kecerahan (50×50)"):
                        gr.HTML('<div class="section-label">Ekstraksi Nilai Piksel Grayscale Kiri Atas</div>')
                        with gr.Accordion("📐 Matriks Tahap Input", open=False):
                            t4_mat_gray   = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("📐 Matriks Tahap Median Filter", open=False):
                            t4_mat_median = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("📐 Matriks Tahap CLAHE (LAB)", open=False):
                            t4_mat_clahe  = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("📐 Matriks Tahap Sharpening", open=False):
                            t4_mat_sharp  = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("📐 Matriks Tahap Edge Detection", open=False):
                            t4_mat_edge   = gr.Dataframe(wrap=False, max_height=280)

                    with gr.TabItem("📈 Tab 5 · Laporan & Metrik"):
                        gr.HTML('<div class="section-label">Kalkulasi Validasi Objektif</div>')
                        t5_metrics_table = gr.Dataframe(wrap=False, max_height=250)
                        gr.HTML('<div class="section-label" style="margin-top:16px">📋 Hasil Keputusan Forensik Otomatis</div>')
                        t5_forensic = gr.Markdown(elem_classes="forensic-box")

                all_outputs_berantai = [
                    t2_input, t2_median, t2_clahe, t2_sharp, t2_edge,
                    t3_hist_gray, t3_hist_median, t3_hist_clahe, t3_hist_sharp, t3_hist_edge,
                    t4_mat_gray, t4_mat_median, t4_mat_clahe, t4_mat_sharp, t4_mat_edge,
                    t5_metrics_table, t5_forensic, comparison_output,
                    tab1_original, tab1_sharp_out, tab1_edge_out
                ]

                def pipeline_wrapper(img, rot, zoom):
                    results = run_pipeline(img, rot, zoom)
                    (pi, pm, pc, ps, pe, hi, hm, hc, hs, he, mi, mm, mc, ms, me, metrics_df, forensic, comp) = results
                    t1_orig = img if img is not None else pi
                    return (pi, pm, pc, ps, pe, hi, hm, hc, hs, he, mi, mm, mc, ms, me, metrics_df, forensic, comp, t1_orig, ps, pe)

                run_btn.click(
                    fn=pipeline_wrapper,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=all_outputs_berantai
                )

            # ========================================================
            # TAB UTAMA 2: MODE MANDIRI (KUSTOM / BARU)
            # ========================================================
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.HTML('<div class="section-label">📁 Upload & Konfigurasi</div>')
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=220)

                        gr.HTML('<div class="section-label" style="margin-top:12px">🛠️ Pilih Filter (Urutan Terjaga)</div>')
                        steps_m = gr.CheckboxGroup(
                            choices=["Median Filter", "CLAHE (LAB)", "Sharpening", "Edge Detection"],
                            label="Opsi Proses Kustom",
                            value=[]
                        )

                        gr.HTML('<div class="section-label" style="margin-top:12px">⚙️ Kontrol Geometri</div>')
                        rot_m = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_m = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")

                        btn_m = gr.Button("▶ Terapkan Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        gr.HTML('<div class="section-label">👁️ Pratinjau Komparasi</div>')
                        comp_out_m = gr.Image(label="Komparasi Kustom (Sisi Berdampingan)", height=240)

                        with gr.Row():
                            with gr.Column():
                                gr.HTML('<div class="section-label">Hasil Akhir</div>')
                                img_out_m = gr.Image(label="Output Visual", height=250)
                            with gr.Column():
                                gr.HTML('<div class="section-label">Statistik Distribusi</div>')
                                hist_out_m = gr.Image(label="Histogram Output", height=250)

                        gr.HTML('<div class="section-label" style="margin-top:10px">📊 Analisis Angka</div>')
                        with gr.Row():
                            mat_out_m = gr.Dataframe(label="Matriks Piksel Kecerahan", max_height=180)
                            metrics_out_m = gr.Dataframe(label="Evaluasi Metrik", max_height=180)

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, steps_m, rot_m, zoom_m],
                    outputs=[img_out_m, comp_out_m, hist_out_m, mat_out_m, metrics_out_m]
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_2712/3266010094.py:386: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:
/tmp/ipykernel_2712/3266010094.py:386: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:
/tmp/ipykernel_2712/3266010094.py:386: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8eea4c87f38bb45d10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8eea4c87f38bb45d10.gradio.live


**Asistensi 4**

In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Fitur: Mode Berantai (Otomatis) & Mode Mandiri (Kustom + Pipeline Tracker)
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Wajib untuk lingkungan tanpa GUI seperti Colab
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math

# ============================================================
# SECTION 1: UTILITY & METRIC FUNCTIONS
# ============================================================

def pil_to_cv2(pil_img):
    """Konversi PIL Image ke NumPy array BGR (OpenCV)."""
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    """Konversi NumPy array BGR ke PIL Image Berwarna."""
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def gray_to_pil(gray_img):
    """Konversi grayscale NumPy array ke PIL Image."""
    return Image.fromarray(gray_img)

def compute_mse(original_gray, processed_gray):
    original_gray = original_gray.astype(np.float64)
    processed_gray = processed_gray.astype(np.float64)
    if original_gray.shape != processed_gray.shape:
        processed_gray = cv2.resize(processed_gray, (original_gray.shape[1], original_gray.shape[0]))
    return float(np.mean((original_gray - processed_gray) ** 2))

def compute_psnr(mse):
    if mse == 0: return float('inf')
    return float(20 * math.log10(255.0 / math.sqrt(mse)))

def compute_sharpness(gray_img):
    return float(cv2.Laplacian(gray_img, cv2.CV_64F).var())

def compute_contrast(gray_img):
    return float(gray_img.std())

def compute_brightness(gray_img):
    return float(gray_img.mean())

def get_metrics(original_gray, processed_bgr_or_gray, label):
    if len(processed_bgr_or_gray.shape) == 3:
        target_gray = cv2.cvtColor(processed_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        target_gray = processed_bgr_or_gray

    mse = compute_mse(original_gray, target_gray)
    psnr = compute_psnr(mse)
    return {
        "Tahap": label,
        "MSE": round(mse, 4),
        "PSNR (dB)": round(psnr, 4) if psnr != float('inf') else "∞",
        "Sharpness": round(compute_sharpness(target_gray), 4),
        "Contrast": round(compute_contrast(target_gray), 4),
        "Brightness": round(compute_brightness(target_gray), 4),
    }

# ============================================================
# SECTION 2: RGB COLOR PIPELINE & KUSTOM PROCESSING FUNCTIONS
# ============================================================

def step_median_filter_rgb(bgr_img, ksize=5):
    return cv2.medianBlur(bgr_img, ksize)

def step_clahe_rgb(bgr_img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cl = clahe.apply(l)
    enhanced_lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

def step_sharpening_rgb(bgr_img):
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(bgr_img, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def step_rotate_rgb(bgr_img, angle_str):
    angle_map = {"Tidak Ada": None, "90°": cv2.ROTATE_90_CLOCKWISE, "180°": cv2.ROTATE_180, "270°": cv2.ROTATE_90_COUNTERCLOCKWISE}
    code = angle_map.get(angle_str, None)
    if code is not None:
        return cv2.rotate(bgr_img, code)
    return bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    if zoom_factor == 1.0: return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    # Crop to original size if zoomed in
    if zoom_factor > 1.0:
        start_h, start_w = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return zoomed

def step_edge_detection(bgr_img):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, threshold1=50, threshold2=150)

# --- FUNGSI TAMBAHAN MODE MANDIRI ---
def step_grayscale_3ch(bgr_img):
    """Konversi Grayscale tapi dipertahankan 3 channel agar pipeline tidak crash."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def step_deskew(bgr_img, angle):
    """Rotasi kustom untuk koreksi kemiringan citra."""
    if angle == 0.0: return bgr_img
    (h, w) = bgr_img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(bgr_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def step_contrast(bgr_img, alpha):
    """Atur kontras. Alpha > 1 menaikkan kontras."""
    if alpha == 1.0: return bgr_img
    return cv2.convertScaleAbs(bgr_img, alpha=alpha, beta=0)

def step_denoise(bgr_img):
    """Reduksi Noise (Non-Local Means)."""
    return cv2.fastNlMeansDenoisingColored(bgr_img, None, 10, 10, 7, 21)

def step_edge_3ch(bgr_img):
    """Edge detection (Canny) dipertahankan 3 channel."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(bgr_or_gray_img, title):
    fig, ax = plt.subplots(figsize=(5, 3), facecolor='#111827')
    ax.set_facecolor('#111827')
    gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY) if len(bgr_or_gray_img.shape) == 3 else bgr_or_gray_img
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#2dd4bf')
    ax.set_xlabel("Intensitas Piksel", fontsize=9, color='#9ca3af')
    ax.set_ylabel("Frekuensi", fontsize=9, color='#9ca3af')
    ax.tick_params(colors='#9ca3af', labelsize=8)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15, color='#9ca3af')
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_or_gray_img, size=50):
    gray_img = cv2.cvtColor(bgr_or_gray_img, cv2.COLOR_BGR2GRAY) if len(bgr_or_gray_img.shape) == 3 else bgr_or_gray_img
    h, w = gray_img.shape
    region = gray_img[:min(size, h), :min(size, w)]
    df = pd.DataFrame(region)
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    h1, w1, _ = original_bgr.shape
    h2, w2, _ = final_bgr.shape
    max_h = max(h1, h2)
    pad1 = np.zeros((max_h, w1, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2 = np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150
    combined = np.hstack([pad1, divider, pad2])
    cv2.putText(combined, "ASLI", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "RESTORASI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

def generate_forensic_summary(metrics_list):
    if not metrics_list: return "Tidak ada data metrik untuk dianalisis."
    lines = ["📋 **LAPORAN ANALISIS FORENSIK CITRA**\n", "---"]
    orig_m  = next((m for m in metrics_list if m["Tahap"] == "Input RGB"), None)
    sharp_m = next((m for m in metrics_list if m["Tahap"] == "Sharpening RGB"), None)
    if orig_m and sharp_m:
        s_before, s_after = orig_m["Sharpness"], sharp_m["Sharpness"]
        diff = s_after - s_before
        dir = "meningkat" if diff > 0 else "menurun"
        lines.append(f"🔍 **Metrik Ketajaman (Sharpness):** Tingkat ketajaman {dir} dari **{s_before:.2f}** menjadi **{s_after:.2f}**.")
    lines.append("\n✅ **Kesimpulan Forensik:** Evaluasi berhasil dijalankan secara objektif berdasarkan metrik di atas.")
    return "\n\n".join(lines)

# ============================================================
# SECTION 4: PIPELINE FUNCTIONS (BERANTAI & MANDIRI)
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Fungsi eksekusi Mode Berantai (Otomatis)."""
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        empty_hist = plot_histogram(np.zeros((100, 100), dtype=np.uint8), "Tidak Ada Data")
        return (empty, empty, empty, empty, empty, empty, empty_hist, empty_hist, empty_hist, empty_hist, empty_hist,
                empty_df, empty_df, empty_df, empty_df, empty_df, empty_df, "Silakan upload citra terlebih dahulu.", empty)

    bgr_input = pil_to_cv2(input_image)
    original_gray = cv2.cvtColor(bgr_input, cv2.COLOR_BGR2GRAY)

    median_bgr = step_median_filter_rgb(bgr_input)
    clahe_bgr  = step_clahe_rgb(median_bgr)
    sharp_bgr  = step_sharpening_rgb(clahe_bgr)
    rotated_bgr = step_rotate_rgb(sharp_bgr, rotation_choice)
    zoomed_bgr  = step_zoom_rgb(rotated_bgr, float(zoom_factor))
    edge_gray   = step_edge_detection(zoomed_bgr)

    metrics_list = [
        get_metrics(original_gray, bgr_input, "Input RGB"),
        get_metrics(original_gray, median_bgr, "Median Filter RGB"),
        get_metrics(original_gray, clahe_bgr, "CLAHE RGB"),
        get_metrics(original_gray, sharp_bgr, "Sharpening RGB"),
        get_metrics(original_gray, edge_gray, "Edge Detection"),
    ]
    metrics_df = pd.DataFrame(metrics_list)
    forensic_text = generate_forensic_summary(metrics_list)
    comp_img = make_comparison_image(bgr_input, zoomed_bgr)

    return (
        input_image, cv2_to_pil(median_bgr), cv2_to_pil(clahe_bgr), cv2_to_pil(sharp_bgr), gray_to_pil(edge_gray),
        plot_histogram(bgr_input, "Histogram Input RGB"), plot_histogram(median_bgr, "Histogram Median"),
        plot_histogram(clahe_bgr, "Histogram CLAHE"), plot_histogram(sharp_bgr, "Histogram Sharpening"),
        plot_histogram(edge_gray, "Histogram Edge"),
        get_matrix_dataframe(bgr_input), get_matrix_dataframe(median_bgr), get_matrix_dataframe(clahe_bgr),
        get_matrix_dataframe(sharp_bgr), get_matrix_dataframe(edge_gray),
        metrics_df, forensic_text, comp_img
    )

def run_mandiri_pipeline(input_image, deskew_val, scale_val, contrast_val, selected_steps):
    """Fungsi eksekusi kustom Mode Mandiri dengan Tracking Pipeline."""
    if input_image is None:
        empty_img = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_hist = plot_histogram(np.zeros((10, 10), dtype=np.uint8), "Tidak Ada Data")
        return empty_img, empty_img, empty_hist, pd.DataFrame(), pd.DataFrame(), []

    bgr_img = pil_to_cv2(input_image)
    original_gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    current_img = bgr_img.copy()

    # Tracking Pipeline Stages (Gallery Output)
    pipeline_gallery = [(cv2_to_pil(current_img), "1. Input Asli")]
    step_num = 2

    # 1. Koreksi Kemiringan
    if deskew_val != 0.0:
        current_img = step_deskew(current_img, deskew_val)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Kemiringan ({deskew_val}°)"))
        step_num += 1

    # 2. Scaling
    if scale_val != 1.0:
        current_img = step_zoom_rgb(current_img, scale_val)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Scaling ({scale_val}x)"))
        step_num += 1

    # 3. Grayscale
    if "Grayscale" in selected_steps:
        current_img = step_grayscale_3ch(current_img)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Grayscale"))
        step_num += 1

    # 4. Kontras
    if contrast_val != 1.0:
        current_img = step_contrast(current_img, contrast_val)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Kontras ({contrast_val})"))
        step_num += 1

    # 5. Denoise
    if "Noise Reduction" in selected_steps:
        current_img = step_denoise(current_img)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Noise Reduction"))
        step_num += 1

    # 6. Deblurring (Sharpening)
    if "Deblurring" in selected_steps:
        current_img = step_sharpening_rgb(current_img) # Reuse sharpening filter
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Deblurring"))
        step_num += 1

    # 7. Edge Detection
    if "Edge Detection" in selected_steps:
        current_img = step_edge_3ch(current_img)
        pipeline_gallery.append((cv2_to_pil(current_img), f"{step_num}. Edge Detection"))
        step_num += 1

    # Final Output Extractor
    comp_img = make_comparison_image(bgr_img, current_img)
    hist_out = plot_histogram(current_img, "Histogram Hasil Mandiri")
    mat_out = get_matrix_dataframe(current_img)

    metrics_df = pd.DataFrame([
        get_metrics(original_gray, bgr_img, "Citra Asli"),
        get_metrics(original_gray, current_img, "Output Mandiri Akhir")
    ])

    return cv2_to_pil(current_img), comp_img, hist_out, mat_out, metrics_df, pipeline_gallery

# ============================================================
# SECTION 5: PREMIUM DARK THEME CSS & UI BUILD
# ============================================================

CUSTOM_CSS = """
body, .gradio-container { background-color: #0b0f19 !important; color: #e2e8f0 !important; font-family: 'Segoe UI', Tahoma, sans-serif !important; }
.app-header { background: linear-gradient(135deg, #0f172a, #115e59); padding: 24px; border-radius: 14px; margin-bottom: 20px; text-align: center; border: 1px solid #14b8a6; }
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.section-label { font-size: 1rem; font-weight: 700; color: #2dd4bf; margin-bottom: 8px; border-bottom: 2px solid #0f766e; display: inline-block; }
.run-btn { background: linear-gradient(90deg, #0f766e, #14b8a6) !important; color: white !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
.forensic-box { background: #111827; border-left: 5px solid #14b8a6; border-radius: 10px; padding: 16px; color: #e2e8f0; }
"""

def build_ui():
    force_dark_mode_js = "() => document.body.classList.add('dark')"
    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:
        gr.HTML('<div class="app-header"><h1>🔬 Dashboard Restorasi & Analisis Forensik Citra</h1><p>Mode Berantai (Otomatis) & Mode Mandiri (Kustom)</p></div>')

        with gr.Tabs():
            # ==========================================
            # TAB UTAMA 1: MODE BERANTAI
            # ==========================================
            with gr.TabItem("⛓️ Mode Berantai (Pipeline Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.HTML('<div class="section-label">📁 Upload Citra</div>')
                        input_image = gr.Image(type="pil", label="Unggah Citra", height=220)
                        gr.HTML('<div class="section-label" style="margin-top:12px">⚙️ Kontrol Geometri</div>')
                        rotation_choice = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_slider = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")
                        run_btn = gr.Button("▶ Jalankan Pipeline Forensik", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        gr.HTML('<div class="section-label">🖼️ Komparasi Sisi Berdampingan</div>')
                        comparison_output = gr.Image(label="Side-by-Side", height=260)

                gr.HTML("<hr style='border-color:#1f2937; margin:10px 0'>")
                with gr.Tabs():
                    with gr.TabItem("📥 Hasil Akhir"):
                        with gr.Row():
                            tab1_original = gr.Image(label="Citra Asli", height=300)
                            tab1_sharp_out = gr.Image(label="Output Restorasi (Warna)", height=300)
                            tab1_edge_out = gr.Image(label="Output Deteksi Tepi", height=300)

                    with gr.TabItem("🔄 Tahapan Pipeline"):
                        with gr.Row():
                            t2_input = gr.Image(label="1. Input", height=200)
                            t2_median = gr.Image(label="2. Median Filter", height=200)
                            t2_clahe = gr.Image(label="3. CLAHE", height=200)
                        with gr.Row():
                            t2_sharp = gr.Image(label="4. Sharpening", height=200)
                            t2_edge = gr.Image(label="5. Edge Detection", height=200)

                    with gr.TabItem("📊 Histogram"):
                        with gr.Row():
                            t3_hist_gray = gr.Image(label="Input", height=220)
                            t3_hist_median = gr.Image(label="Median", height=220)
                            t3_hist_clahe = gr.Image(label="CLAHE", height=220)
                        with gr.Row():
                            t3_hist_sharp = gr.Image(label="Sharpening", height=220)
                            t3_hist_edge = gr.Image(label="Edge", height=220)

                    with gr.TabItem("🔢 Matriks (50x50)"):
                        with gr.Accordion("Matriks Input", open=False): t4_mat_gray = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("Matriks Median", open=False): t4_mat_median = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("Matriks CLAHE", open=False): t4_mat_clahe = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("Matriks Sharpening", open=False): t4_mat_sharp = gr.Dataframe(wrap=False, max_height=280)
                        with gr.Accordion("Matriks Edge", open=False): t4_mat_edge = gr.Dataframe(wrap=False, max_height=280)

                    with gr.TabItem("📈 Laporan Forensik"):
                        t5_metrics_table = gr.Dataframe(wrap=False, max_height=250)
                        t5_forensic = gr.Markdown(elem_classes="forensic-box")

                def berantai_wrapper(img, rot, zoom):
                    res = run_pipeline(img, rot, zoom)
                    t1_orig = img if img is not None else res[0]
                    return res + (t1_orig, res[3], res[4])

                run_btn.click(
                    fn=berantai_wrapper,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=[t2_input, t2_median, t2_clahe, t2_sharp, t2_edge, t3_hist_gray, t3_hist_median, t3_hist_clahe, t3_hist_sharp, t3_hist_edge, t4_mat_gray, t4_mat_median, t4_mat_clahe, t4_mat_sharp, t4_mat_edge, t5_metrics_table, t5_forensic, comparison_output, tab1_original, tab1_sharp_out, tab1_edge_out]
                )

            # ==========================================
            # TAB UTAMA 2: MODE MANDIRI (NEW)
            # ==========================================
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.HTML('<div class="section-label">📁 Upload Citra Base</div>')
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=200)

                        gr.HTML('<div class="section-label" style="margin-top:12px">🛠️ Kontrol Linear Slider</div>')
                        deskew_m = gr.Slider(-45.0, 45.0, step=1.0, value=0.0, label="📐 Koreksi Kemiringan (Derajat)")
                        scale_m = gr.Slider(0.1, 4.0, step=0.1, value=1.0, label="📏 Scaling / Resize")
                        contrast_m = gr.Slider(0.5, 3.0, step=0.1, value=1.0, label="🌗 Kontras (Alpha)")

                        gr.HTML('<div class="section-label" style="margin-top:12px">🎛️ Modul Efek Aktif</div>')
                        steps_m = gr.CheckboxGroup(
                            choices=["Grayscale", "Noise Reduction", "Deblurring", "Edge Detection"],
                            label="Terapkan Efek (Urutan Eksekusi Terjaga Otomatis)", value=[]
                        )
                        btn_m = gr.Button("▶ Eksekusi Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        with gr.Tabs():
                            with gr.TabItem("👁️ Pratinjau Output Akhir"):
                                comp_out_m = gr.Image(label="Komparasi Kustom (Asli vs Akhir)", height=240)
                                with gr.Row():
                                    img_out_m = gr.Image(label="Output Visual Tunggal", height=250)
                                    hist_out_m = gr.Image(label="Histogram Output", height=250)
                                with gr.Row():
                                    mat_out_m = gr.Dataframe(label="Matriks Piksel", max_height=150)
                                    metrics_out_m = gr.Dataframe(label="Evaluasi Metrik", max_height=150)

                            with gr.TabItem("🎬 Tracker Tahapan Pipeline"):
                                gr.HTML("<p style='color:#9ca3af; font-size:0.9rem'>Di bawah ini adalah riwayat perubahan dari setiap efek yang Anda pilih secara berurutan.</p>")
                                gallery_m = gr.Gallery(label="Tahapan Pipeline Kustom", show_label=True, elem_id="gallery", columns=[3], rows=[2], object_fit="contain", height="auto")

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, deskew_m, scale_m, contrast_m, steps_m],
                    outputs=[img_out_m, comp_out_m, hist_out_m, mat_out_m, metrics_out_m, gallery_m]
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_765/2859951863.py:327: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:
/tmp/ipykernel_765/2859951863.py:327: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:
/tmp/ipykernel_765/2859951863.py:327: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js=force_dark_mode_js) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://62e8876c4dc1f18de7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://62e8876c4dc1f18de7.gradio.live


In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Fitur: Pipeline Berantai (7 Tahap Sesuai Flowchart) & Mode Mandiri
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math

# ============================================================
# SECTION 1: UTILITY & METRIC FUNCTIONS
# ============================================================

def pil_to_cv2(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def compute_mse(original_gray, processed_gray):
    original_gray = original_gray.astype(np.float64)
    processed_gray = processed_gray.astype(np.float64)
    if original_gray.shape != processed_gray.shape:
        processed_gray = cv2.resize(processed_gray, (original_gray.shape[1], original_gray.shape[0]))
    return float(np.mean((original_gray - processed_gray) ** 2))

def compute_psnr(mse):
    if mse == 0: return float('inf')
    return float(20 * math.log10(255.0 / math.sqrt(mse)))

def compute_sharpness(gray_img):
    return float(cv2.Laplacian(gray_img, cv2.CV_64F).var())

def compute_contrast(gray_img):
    return float(gray_img.std())

def compute_brightness(gray_img):
    return float(gray_img.mean())

def get_metrics(original_gray, processed_bgr, label):
    target_gray = cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2GRAY)
    mse = compute_mse(original_gray, target_gray)
    psnr = compute_psnr(mse)
    return {
        "Tahap": label,
        "MSE": round(mse, 4),
        "PSNR (dB)": round(psnr, 4) if psnr != float('inf') else "∞",
        "Sharpness": round(compute_sharpness(target_gray), 4),
        "Contrast": round(compute_contrast(target_gray), 4),
        "Brightness": round(compute_brightness(target_gray), 4),
    }

# ============================================================
# SECTION 2: 7-STEP PIPELINE FUNCTIONS
# ============================================================

def step_denoise(bgr_img):
    """Tahap 1: Denoising (Noise Reduction)"""
    return cv2.fastNlMeansDenoisingColored(bgr_img, None, 10, 10, 7, 21)

def step_deblur(bgr_img):
    """Tahap 2: Unblur / Deblur (Unsharp Masking Ringan)"""
    gaussian = cv2.GaussianBlur(bgr_img, (0, 0), 2.0)
    return cv2.addWeighted(bgr_img, 1.5, gaussian, -0.5, 0)

def step_clahe_rgb(bgr_img, clip_limit=2.0, tile_grid=(8, 8)):
    """Tahap 3: Contrast Enhancement (CLAHE)"""
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

def step_sharpening_rgb(bgr_img):
    """Tahap 4: Sharpening (Penajaman Detail)"""
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(bgr_img, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def step_edge_enhance(bgr_img):
    """Tahap 5: Edge Enhancement (Penegasan Tepi)"""
    kernel_edge = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    edges = cv2.filter2D(bgr_img, -1, kernel_edge)
    return cv2.addWeighted(bgr_img, 1.0, edges, 0.3, 0)

def step_reconstruct(bgr_img):
    """Tahap 6: Rekonstruksi Citra (Bilateral Filter untuk menghaluskan)"""
    return cv2.bilateralFilter(bgr_img, d=7, sigmaColor=50, sigmaSpace=50)

def step_rotate_rgb(bgr_img, angle_str):
    angle_map = {"Tidak Ada": None, "90°": cv2.ROTATE_90_CLOCKWISE, "180°": cv2.ROTATE_180, "270°": cv2.ROTATE_90_COUNTERCLOCKWISE}
    code = angle_map.get(angle_str, None)
    return cv2.rotate(bgr_img, code) if code is not None else bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    if zoom_factor == 1.0: return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    if zoom_factor > 1.0:
        start_h, start_w = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return zoomed

# Mode Mandiri Helpers
def step_deskew(bgr_img, angle):
    if angle == 0.0: return bgr_img
    h, w = bgr_img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(bgr_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(bgr_img, title):
    fig, ax = plt.subplots(figsize=(4, 3), facecolor='#111827')
    ax.set_facecolor('#111827')
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold', color='#2dd4bf')
    ax.tick_params(colors='#9ca3af', labelsize=7)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15)
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=90, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_img, size=50):
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    df = pd.DataFrame(gray_img[:min(size, gray_img.shape[0]), :min(size, gray_img.shape[1])])
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    h1, w1 = original_bgr.shape[:2]
    h2, w2 = final_bgr.shape[:2]
    max_h = max(h1, h2)
    pad1, pad2 = np.zeros((max_h, w1, 3), dtype=np.uint8), np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150
    combined = np.hstack([pad1, divider, pad2])
    cv2.putText(combined, "ASLI", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "HASIL REKONSTRUKSI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

def generate_forensic_summary(metrics_list):
    if not metrics_list: return "Tidak ada data metrik."
    lines = ["📋 **LAPORAN ANALISIS FORENSIK CITRA**\n", "---"]
    orig_m  = metrics_list[0]
    final_m = metrics_list[-1]
    s_before, s_after = orig_m["Sharpness"], final_m["Sharpness"]
    diff = s_after - s_before
    lines.append(f"🔍 **Evaluasi Kualitas:** Ketajaman (Sharpness) {'meningkat' if diff > 0 else 'menurun'} dari **{s_before:.2f}** menjadi **{s_after:.2f}**.")
    lines.append(f"📊 **Skor PSNR Akhir:** **{final_m['PSNR (dB)']}**.")
    return "\n\n".join(lines)

# ============================================================
# SECTION 4: PIPELINE EXECUTORS
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Eksekusi Mode Berantai 7 Tahap"""
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        empty_hist = plot_histogram(np.zeros((100, 100, 3), dtype=np.uint8), "Kosong")
        # Mengembalikan list kosong sesuai jumlah output (7 img, 7 hist, 7 mat, 2 text, 3 display)
        return [empty]*7 + [empty_hist]*7 + [empty_df]*7 + [empty_df, "Silakan upload citra.", empty, empty, empty]

    bgr_input = pil_to_cv2(input_image)

    # 1. Input Base (Setelah rotasi & zoom)
    base_bgr = step_zoom_rgb(step_rotate_rgb(bgr_input, rotation_choice), float(zoom_factor))
    original_gray = cv2.cvtColor(base_bgr, cv2.COLOR_BGR2GRAY)

    # Eksekusi Pipeline
    img_denoise = step_denoise(base_bgr)
    img_deblur  = step_deblur(img_denoise)
    img_clahe   = step_clahe_rgb(img_deblur)
    img_sharp   = step_sharpening_rgb(img_clahe)
    img_edge    = step_edge_enhance(img_sharp)
    img_recon   = step_reconstruct(img_edge)

    stages = [
        (base_bgr, "1. Input Citra"),
        (img_denoise, "2. Denoising"),
        (img_deblur, "3. Deblurring"),
        (img_clahe, "4. CLAHE"),
        (img_sharp, "5. Sharpening"),
        (img_edge, "6. Edge Enhancement"),
        (img_recon, "7. Rekonstruksi Akhir")
    ]

    metrics_list = [get_metrics(original_gray, img, title) for img, title in stages]

    # Kumpulkan Output Visual
    out_images = [cv2_to_pil(img) for img, _ in stages]
    out_hists  = [plot_histogram(img, f"Hist: {title}") for img, title in stages]
    out_mats   = [get_matrix_dataframe(img) for img, _ in stages]

    comp_img = make_comparison_image(base_bgr, img_recon)

    return (*out_images, *out_hists, *out_mats, pd.DataFrame(metrics_list), generate_forensic_summary(metrics_list), comp_img, out_images[0], out_images[-1])

def run_mandiri_pipeline(input_image, deskew_val, scale_val, contrast_val, selected_steps):
    if input_image is None: return Image.new("RGB", (400, 300), color=(17, 24, 39)), Image.new("RGB", (400, 300), color=(17, 24, 39)), None, pd.DataFrame(), pd.DataFrame(), []
    bgr_img = pil_to_cv2(input_image)
    current_img = bgr_img.copy()
    gallery = [(cv2_to_pil(current_img), "1. Input Asli")]

    if deskew_val != 0.0:
        current_img = step_deskew(current_img, deskew_val); gallery.append((cv2_to_pil(current_img), f"Kemiringan ({deskew_val}°)"))
    if scale_val != 1.0:
        current_img = step_zoom_rgb(current_img, scale_val); gallery.append((cv2_to_pil(current_img), f"Scaling ({scale_val}x)"))
    if "Grayscale" in selected_steps:
        current_img = cv2.cvtColor(cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR); gallery.append((cv2_to_pil(current_img), "Grayscale"))
    if contrast_val != 1.0:
        current_img = cv2.convertScaleAbs(current_img, alpha=contrast_val, beta=0); gallery.append((cv2_to_pil(current_img), f"Kontras ({contrast_val})"))
    if "Noise Reduction" in selected_steps:
        current_img = step_denoise(current_img); gallery.append((cv2_to_pil(current_img), "Noise Reduction"))
    if "Deblurring" in selected_steps:
        current_img = step_deblur(current_img); gallery.append((cv2_to_pil(current_img), "Deblurring"))
    if "Edge Detection" in selected_steps:
        gray_edge = cv2.Canny(cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY), 50, 150)
        current_img = cv2.cvtColor(gray_edge, cv2.COLOR_GRAY2BGR); gallery.append((cv2_to_pil(current_img), "Edge Detection"))

    metrics = pd.DataFrame([get_metrics(cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY), bgr_img, "Asli"), get_metrics(cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY), current_img, "Akhir")])
    return cv2_to_pil(current_img), make_comparison_image(bgr_img, current_img), plot_histogram(current_img, "Histogram"), get_matrix_dataframe(current_img), metrics, gallery

# ============================================================
# SECTION 5: UI BUILD
# ============================================================

CUSTOM_CSS = """
body, .gradio-container { background-color: #0b0f19 !important; color: #e2e8f0 !important; font-family: 'Segoe UI', Tahoma, sans-serif !important; }
.app-header { background: linear-gradient(135deg, #0f172a, #115e59); padding: 24px; border-radius: 14px; margin-bottom: 20px; text-align: center; border: 1px solid #14b8a6; }
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.section-label { font-size: 1rem; font-weight: 700; color: #2dd4bf; margin-bottom: 8px; border-bottom: 2px solid #0f766e; display: inline-block; }
.run-btn { background: linear-gradient(90deg, #0f766e, #14b8a6) !important; color: white !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
"""

def build_ui():
    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
        gr.HTML('<div class="app-header"><h1>🔬 Dashboard Restorasi & Analisis Forensik Citra</h1></div>')

        with gr.Tabs():
            # TAB 1: MODE BERANTAI
            with gr.TabItem("⛓️ Mode Berantai (Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        input_image = gr.Image(type="pil", label="Unggah Citra Buram", height=220)
                        rotation_choice = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_slider = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")
                        run_btn = gr.Button("▶ Jalankan 7 Tahap Pipeline", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        comparison_output = gr.Image(label="Evaluasi (Sebelum vs Sesudah)", height=320)

                with gr.Tabs():
                    with gr.TabItem("🔄 Tahapan Pipeline (7 Tahap)"):
                        with gr.Row():
                            t_in = gr.Image(label="1. Input", height=200)
                            t_den = gr.Image(label="2. Denoising", height=200)
                            t_deb = gr.Image(label="3. Deblurring", height=200)
                            t_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            t_sha = gr.Image(label="5. Sharpening", height=200)
                            t_edg = gr.Image(label="6. Edge Enhance", height=200)
                            t_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("📊 Histogram"):
                        with gr.Row():
                            h_in = gr.Image(label="1. Input", height=200)
                            h_den = gr.Image(label="2. Denoising", height=200)
                            h_deb = gr.Image(label="3. Deblurring", height=200)
                            h_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            h_sha = gr.Image(label="5. Sharpening", height=200)
                            h_edg = gr.Image(label="6. Edge Enhance", height=200)
                            h_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("🔢 Matriks Piksel"):
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 1. Input"); m_in = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 2. Denoising"); m_den = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 3. Deblurring"); m_deb = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 4. CLAHE"); m_cla = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 5. Sharpening"); m_sha = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 6. Edge Enhance"); m_edg = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 7. Rekonstruksi Akhir"); m_rec = gr.Dataframe(max_height=200)

                    with gr.TabItem("📈 Laporan Analisis"):
                        metrics_table = gr.Dataframe()
                        forensic_text = gr.Markdown()

                    with gr.TabItem("📥 Output Final"):
                        with gr.Row():
                            final_orig = gr.Image(label="Citra Asli")
                            final_out = gr.Image(label="Citra Hasil Restorasi")

                run_btn.click(
                    fn=run_pipeline,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=[
                        t_in, t_den, t_deb, t_cla, t_sha, t_edg, t_rec,
                        h_in, h_den, h_deb, h_cla, h_sha, h_edg, h_rec,
                        m_in, m_den, m_deb, m_cla, m_sha, m_edg, m_rec,
                        metrics_table, forensic_text, comparison_output, final_orig, final_out
                    ]
                )

            # TAB 2: MODE MANDIRI
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=200)
                        deskew_m = gr.Slider(-45.0, 45.0, step=1.0, value=0.0, label="📐 Koreksi Kemiringan")
                        scale_m = gr.Slider(0.1, 4.0, step=0.1, value=1.0, label="📏 Scaling")
                        contrast_m = gr.Slider(0.5, 3.0, step=0.1, value=1.0, label="🌗 Kontras")
                        steps_m = gr.CheckboxGroup(["Grayscale", "Noise Reduction", "Deblurring", "Edge Detection"], label="Terapkan Efek")
                        btn_m = gr.Button("▶ Eksekusi Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        with gr.Tabs():
                            with gr.TabItem("👁️ Pratinjau"):
                                comp_out_m = gr.Image(label="Komparasi (Asli vs Akhir)")
                                img_out_m = gr.Image(label="Output Visual Akhir")
                            with gr.TabItem("📊 Data"):
                                hist_out_m = gr.Image(label="Histogram Output")
                                metrics_out_m = gr.Dataframe(label="Evaluasi Metrik")
                                mat_out_m = gr.Dataframe(label="Matriks Piksel", max_height=200)
                            with gr.TabItem("🎬 Pipeline Tracker"):
                                gallery_m = gr.Gallery(label="Tahapan Custom", columns=3, object_fit="contain")

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, deskew_m, scale_m, contrast_m, steps_m],
                    outputs=[img_out_m, comp_out_m, hist_out_m, mat_out_m, metrics_out_m, gallery_m]
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_765/1012090566.py:256: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_765/1012090566.py:256: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_765/1012090566.py:256: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.a

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fae41e485fca7c640e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fae41e485fca7c640e.gradio.live


Jam Malam

In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Fitur: Adaptive Pipeline (11 Tahap) & Mode Mandiri Kustom
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math
import time

# ============================================================
# SECTION 1: UTILITY & METRICS (EVALUASI)
# ============================================================

def pil_to_cv2(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def compute_mse(img1, img2):
    return np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)

def compute_psnr(mse):
    return float(20 * math.log10(255.0 / math.sqrt(mse))) if mse > 0 else float('inf')

def compute_ssim(img1, img2):
    """Implementasi SSIM Sederhana berbasis NumPy"""
    C1 = (0.01 * 255)**2
    C2 = (0.03 * 255)**2
    i1, i2 = img1.astype(np.float64), img2.astype(np.float64)
    mu1, mu2 = cv2.GaussianBlur(i1, (11, 11), 1.5), cv2.GaussianBlur(i2, (11, 11), 1.5)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1 * mu2
    sigma1_sq = cv2.GaussianBlur(i1**2, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(i2**2, (11, 11), 1.5) - mu2_sq
    sigma12 = cv2.GaussianBlur(i1 * i2, (11, 11), 1.5) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return np.mean(ssim_map)

def compute_entropy(gray_img):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()
    hist = hist[hist > 0] / np.sum(hist)
    return -np.sum(hist * np.log2(hist))

def evaluate_result(original_bgr, processed_bgr, label, start_time=None):
    orig_gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
    proc_gray = cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2GRAY)

    mse = compute_mse(orig_gray, proc_gray)
    return {
        "Tahap": label,
        "PSNR (dB)": round(compute_psnr(mse), 2),
        "SSIM": round(compute_ssim(orig_gray, proc_gray), 4),
        "MSE": round(mse, 2),
        "Sharpness": round(np.var(cv2.Laplacian(proc_gray, cv2.CV_64F)), 2),
        "Contrast": round(proc_gray.std(), 2),
        "Brightness": round(proc_gray.mean(), 2),
        "Entropy": round(compute_entropy(proc_gray), 2),
        "Time (s)": round(time.time() - start_time, 3) if start_time else 0.0
    }

# ============================================================
# SECTION 2: ADAPTIVE PIPELINE FUNCTIONS
# ============================================================

def analyze_image(bgr_img):
    """Menganalisis karakteristik kualitas citra."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = np.var(cv2.Laplacian(gray, cv2.CV_64F))
    noise_level = np.std(cv2.blur(gray, (3,3)) - gray) # Estimasi kasar noise
    contrast = gray.std()
    brightness = gray.mean()
    entropy = compute_entropy(gray)

    return {
        "blur_score": blur_score,
        "noise_level": noise_level,
        "contrast": contrast,
        "brightness": brightness,
        "entropy": entropy
    }

def classify_blur(bgr_img, analysis_data):
    """Mengklasifikasikan jenis blur menggunakan gradien terarah (Sobel)."""
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = analysis_data["blur_score"]

    if blur_score > 800:
        return "Tidak Signifikan (Jernih)"

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    var_x, var_y = np.var(sobelx), np.var(sobely)

    ratio = var_x / (var_y + 1e-6)
    if ratio > 2.0 or ratio < 0.5:
        return "Motion Blur"
    elif blur_score < 150:
        return "Gaussian Blur"
    elif blur_score < 400:
        return "Defocus Blur"
    return "Unknown Blur"

def adaptive_denoise(bgr_img, noise_level, manual_strength=None):
    strength = manual_strength if manual_strength else min(max(noise_level * 0.8, 3), 15)
    return cv2.fastNlMeansDenoisingColored(bgr_img, None, h=strength, hColor=strength, templateWindowSize=7, searchWindowSize=21)

def adaptive_deblur(bgr_img, blur_score, blur_type, iterations=1):
    """Pseudo Richardson-Lucy / Adaptive Unsharp based on blur severity."""
    # Mensimulasikan iterasi pemulihan
    result = bgr_img.copy()
    strength = 1.5 if blur_score > 300 else 2.5
    for _ in range(int(iterations)):
        gaussian = cv2.GaussianBlur(result, (0, 0), 2.0)
        result = cv2.addWeighted(result, strength, gaussian, -(strength-1.0), 0)
    return result

def gamma_correction(bgr_img, brightness, manual_gamma=None):
    gamma = manual_gamma if manual_gamma else (1.0 + (128 - brightness) / 255.0)
    gamma = max(0.5, min(gamma, 2.0)) # Batasi ekstrem
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(bgr_img, table), gamma

def adaptive_clahe(bgr_img, contrast, manual_clip=None):
    clip = manual_clip if manual_clip else max(1.0, 4.0 - (contrast / 20.0))
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR), clip

def adaptive_sharpen(bgr_img, blur_score, manual_amount=None):
    amount = manual_amount if manual_amount else max(1.0, 3.0 - (blur_score / 300.0))
    gaussian = cv2.GaussianBlur(bgr_img, (0, 0), 3.0)
    sharpened = cv2.addWeighted(bgr_img, 1.0 + amount, gaussian, -amount, 0)
    return sharpened, amount

def edge_enhancement(bgr_img, strength=0.3):
    kernel_edge = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    edges = cv2.filter2D(bgr_img, -1, kernel_edge)
    return cv2.addWeighted(bgr_img, 1.0, edges, strength, 0)

def artifact_reduction(bgr_img, strength=10):
    """Mengurangi efek halo/ringing menggunakan Bilateral ringan."""
    return cv2.bilateralFilter(bgr_img, d=5, sigmaColor=strength, sigmaSpace=strength)

def generate_report(analysis, blur_type, gamma, clahe_clip, sharpen_amount):
    lines = [
        "📋 **LAPORAN ANALISIS FORENSIK CITRA OTOMATIS**",
        "---",
        f"🔍 **Klasifikasi Blur:** `{blur_type}`",
        f"📊 **Skor Awal:** Blur ({analysis['blur_score']:.2f}) | Noise ({analysis['noise_level']:.2f}) | Kontras ({analysis['contrast']:.2f}) | Kecerahan ({analysis['brightness']:.2f})",
        "\n⚙️ **Tindakan Adaptif Sistem:**"
    ]

    n_lvl = "Tinggi" if analysis['noise_level'] > 10 else "Sedang" if analysis['noise_level'] > 5 else "Rendah"
    c_lvl = "Rendah" if analysis['contrast'] < 40 else "Tinggi"

    lines.append(f"- **Noise {n_lvl} terdeteksi.** Sistem mengaplikasikan Denoising adaptif sesuai level noise.")
    if blur_type != "Tidak Signifikan (Jernih)":
        lines.append(f"- **Blur terdeteksi.** Sistem menerapkan Deblurring iteratif.")
    lines.append(f"- **Kecerahan disesuaikan.** Gamma otomatis diset ke `{gamma:.2f}`.")
    lines.append(f"- **Kontras {c_lvl}.** CLAHE Clip Limit diset ke `{clahe_clip:.2f}`.")
    lines.append(f"- **Penajaman Detail.** Radius Unsharp Mask dihitung dengan amount `{sharpen_amount:.2f}`.")
    lines.append("- **Reduksi Artefak:** Aktif (Mencegah efek halo & kontur palsu).")

    lines.append("\n✅ **Kesimpulan:** Citra berhasil dipulihkan secara otomatis berdasarkan karakteristik piksel spesifiknya.")
    return "\n".join(lines)

# Transformasi Dasar
def step_rotate_rgb(bgr_img, angle_str):
    angle_map = {"Tidak Ada": None, "90°": cv2.ROTATE_90_CLOCKWISE, "180°": cv2.ROTATE_180, "270°": cv2.ROTATE_90_COUNTERCLOCKWISE}
    code = angle_map.get(angle_str, None)
    return cv2.rotate(bgr_img, code) if code is not None else bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    if zoom_factor == 1.0: return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    if zoom_factor > 1.0:
        start_h, start_w = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return zoomed

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(bgr_img, title):
    fig, ax = plt.subplots(figsize=(4, 3), facecolor='#111827')
    ax.set_facecolor('#111827')
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold', color='#2dd4bf')
    ax.tick_params(colors='#9ca3af', labelsize=7)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15)
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=90, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_img, size=50):
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    df = pd.DataFrame(gray_img[:min(size, gray_img.shape[0]), :min(size, gray_img.shape[1])])
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    h1, w1 = original_bgr.shape[:2]
    h2, w2 = final_bgr.shape[:2]
    max_h = max(h1, h2)
    pad1, pad2 = np.zeros((max_h, w1, 3), dtype=np.uint8), np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150
    combined = np.hstack([pad1, divider, pad2])
    cv2.putText(combined, "ASLI", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "HASIL REKONSTRUKSI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

# ============================================================
# SECTION 4: EXECUTORS (AUTO & MANUAL)
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Eksekusi Mode Berantai Otomatis (Mempertahankan 7 Slot UI)"""
    start_t = time.time()
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        return [empty]*7 + [plot_histogram(np.zeros((10,10,3), dtype=np.uint8), "Kosong")]*7 + [empty_df]*7 + [empty_df, "Silakan upload citra.", empty, empty, empty]

    bgr_input = pil_to_cv2(input_image)
    base_bgr = step_zoom_rgb(step_rotate_rgb(bgr_input, rotation_choice), float(zoom_factor))

    # --- INTELLIGENT ANALYSIS ---
    analysis = analyze_image(base_bgr)
    blur_type = classify_blur(base_bgr, analysis)

    # --- PIPELINE EKSEKUSI ---
    # Memetakan 11 tahap konseptual ke 7 slot UI Visual

    # Slot 2: Denoise
    img_denoise = adaptive_denoise(base_bgr, analysis["noise_level"])

    # Slot 3: Deblur
    iters = 3 if blur_type == "Motion Blur" else 1
    img_deblur = adaptive_deblur(img_denoise, analysis["blur_score"], blur_type, iterations=iters)

    # Slot 4: Gamma + CLAHE
    img_gamma, gamma_val = gamma_correction(img_deblur, analysis["brightness"])
    img_clahe, clahe_val = adaptive_clahe(img_gamma, analysis["contrast"])

    # Slot 5: Sharpen
    img_sharp, sharp_val = adaptive_sharpen(img_clahe, analysis["blur_score"])

    # Slot 6: Edge Enhance
    img_edge = edge_enhancement(img_sharp)

    # Slot 7: Artifact Reduction & Rekonstruksi Akhir
    img_recon = artifact_reduction(img_edge)

    # --- KOMPILASI HASIL ---
    stages = [
        (base_bgr, "1. Input Citra"),
        (img_denoise, "2. Denoising Adaptif"),
        (img_deblur, "3. Deblurring"),
        (img_clahe, "4. Gamma & CLAHE"),
        (img_sharp, "5. Sharpening Adaptif"),
        (img_edge, "6. Edge Enhancement"),
        (img_recon, "7. Rekonstruksi (Artifact Reduction)")
    ]

    metrics_list = [evaluate_result(base_bgr, img, title, start_t) for img, title in stages]

    out_images = [cv2_to_pil(img) for img, _ in stages]
    out_hists  = [plot_histogram(img, f"Hist: {title}") for img, title in stages]
    out_mats   = [get_matrix_dataframe(img) for img, _ in stages]

    comp_img = make_comparison_image(base_bgr, img_recon)
    report = generate_report(analysis, blur_type, gamma_val, clahe_val, sharp_val)

    return (*out_images, *out_hists, *out_mats, pd.DataFrame(metrics_list), report, comp_img, out_images[0], out_images[-1])


def run_mandiri_pipeline(input_image, deskew_val, scale_val, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, selected_steps):
    """Mode Mandiri Kustom dengan Parameter Lengkap"""
    start_t = time.time()
    if input_image is None:
        return Image.new("RGB", (400, 300), color=(17, 24, 39)), Image.new("RGB", (400, 300), color=(17, 24, 39)), None, pd.DataFrame(), pd.DataFrame(), []

    bgr_img = pil_to_cv2(input_image)
    current_img = bgr_img.copy()
    gallery = [(cv2_to_pil(current_img), "1. Input Asli")]

    # Transformasi Awal
    if deskew_val != 0.0:
        h, w = current_img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), deskew_val, 1.0)
        current_img = cv2.warpAffine(current_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        gallery.append((cv2_to_pil(current_img), f"Kemiringan ({deskew_val}°)"))
    if scale_val != 1.0:
        current_img = step_zoom_rgb(current_img, scale_val); gallery.append((cv2_to_pil(current_img), f"Scaling ({scale_val}x)"))

    # Pipeline Opsional
    if "Denoising" in selected_steps:
        current_img = adaptive_denoise(current_img, noise_level=0, manual_strength=denoise_str)
        gallery.append((cv2_to_pil(current_img), f"Denoising (Str:{denoise_str})"))
    if "Deblurring" in selected_steps:
        current_img = adaptive_deblur(current_img, blur_score=0, blur_type="", iterations=deblur_iter)
        gallery.append((cv2_to_pil(current_img), f"Deblurring (Iter:{deblur_iter})"))
    if "Gamma Correction" in selected_steps:
        current_img, _ = gamma_correction(current_img, brightness=0, manual_gamma=gamma_val)
        gallery.append((cv2_to_pil(current_img), f"Gamma ({gamma_val})"))
    if "CLAHE" in selected_steps:
        current_img, _ = adaptive_clahe(current_img, contrast=0, manual_clip=clahe_clip)
        gallery.append((cv2_to_pil(current_img), f"CLAHE (Clip:{clahe_clip})"))
    if "Sharpening" in selected_steps:
        current_img, _ = adaptive_sharpen(current_img, blur_score=0, manual_amount=sharp_amt)
        gallery.append((cv2_to_pil(current_img), f"Sharpening (Amt:{sharp_amt})"))
    if "Edge Enhance" in selected_steps:
        current_img = edge_enhancement(current_img, strength=edge_str)
        gallery.append((cv2_to_pil(current_img), f"Edge Enhance ({edge_str})"))
    if "Artifact Reduction" in selected_steps:
        current_img = artifact_reduction(current_img, strength=artifact_str)
        gallery.append((cv2_to_pil(current_img), f"Artifact Reduction ({artifact_str})"))

    metrics = pd.DataFrame([
        evaluate_result(bgr_img, bgr_img, "Asli", start_t),
        evaluate_result(bgr_img, current_img, "Akhir Kustom", start_t)
    ])

    return cv2_to_pil(current_img), make_comparison_image(bgr_img, current_img), plot_histogram(current_img, "Histogram"), get_matrix_dataframe(current_img), metrics, gallery

# ============================================================
# SECTION 5: UI BUILD
# ============================================================

CUSTOM_CSS = """
body, .gradio-container { background-color: #0b0f19 !important; color: #e2e8f0 !important; font-family: 'Segoe UI', Tahoma, sans-serif !important; }
.app-header { background: linear-gradient(135deg, #0f172a, #115e59); padding: 24px; border-radius: 14px; margin-bottom: 20px; text-align: center; border: 1px solid #14b8a6; }
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.section-label { font-size: 1rem; font-weight: 700; color: #2dd4bf; margin-bottom: 8px; border-bottom: 2px solid #0f766e; display: inline-block; }
.run-btn { background: linear-gradient(90deg, #0f766e, #14b8a6) !important; color: white !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
"""

def build_ui():
    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
        gr.HTML('<div class="app-header"><h1>🔬 Dashboard Restorasi & Analisis Forensik Citra</h1></div>')

        with gr.Tabs():
            # ========================================================
            # TAB 1: MODE BERANTAI (UI DIPERTAHANKAN 100%)
            # ========================================================
            with gr.TabItem("⛓️ Mode Berantai (Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        input_image = gr.Image(type="pil", label="Unggah Citra Buram", height=220)
                        rotation_choice = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_slider = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")
                        run_btn = gr.Button("▶ Jalankan Auto-Pipeline Terpadu", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        comparison_output = gr.Image(label="Evaluasi (Sebelum vs Sesudah)", height=320)

                with gr.Tabs():
                    with gr.TabItem("🔄 Tahapan Pipeline (7 Tahap)"):
                        with gr.Row():
                            t_in = gr.Image(label="1. Input", height=200)
                            t_den = gr.Image(label="2. Denoising", height=200)
                            t_deb = gr.Image(label="3. Deblurring", height=200)
                            t_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            t_sha = gr.Image(label="5. Sharpening", height=200)
                            t_edg = gr.Image(label="6. Edge Enhance", height=200)
                            t_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("📊 Histogram"):
                        with gr.Row():
                            h_in = gr.Image(label="1. Input", height=200)
                            h_den = gr.Image(label="2. Denoising", height=200)
                            h_deb = gr.Image(label="3. Deblurring", height=200)
                            h_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            h_sha = gr.Image(label="5. Sharpening", height=200)
                            h_edg = gr.Image(label="6. Edge Enhance", height=200)
                            h_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("🔢 Matriks Piksel"):
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 1. Input"); m_in = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 2. Denoising"); m_den = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 3. Deblurring"); m_deb = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 4. CLAHE"); m_cla = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 5. Sharpening"); m_sha = gr.Dataframe(max_height=200)
                            with gr.Column():
                                gr.Markdown("### 6. Edge Enhance"); m_edg = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("### 7. Rekonstruksi Akhir"); m_rec = gr.Dataframe(max_height=200)

                    with gr.TabItem("📈 Laporan Analisis"):
                        metrics_table = gr.Dataframe(label="Tabel Evaluasi Kualitas Visual")
                        forensic_text = gr.Markdown(label="Kesimpulan Forensik")

                    with gr.TabItem("📥 Output Final"):
                        with gr.Row():
                            final_orig = gr.Image(label="Citra Asli")
                            final_out = gr.Image(label="Citra Hasil Restorasi")

                run_btn.click(
                    fn=run_pipeline,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=[
                        t_in, t_den, t_deb, t_cla, t_sha, t_edg, t_rec,
                        h_in, h_den, h_deb, h_cla, h_sha, h_edg, h_rec,
                        m_in, m_den, m_deb, m_cla, m_sha, m_edg, m_rec,
                        metrics_table, forensic_text, comparison_output, final_orig, final_out
                    ]
                )

            # ========================================================
            # TAB 2: MODE MANDIRI (DITAMBAHKAN PARAMETER SLIDER)
            # ========================================================
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=200)

                        gr.HTML('<div class="section-label">Geometri</div>')
                        deskew_m = gr.Slider(-45.0, 45.0, step=1.0, value=0.0, label="📐 Koreksi Kemiringan")
                        scale_m = gr.Slider(0.1, 4.0, step=0.1, value=1.0, label="📏 Scaling")

                        gr.HTML('<div class="section-label">Parameter Proses</div>')
                        denoise_str = gr.Slider(1.0, 30.0, step=1.0, value=10.0, label="Noise Strength")
                        deblur_iter = gr.Slider(1, 10, step=1, value=2, label="Deblur Iterations")
                        gamma_val = gr.Slider(0.1, 3.0, step=0.1, value=1.0, label="Gamma Value")
                        clahe_clip = gr.Slider(1.0, 10.0, step=0.5, value=2.0, label="CLAHE Clip Limit")
                        sharp_amt = gr.Slider(0.1, 5.0, step=0.1, value=1.5, label="Sharpen Amount")
                        edge_str = gr.Slider(0.1, 1.0, step=0.1, value=0.3, label="Edge Strength")
                        artifact_str = gr.Slider(1.0, 50.0, step=1.0, value=10.0, label="Artifact Reduct Strength")

                        steps_m = gr.CheckboxGroup(
                            ["Denoising", "Deblurring", "Gamma Correction", "CLAHE", "Sharpening", "Edge Enhance", "Artifact Reduction"],
                            label="Terapkan Efek (Urut)"
                        )
                        btn_m = gr.Button("▶ Eksekusi Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        with gr.Tabs():
                            with gr.TabItem("👁️ Pratinjau"):
                                comp_out_m = gr.Image(label="Komparasi (Asli vs Akhir)")
                                img_out_m = gr.Image(label="Output Visual Akhir")
                            with gr.TabItem("📊 Data & Evaluasi"):
                                hist_out_m = gr.Image(label="Histogram Output")
                                metrics_out_m = gr.Dataframe(label="Evaluasi Metrik Akhir")
                                mat_out_m = gr.Dataframe(label="Matriks Piksel", max_height=200)
                            with gr.TabItem("🎬 Pipeline Tracker"):
                                gallery_m = gr.Gallery(label="Tahapan Custom", columns=3, object_fit="contain")

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, deskew_m, scale_m, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, steps_m],
                    outputs=[img_out_m, comp_out_m, hist_out_m, mat_out_m, metrics_out_m, gallery_m]
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_765/2789799130.py:361: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_765/2789799130.py:361: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_765/2789799130.py:361: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.a

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44e239c041972804cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44e239c041972804cb.gradio.live


**Percobaan terakhirrr**

In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Fitur: Adaptive Pipeline & Mode Mandiri Kustom (Dengan Tracker Matriks & Histogram)
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math
import time

# ============================================================
# SECTION 1: UTILITY & METRICS (EVALUASI)
# ============================================================

def pil_to_cv2(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def compute_mse(img1, img2):
    return np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)

def compute_psnr(mse):
    return float(20 * math.log10(255.0 / math.sqrt(mse))) if mse > 0 else float('inf')

def compute_ssim(img1, img2):
    C1 = (0.01 * 255)**2
    C2 = (0.03 * 255)**2
    i1, i2 = img1.astype(np.float64), img2.astype(np.float64)
    mu1, mu2 = cv2.GaussianBlur(i1, (11, 11), 1.5), cv2.GaussianBlur(i2, (11, 11), 1.5)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1 * mu2
    sigma1_sq = cv2.GaussianBlur(i1**2, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(i2**2, (11, 11), 1.5) - mu2_sq
    sigma12 = cv2.GaussianBlur(i1 * i2, (11, 11), 1.5) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return np.mean(ssim_map)

def compute_entropy(gray_img):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()
    hist = hist[hist > 0] / np.sum(hist)
    return -np.sum(hist * np.log2(hist))

def evaluate_result(original_bgr, processed_bgr, label, start_time=None):
    orig_gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
    proc_gray = cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2GRAY)

    mse = compute_mse(orig_gray, proc_gray)
    return {
        "Tahap": label,
        "PSNR (dB)": round(compute_psnr(mse), 2),
        "SSIM": round(compute_ssim(orig_gray, proc_gray), 4),
        "MSE": round(mse, 2),
        "Sharpness": round(np.var(cv2.Laplacian(proc_gray, cv2.CV_64F)), 2),
        "Contrast": round(proc_gray.std(), 2),
        "Brightness": round(proc_gray.mean(), 2),
        "Entropy": round(compute_entropy(proc_gray), 2),
        "Time (s)": round(time.time() - start_time, 3) if start_time else 0.0
    }

# ============================================================
# SECTION 2: ADAPTIVE PIPELINE FUNCTIONS
# ============================================================

def analyze_image(bgr_img):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = np.var(cv2.Laplacian(gray, cv2.CV_64F))
    noise_level = np.std(cv2.blur(gray, (3,3)) - gray)
    contrast = gray.std()
    brightness = gray.mean()
    entropy = compute_entropy(gray)

    return {
        "blur_score": blur_score,
        "noise_level": noise_level,
        "contrast": contrast,
        "brightness": brightness,
        "entropy": entropy
    }

def classify_blur(bgr_img, analysis_data):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = analysis_data["blur_score"]

    if blur_score > 800:
        return "Tidak Signifikan (Jernih)"

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    var_x, var_y = np.var(sobelx), np.var(sobely)

    ratio = var_x / (var_y + 1e-6)
    if ratio > 2.0 or ratio < 0.5:
        return "Motion Blur"
    elif blur_score < 150:
        return "Gaussian Blur"
    elif blur_score < 400:
        return "Defocus Blur"
    return "Unknown Blur"

def adaptive_denoise(bgr_img, noise_level, manual_strength=None):
    strength = manual_strength if manual_strength else min(max(noise_level * 0.8, 3), 15)
    return cv2.fastNlMeansDenoisingColored(bgr_img, None, h=strength, hColor=strength, templateWindowSize=7, searchWindowSize=21)

def adaptive_deblur(bgr_img, blur_score, blur_type, iterations=1):
    result = bgr_img.copy()
    strength = 1.5 if blur_score > 300 else 2.5
    for _ in range(int(iterations)):
        gaussian = cv2.GaussianBlur(result, (0, 0), 2.0)
        result = cv2.addWeighted(result, strength, gaussian, -(strength-1.0), 0)
    return result

def gamma_correction(bgr_img, brightness, manual_gamma=None):
    gamma = manual_gamma if manual_gamma else (1.0 + (128 - brightness) / 255.0)
    gamma = max(0.5, min(gamma, 2.0))
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(bgr_img, table), gamma

def adaptive_clahe(bgr_img, contrast, manual_clip=None):
    clip = manual_clip if manual_clip else max(1.0, 4.0 - (contrast / 20.0))
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR), clip

def adaptive_sharpen(bgr_img, blur_score, manual_amount=None):
    amount = manual_amount if manual_amount else max(1.0, 3.0 - (blur_score / 300.0))
    gaussian = cv2.GaussianBlur(bgr_img, (0, 0), 3.0)
    sharpened = cv2.addWeighted(bgr_img, 1.0 + amount, gaussian, -amount, 0)
    return sharpened, amount

def edge_enhancement(bgr_img, strength=0.3):
    kernel_edge = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    edges = cv2.filter2D(bgr_img, -1, kernel_edge)
    return cv2.addWeighted(bgr_img, 1.0, edges, strength, 0)

def artifact_reduction(bgr_img, strength=10):
    return cv2.bilateralFilter(bgr_img, d=5, sigmaColor=strength, sigmaSpace=strength)

def generate_report(analysis, blur_type, gamma, clahe_clip, sharpen_amount):
    lines = [
        "📋 **LAPORAN ANALISIS FORENSIK CITRA OTOMATIS**",
        "---",
        f"🔍 **Klasifikasi Blur:** `{blur_type}`",
        f"📊 **Skor Awal:** Blur ({analysis['blur_score']:.2f}) | Noise ({analysis['noise_level']:.2f}) | Kontras ({analysis['contrast']:.2f}) | Kecerahan ({analysis['brightness']:.2f})",
        "\n⚙️ **Tindakan Adaptif Sistem:**"
    ]

    n_lvl = "Tinggi" if analysis['noise_level'] > 10 else "Sedang" if analysis['noise_level'] > 5 else "Rendah"
    c_lvl = "Rendah" if analysis['contrast'] < 40 else "Tinggi"

    lines.append(f"- **Noise {n_lvl} terdeteksi.** Sistem mengaplikasikan Denoising adaptif sesuai level noise.")
    if blur_type != "Tidak Signifikan (Jernih)":
        lines.append(f"- **Blur terdeteksi.** Sistem menerapkan Deblurring iteratif.")
    lines.append(f"- **Kecerahan disesuaikan.** Gamma otomatis diset ke `{gamma:.2f}`.")
    lines.append(f"- **Kontras {c_lvl}.** CLAHE Clip Limit diset ke `{clahe_clip:.2f}`.")
    lines.append(f"- **Penajaman Detail.** Radius Unsharp Mask dihitung dengan amount `{sharpen_amount:.2f}`.")
    lines.append("- **Reduksi Artefak:** Aktif (Mencegah efek halo & kontur palsu).")

    lines.append("\n✅ **Kesimpulan:** Citra berhasil dipulihkan secara otomatis berdasarkan karakteristik piksel spesifiknya.")
    return "\n".join(lines)

def step_rotate_rgb(bgr_img, angle_str):
    angle_map = {"Tidak Ada": None, "90°": cv2.ROTATE_90_CLOCKWISE, "180°": cv2.ROTATE_180, "270°": cv2.ROTATE_90_COUNTERCLOCKWISE}
    code = angle_map.get(angle_str, None)
    return cv2.rotate(bgr_img, code) if code is not None else bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    if zoom_factor == 1.0: return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    if zoom_factor > 1.0:
        start_h, start_w = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return zoomed

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(bgr_img, title):
    fig, ax = plt.subplots(figsize=(4, 3), facecolor='#111827')
    ax.set_facecolor('#111827')
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold', color='#2dd4bf')
    ax.tick_params(colors='#9ca3af', labelsize=7)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15)
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=90, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_img, size=50):
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    df = pd.DataFrame(gray_img[:min(size, gray_img.shape[0]), :min(size, gray_img.shape[1])])
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    h1, w1 = original_bgr.shape[:2]
    h2, w2 = final_bgr.shape[:2]
    max_h = max(h1, h2)
    pad1, pad2 = np.zeros((max_h, w1, 3), dtype=np.uint8), np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2[:h2, :w2] = final_bgr
    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150
    combined = np.hstack([pad1, divider, pad2])
    cv2.putText(combined, "ASLI", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "HASIL REKONSTRUKSI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

# ============================================================
# SECTION 4: EXECUTORS (AUTO & MANUAL)
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Eksekusi Mode Berantai Otomatis"""
    start_t = time.time()
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        return [empty]*7 + [plot_histogram(np.zeros((10,10,3), dtype=np.uint8), "Kosong")]*7 + [empty_df]*7 + [empty_df, "Silakan upload citra.", empty, empty, empty]

    bgr_input = pil_to_cv2(input_image)
    base_bgr = step_zoom_rgb(step_rotate_rgb(bgr_input, rotation_choice), float(zoom_factor))

    analysis = analyze_image(base_bgr)
    blur_type = classify_blur(base_bgr, analysis)

    img_denoise = adaptive_denoise(base_bgr, analysis["noise_level"])
    iters = 3 if blur_type == "Motion Blur" else 1
    img_deblur = adaptive_deblur(img_denoise, analysis["blur_score"], blur_type, iterations=iters)
    img_gamma, gamma_val = gamma_correction(img_deblur, analysis["brightness"])
    img_clahe, clahe_val = adaptive_clahe(img_gamma, analysis["contrast"])
    img_sharp, sharp_val = adaptive_sharpen(img_clahe, analysis["blur_score"])
    img_edge = edge_enhancement(img_sharp)
    img_recon = artifact_reduction(img_edge)

    stages = [
        (base_bgr, "1. Input Citra"),
        (img_denoise, "2. Denoising Adaptif"),
        (img_deblur, "3. Deblurring"),
        (img_clahe, "4. Gamma & CLAHE"),
        (img_sharp, "5. Sharpening Adaptif"),
        (img_edge, "6. Edge Enhancement"),
        (img_recon, "7. Rekonstruksi Akhir")
    ]

    metrics_list = [evaluate_result(base_bgr, img, title, start_t) for img, title in stages]

    out_images = [cv2_to_pil(img) for img, _ in stages]
    out_hists  = [plot_histogram(img, f"Hist: {title}") for img, title in stages]
    out_mats   = [get_matrix_dataframe(img) for img, _ in stages]
    comp_img = make_comparison_image(base_bgr, img_recon)
    report = generate_report(analysis, blur_type, gamma_val, clahe_val, sharp_val)

    return (*out_images, *out_hists, *out_mats, pd.DataFrame(metrics_list), report, comp_img, out_images[0], out_images[-1])


def run_mandiri_pipeline(input_image, deskew_val, scale_val, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, selected_steps):
    """Mode Mandiri Kustom dengan Pipeline Tracker (Matriks & Histogram) Dinamis"""
    start_t = time.time()

    # Handling input kosong (menonaktifkan seluruh 10 slot matriks)
    if input_image is None:
        empty_img = Image.new("RGB", (400, 300), color=(17, 24, 39))
        return (empty_img, empty_img, pd.DataFrame(), [], []) + tuple([gr.update(visible=False)]*10)

    bgr_img = pil_to_cv2(input_image)
    current_img = bgr_img.copy()

    # Lacak seluruh tahapan yang dilewati secara dinamis
    states = [(current_img.copy(), "1. Input Asli")]
    step_num = 2

    if deskew_val != 0.0:
        h, w = current_img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), deskew_val, 1.0)
        current_img = cv2.warpAffine(current_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        states.append((current_img.copy(), f"{step_num}. Kemiringan ({deskew_val}°)")); step_num+=1

    if scale_val != 1.0:
        current_img = step_zoom_rgb(current_img, scale_val)
        states.append((current_img.copy(), f"{step_num}. Scaling ({scale_val}x)")); step_num+=1

    if "Denoising" in selected_steps:
        current_img = adaptive_denoise(current_img, noise_level=0, manual_strength=denoise_str)
        states.append((current_img.copy(), f"{step_num}. Denoising")); step_num+=1

    if "Deblurring" in selected_steps:
        current_img = adaptive_deblur(current_img, blur_score=0, blur_type="", iterations=deblur_iter)
        states.append((current_img.copy(), f"{step_num}. Deblurring")); step_num+=1

    if "Gamma Correction" in selected_steps:
        current_img, _ = gamma_correction(current_img, brightness=0, manual_gamma=gamma_val)
        states.append((current_img.copy(), f"{step_num}. Gamma ({gamma_val})")); step_num+=1

    if "CLAHE" in selected_steps:
        current_img, _ = adaptive_clahe(current_img, contrast=0, manual_clip=clahe_clip)
        states.append((current_img.copy(), f"{step_num}. CLAHE")); step_num+=1

    if "Sharpening" in selected_steps:
        current_img, _ = adaptive_sharpen(current_img, blur_score=0, manual_amount=sharp_amt)
        states.append((current_img.copy(), f"{step_num}. Sharpening")); step_num+=1

    if "Edge Enhance" in selected_steps:
        current_img = edge_enhancement(current_img, strength=edge_str)
        states.append((current_img.copy(), f"{step_num}. Edge Enhance")); step_num+=1

    if "Artifact Reduction" in selected_steps:
        current_img = artifact_reduction(current_img, strength=artifact_str)
        states.append((current_img.copy(), f"{step_num}. Artifact Reduction")); step_num+=1

    # Membuat Output Gallery dan Metrics
    gallery_out = [(cv2_to_pil(img), title) for img, title in states]
    hist_gallery_out = [(plot_histogram(img, title), title) for img, title in states]

    metrics = pd.DataFrame([
        evaluate_result(bgr_img, bgr_img, "Asli", start_t),
        evaluate_result(bgr_img, current_img, "Akhir Kustom", start_t)
    ])

    # Update Matriks Dataframes secara dinamis (maksimum 10 slot)
    mat_updates = []
    for i in range(10):
        if i < len(states):
            img, title = states[i]
            df = get_matrix_dataframe(img)
            mat_updates.append(gr.update(value=df, label=title, visible=True))
        else:
            mat_updates.append(gr.update(value=None, visible=False))

    # Mengembalikan nilai yang sesuai dengan List Input pada btn_m.click
    return (cv2_to_pil(current_img), make_comparison_image(bgr_img, current_img), metrics, gallery_out, hist_gallery_out) + tuple(mat_updates)


# ============================================================
# SECTION 5: UI BUILD
# ============================================================

CUSTOM_CSS = """
body, .gradio-container { background-color: #0b0f19 !important; color: #e2e8f0 !important; font-family: 'Segoe UI', Tahoma, sans-serif !important; }
.app-header { background: linear-gradient(135deg, #0f172a, #115e59); padding: 24px; border-radius: 14px; margin-bottom: 20px; text-align: center; border: 1px solid #14b8a6; }
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.section-label { font-size: 1rem; font-weight: 700; color: #2dd4bf; margin-bottom: 8px; border-bottom: 2px solid #0f766e; display: inline-block; }
.run-btn { background: linear-gradient(90deg, #0f766e, #14b8a6) !important; color: white !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
"""

def build_ui():
    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
        gr.HTML('<div class="app-header"><h1>🔬 Dashboard Restorasi & Analisis Forensik Citra</h1></div>')

        with gr.Tabs():
            # ========================================================
            # TAB 1: MODE BERANTAI (TETAP SAMA)
            # ========================================================
            with gr.TabItem("⛓️ Mode Berantai (Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        input_image = gr.Image(type="pil", label="Unggah Citra Buram", height=220)
                        rotation_choice = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_slider = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")
                        run_btn = gr.Button("▶ Jalankan Auto-Pipeline Terpadu", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        comparison_output = gr.Image(label="Evaluasi (Sebelum vs Sesudah)", height=320)

                with gr.Tabs():
                    with gr.TabItem("🔄 Tahapan Pipeline (7 Tahap)"):
                        with gr.Row():
                            t_in = gr.Image(label="1. Input", height=200)
                            t_den = gr.Image(label="2. Denoising", height=200)
                            t_deb = gr.Image(label="3. Deblurring", height=200)
                            t_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            t_sha = gr.Image(label="5. Sharpening", height=200)
                            t_edg = gr.Image(label="6. Edge Enhance", height=200)
                            t_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("📊 Histogram"):
                        with gr.Row():
                            h_in = gr.Image(label="1. Input", height=200)
                            h_den = gr.Image(label="2. Denoising", height=200)
                            h_deb = gr.Image(label="3. Deblurring", height=200)
                            h_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            h_sha = gr.Image(label="5. Sharpening", height=200)
                            h_edg = gr.Image(label="6. Edge Enhance", height=200)
                            h_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("🔢 Matriks Piksel"):
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 1. Input"); m_in = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 2. Denoising"); m_den = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 3. Deblurring"); m_deb = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 4. CLAHE"); m_cla = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 5. Sharpening"); m_sha = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 6. Edge Enhance"); m_edg = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 7. Rekonstruksi Akhir"); m_rec = gr.Dataframe(max_height=200)

                    with gr.TabItem("📈 Laporan Analisis"):
                        metrics_table = gr.Dataframe(label="Tabel Evaluasi Kualitas Visual")
                        forensic_text = gr.Markdown(label="Kesimpulan Forensik")

                    with gr.TabItem("📥 Output Final"):
                        with gr.Row():
                            final_orig = gr.Image(label="Citra Asli")
                            final_out = gr.Image(label="Citra Hasil Restorasi")

                run_btn.click(
                    fn=run_pipeline,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=[
                        t_in, t_den, t_deb, t_cla, t_sha, t_edg, t_rec,
                        h_in, h_den, h_deb, h_cla, h_sha, h_edg, h_rec,
                        m_in, m_den, m_deb, m_cla, m_sha, m_edg, m_rec,
                        metrics_table, forensic_text, comparison_output, final_orig, final_out
                    ]
                )

            # ========================================================
            # TAB 2: MODE MANDIRI (TRACKER MATRIKS & HISTOGRAM DINAMIS)
            # ========================================================
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=200)

                        gr.HTML('<div class="section-label">Geometri</div>')
                        deskew_m = gr.Slider(-45.0, 45.0, step=1.0, value=0.0, label="📐 Koreksi Kemiringan")
                        scale_m = gr.Slider(0.1, 4.0, step=0.1, value=1.0, label="📏 Scaling")

                        gr.HTML('<div class="section-label">Parameter Proses</div>')
                        denoise_str = gr.Slider(1.0, 30.0, step=1.0, value=10.0, label="Noise Strength")
                        deblur_iter = gr.Slider(1, 10, step=1, value=2, label="Deblur Iterations")
                        gamma_val = gr.Slider(0.1, 3.0, step=0.1, value=1.0, label="Gamma Value")
                        clahe_clip = gr.Slider(1.0, 10.0, step=0.5, value=2.0, label="CLAHE Clip Limit")
                        sharp_amt = gr.Slider(0.1, 5.0, step=0.1, value=1.5, label="Sharpen Amount")
                        edge_str = gr.Slider(0.1, 1.0, step=0.1, value=0.3, label="Edge Strength")
                        artifact_str = gr.Slider(1.0, 50.0, step=1.0, value=10.0, label="Artifact Reduct Strength")

                        steps_m = gr.CheckboxGroup(
                            ["Denoising", "Deblurring", "Gamma Correction", "CLAHE", "Sharpening", "Edge Enhance", "Artifact Reduction"],
                            label="Terapkan Efek (Urut)"
                        )
                        btn_m = gr.Button("▶ Eksekusi Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        with gr.Tabs():
                            with gr.TabItem("👁️ Pratinjau"):
                                comp_out_m = gr.Image(label="Komparasi (Asli vs Akhir)")
                                img_out_m = gr.Image(label="Output Visual Akhir")

                            with gr.TabItem("📊 Evaluasi Metrik"):
                                metrics_out_m = gr.Dataframe(label="Tabel Evaluasi Akhir")

                            with gr.TabItem("🎬 Citra Tracker"):
                                gallery_m = gr.Gallery(label="Visual Tahapan", columns=4, object_fit="contain")

                            with gr.TabItem("📈 Histogram Tracker"):
                                hist_gallery_m = gr.Gallery(label="Histogram Tahapan", columns=4, object_fit="contain")

                            with gr.TabItem("🔢 Matriks Tracker"):
                                # Membuat 10 slot statis untuk Matriks (maksimal step pada mode mandiri)
                                mat_m_boxes = []
                                with gr.Row():
                                    for i in range(5):
                                        with gr.Column():
                                            mat_m_boxes.append(gr.Dataframe(visible=False, max_height=200))
                                with gr.Row():
                                    for i in range(5, 10):
                                        with gr.Column():
                                            mat_m_boxes.append(gr.Dataframe(visible=False, max_height=200))

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, deskew_m, scale_m, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, steps_m],
                    outputs=[img_out_m, comp_out_m, metrics_out_m, gallery_m, hist_gallery_m] + mat_m_boxes
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_2920/3444837033.py:365: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_2920/3444837033.py:365: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
/tmp/ipykernel_2920/3444837033.py:365: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classLis

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9d6d110f55208e086e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
